# HierAdapter codet5+ 220M, 512 token code vulnerability

In [ ]:
import os
import sys
import copy
import platform
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_cosine_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix,
    balanced_accuracy_score, average_precision_score
)
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

MODEL_NAME      = "Salesforce/codet5p-220m"
TRAIN_CSV       = '/traincodex.csv'
TEST_CSV        = '/testcodex.csv'
MAX_LENGTH      = 512
D_MODEL         = 768
BATCH_SIZE      = 16
GRAD_ACCUM      = 2
EPOCHS          = 30
LR              = 5e-5
WARMUP_RATIO    = 0.15
BOTTLENECK      = 6
NUM_LEVELS      = 4
LAYERS_LEVEL    = 3
VAL_SPLIT       = 0.20
SEED            = 42
RDROP_ALPHA     = 0.3
MIXUP_ALPHA     = 0.2
MIXUP_PROB      = 0.5
SWA_START_RATIO = 0.6
SWA_LR          = 1e-5
PATIENCE        = 7

if torch.cuda.is_available():
    DEVICE      = torch.device("cuda:0")
    NUM_WORKERS = 4
    PIN_MEMORY  = True
elif torch.backends.mps.is_available():
    DEVICE      = torch.device("mps")
    NUM_WORKERS = 0
    PIN_MEMORY  = False
else:
    DEVICE      = torch.device("cpu")
    NUM_WORKERS = 0
    PIN_MEMORY  = False

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


class CodeDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.codes     = df['func_cleaned'].astype(str).tolist()
        self.labels    = df['label'].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.codes[idx],
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


class AttentionPooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attn = nn.Linear(d_model, 1)

    def forward(self, hidden, attention_mask):
        scores  = self.attn(hidden).squeeze(-1)
        scores  = scores.masked_fill(attention_mask == 0, -1e9)
        weights = torch.softmax(scores, dim=-1)
        return (hidden * weights.unsqueeze(-1)).sum(1)


class DynamicAlphaFusion(nn.Module):
    def __init__(self, d_model, num_levels):
        super().__init__()
        self.gate = nn.Linear(d_model, num_levels)

    def forward(self, base_cls, residuals):
        alphas   = torch.softmax(self.gate(base_cls), dim=-1)
        weighted = sum(alphas[:, i:i+1] * residuals[i] for i in range(len(residuals)))
        return base_cls + weighted


class LaplacianResidualAdapter(nn.Module):
    def __init__(self, d_model, bottleneck_dim):
        super().__init__()
        self.down    = nn.Linear(d_model, bottleneck_dim)
        self.act     = nn.GELU()
        self.up      = nn.Linear(bottleneck_dim, d_model)
        self.dropout = nn.Dropout(0.1)
        nn.init.kaiming_normal_(self.down.weight, nonlinearity='linear')
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return self.up(self.dropout(self.act(self.down(x))))


class LAPPEFT(nn.Module):
    def __init__(self):
        super().__init__()
        full_model   = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
        self.encoder = full_model.encoder
        for p in self.encoder.parameters():
            p.requires_grad = False
        del full_model
        self.pool_layers = nn.ModuleList([
            AttentionPooling(D_MODEL) for _ in range(NUM_LEVELS + 1)
        ])
        self.lras = nn.ModuleList([
            LaplacianResidualAdapter(D_MODEL, BOTTLENECK)
            for _ in range(NUM_LEVELS)
        ])
        self.level_norms = nn.ModuleList([nn.LayerNorm(D_MODEL) for _ in range(NUM_LEVELS)])
        self.fusion      = DynamicAlphaFusion(D_MODEL, NUM_LEVELS)
        self.classifier  = nn.Sequential(
            nn.Linear(D_MODEL, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )

    def encode(self, input_ids, attention_mask):
        out        = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden     = out.hidden_states
        num_layers = len(hidden)
        base       = self.pool_layers[NUM_LEVELS](hidden[-1], attention_mask)
        level_indices = [
            min((lvl + 1) * LAYERS_LEVEL, num_layers - 1) for lvl in range(NUM_LEVELS)
        ]
        level_cls = [
            self.pool_layers[lvl](hidden[level_indices[lvl]], attention_mask)
            for lvl in range(NUM_LEVELS)
        ]
        residuals = []
        for lvl in range(NUM_LEVELS):
            diff = level_cls[lvl] - level_cls[lvl + 1] if lvl < NUM_LEVELS - 1 else level_cls[lvl]
            r    = self.lras[lvl](diff)
            residuals.append(self.level_norms[lvl](r))
        recon = self.fusion(base, residuals)
        return recon

    def forward(self, input_ids, attention_mask, cls_embed=None):
        if cls_embed is not None:
            return self.classifier(cls_embed)
        recon = self.encode(input_ids, attention_mask)
        return self.classifier(recon)


def mixup_embeddings(embed, labels, alpha=MIXUP_ALPHA):
    lam   = np.random.beta(alpha, alpha)
    idx   = torch.randperm(embed.size(0), device=embed.device)
    mixed = lam * embed + (1 - lam) * embed[idx]
    return mixed, labels, labels[idx], lam


def rdrop_kl_loss(logits1, logits2):
    p = F.log_softmax(logits1, dim=-1)
    q = F.log_softmax(logits2, dim=-1)
    return (F.kl_div(p, q.exp(), reduction='batchmean') +
            F.kl_div(q, p.exp(), reduction='batchmean')) / 2


def mixup_ce_loss(logits, labels_a, labels_b, lam, weight=None):
    loss_fn = nn.CrossEntropyLoss(weight=weight, label_smoothing=0.05, reduction='none')
    return (lam * loss_fn(logits, labels_a) + (1 - lam) * loss_fn(logits, labels_b)).mean()


def compute_metrics(labels, preds, probs):
    labels = np.array(labels)
    preds  = np.array(preds)
    probs  = np.array(probs)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        'Accuracy':      accuracy_score(labels, preds),
        'Balanced_Acc':  balanced_accuracy_score(labels, preds),
        'Precision':     precision_score(labels, preds, zero_division=0),
        'Recall':        recall_score(labels, preds, zero_division=0),
        'F1':            f1_score(labels, preds, zero_division=0),
        'ROC_AUC':       roc_auc_score(labels, probs),
        'Avg_Precision': average_precision_score(labels, probs),
        'MCC':           matthews_corrcoef(labels, preds),
        'Specificity':   tn / (tn + fp + 1e-9),
        'Sensitivity':   tp / (tp + fn + 1e-9),
        'FPR':           fp / (fp + tn + 1e-9),
        'FNR':           fn / (fn + tp + 1e-9),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn)
    }


def train_epoch(model, loader, criterion, optimizer, scheduler, class_weights, step_counter):
    model.train()
    raw_model                        = model.module if isinstance(model, nn.DataParallel) else model
    total_loss                       = 0.0
    all_labels, all_preds, all_probs = [], [], []
    optimizer.zero_grad()
    for batch_idx, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)
        with torch.no_grad():
            embed = raw_model.encode(input_ids, attention_mask)
        use_mixup = (np.random.rand() < MIXUP_PROB)
        if use_mixup:
            mixed_embed, la, lb, lam = mixup_embeddings(embed, labels)
            logits1 = model(None, None, cls_embed=mixed_embed)
            logits2 = model(None, None, cls_embed=mixed_embed)
            ce_loss = mixup_ce_loss(logits1, la, lb, lam, weight=class_weights)
        else:
            logits1 = model(None, None, cls_embed=embed)
            logits2 = model(None, None, cls_embed=embed)
            ce_loss = criterion(logits1, labels)
        kl_loss = rdrop_kl_loss(logits1, logits2)
        loss    = ce_loss + RDROP_ALPHA * kl_loss
        loss    = loss / GRAD_ACCUM
        loss.backward()
        if (batch_idx + 1) % GRAD_ACCUM == 0 or (batch_idx + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(
                [p for p in raw_model.parameters() if p.requires_grad], 1.0
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            step_counter[0] += 1
        total_loss += (ce_loss + RDROP_ALPHA * kl_loss).item()
        with torch.no_grad():
            probs = torch.softmax(logits1, dim=-1)[:, 1].cpu().numpy()
            preds = logits1.argmax(dim=-1).cpu().numpy()
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)
    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss                       = 0.0
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['label'].to(DEVICE)
            logits         = model(input_ids, attention_mask)
            loss           = criterion(logits, labels)
            total_loss    += loss.item()
            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            preds = logits.argmax(dim=-1).cpu().numpy()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds)
            all_probs.extend(probs)
    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)


def update_swa(swa_model, raw_model):
    with torch.no_grad():
        for swa_p, p in zip(swa_model.parameters(), raw_model.parameters()):
            swa_p.data.copy_(swa_p.data + p.data)


def apply_swa_avg(swa_model, n_swa):
    with torch.no_grad():
        for p in swa_model.parameters():
            p.data.div_(n_swa)


def plot_training_curves(history, save_path='training_curves.png'):
    epochs = list(range(1, len(history['tr_loss']) + 1))
    fig, axes = plt.subplots(3, 3, figsize=(18, 14))
    fig.suptitle('LAPPEFT (CodeT5+) Training Curves', fontsize=16, fontweight='bold')

    def plot_ax(ax, key, title, ylabel):
        ax.plot(epochs, history[f'tr_{key}'], 'b-o', markersize=3, label='Train')
        ax.plot(epochs, history[f'va_{key}'], 'r-o', markersize=3, label='Val')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(True, alpha=0.3)

    plot_ax(axes[0][0], 'loss',    'Loss',          'Loss')
    plot_ax(axes[0][1], 'acc',     'Accuracy',      'Accuracy')
    plot_ax(axes[0][2], 'f1',      'F1 Score',      'F1')
    plot_ax(axes[1][0], 'auc',     'ROC AUC',       'AUC')
    plot_ax(axes[1][1], 'mcc',     'MCC',           'MCC')
    plot_ax(axes[1][2], 'prec',    'Precision',     'Precision')
    plot_ax(axes[2][0], 'rec',     'Recall',        'Recall')
    plot_ax(axes[2][1], 'bacc',    'Balanced Acc',  'Balanced Accuracy')
    plot_ax(axes[2][2], 'avgprec', 'Avg Precision', 'Avg Precision')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Training curves saved to {save_path}")


def plot_loss_curve(history, save_path='train_val_loss_curve.png'):
    epochs = list(range(1, len(history['tr_loss']) + 1))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(epochs, history['tr_loss'], 'b-o', markersize=4, linewidth=1.8, label='Train Loss')
    ax.plot(epochs, history['va_loss'], 'r-o', markersize=4, linewidth=1.8, label='Val Loss')
    ax.set_title('LAPPEFT (CodeT5+) — Train vs Validation Loss per Epoch', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Train/Val loss curve saved to {save_path}")


def plot_final_comparison(best_test_m, swa_test_m, save_path='test_comparison.png'):
    keys      = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Avg_Precision']
    x         = np.arange(len(keys))
    w         = 0.35
    best_vals = [best_test_m[k] for k in keys]
    swa_vals  = [swa_test_m[k] for k in keys] if swa_test_m else None
    fig, ax   = plt.subplots(figsize=(14, 6))
    bars1     = ax.bar(x - w/2, best_vals, w, label='Best Checkpoint', color='steelblue', alpha=0.85)
    if swa_vals:
        bars2 = ax.bar(x + w/2, swa_vals, w, label='SWA Model', color='darkorange', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(keys, rotation=20, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title('Test Set: Best Checkpoint vs SWA Model', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
    if swa_vals:
        for bar in bars2:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Test comparison chart saved to {save_path}")


def plot_confusion_matrix(m, title, save_path):
    cm      = np.array([[m['TN'], m['FP']], [m['FN'], m['TP']]])
    fig, ax = plt.subplots(figsize=(5, 4))
    im      = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred 0', 'Pred 1'])
    ax.set_yticklabels(['True 0', 'True 1'])
    thresh = cm.max() / 2.0
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > thresh else 'black',
                    fontsize=14, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Confusion matrix saved to {save_path}")


def plot_metrics_radar(best_test_m, swa_test_m, save_path='radar_chart.png'):
    keys   = ['Accuracy', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Specificity', 'Sensitivity']
    n      = len(keys)
    angles = [x / float(n) * 2 * np.pi for x in range(n)]
    angles += angles[:1]
    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

    def draw(m, color, label):
        vals  = [m[k] for k in keys]
        vals += vals[:1]
        ax.plot(angles, vals, color=color, linewidth=2, label=label)
        ax.fill(angles, vals, color=color, alpha=0.15)

    draw(best_test_m, 'steelblue', 'Best Checkpoint')
    if swa_test_m:
        draw(swa_test_m, 'darkorange', 'SWA Model')
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(keys, size=9)
    ax.set_ylim(0, 1)
    ax.set_title('Test Metrics Radar', fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Radar chart saved to {save_path}")


def main():
    train_full = pd.read_csv(TRAIN_CSV)
    test_full  = pd.read_csv(TEST_CSV)
    train_df, val_df = train_test_split(
        train_full, test_size=VAL_SPLIT, random_state=SEED, stratify=train_full['label']
    )
    _, test_df = train_test_split(
        test_full, test_size=0.50, random_state=SEED, stratify=test_full['label']
    )
    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
    print(f"Device: {DEVICE}  |  GPUs available: {num_gpus}  |  num_workers: {NUM_WORKERS}  |  pin_memory: {PIN_MEMORY}")
    if num_gpus > 1:
        for i in range(num_gpus):
            print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
    print()
    tokenizer    = AutoTokenizer.from_pretrained(MODEL_NAME)
    train_ds     = CodeDataset(train_df.reset_index(drop=True), tokenizer)
    val_ds       = CodeDataset(val_df.reset_index(drop=True),   tokenizer)
    test_ds      = CodeDataset(test_df.reset_index(drop=True),  tokenizer)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    counts        = train_df['label'].value_counts().sort_index().values
    raw_w         = torch.tensor([1.0 / c for c in counts], dtype=torch.float).to(DEVICE)
    class_weights = raw_w / raw_w.sum() * len(counts)
    raw_model = LAPPEFT().to(DEVICE)
    swa_model = copy.deepcopy(raw_model)
    for p in swa_model.parameters():
        p.requires_grad = False
    if num_gpus > 1:
        model = nn.DataParallel(raw_model, device_ids=list(range(num_gpus)))
    else:
        model = raw_model
    total_p   = sum(p.numel() for p in raw_model.parameters())
    trainable = sum(p.numel() for p in raw_model.parameters() if p.requires_grad)
    print(f"Total Params     : {total_p:,}")
    print(f"Trainable Params : {trainable:,}")
    print(f"Trainable %      : {100.0 * trainable / total_p:.4f}%\n")
    criterion        = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
    trainable_params = [p for p in raw_model.parameters() if p.requires_grad]
    optimizer        = torch.optim.AdamW(
        trainable_params, lr=LR, weight_decay=0.05,
        betas=(0.9, 0.999), eps=1e-8
    )
    effective_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    warmup_steps    = int(WARMUP_RATIO * effective_steps)
    scheduler       = get_cosine_schedule_with_warmup(optimizer, warmup_steps, effective_steps)
    swa_start_epoch = int(SWA_START_RATIO * EPOCHS)
    n_swa           = 0
    step_counter    = [0]
    best_val_f1     = -1.0
    best_epoch      = 0
    best_state      = None
    no_improve      = 0
    history = {
        'tr_loss': [], 'tr_acc': [], 'tr_f1': [], 'tr_auc': [], 'tr_mcc': [],
        'tr_prec': [], 'tr_rec': [], 'tr_bacc': [], 'tr_avgprec': [],
        'va_loss': [], 'va_acc': [], 'va_f1': [], 'va_auc': [], 'va_mcc': [],
        'va_prec': [], 'va_rec': [], 'va_bacc': [], 'va_avgprec': []
    }
    hdr = (f"{'Ep':>3} | {'TrLoss':>8} | {'TrAcc':>7} | {'TrF1':>7} | {'TrAUC':>7} | {'TrMCC':>7} | "
           f"{'VaLoss':>8} | {'VaAcc':>7} | {'VaF1':>7} | {'VaAUC':>7} | {'VaMCC':>7} | {'Note':<12}")
    sep = "-" * len(hdr)
    print(sep)
    print(hdr)
    print(sep)
    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_m = train_epoch(model, train_loader, criterion, optimizer,
                                    scheduler, class_weights, step_counter)
        va_loss, va_m = eval_epoch(model, val_loader, criterion)
        history['tr_loss'].append(tr_loss)
        history['tr_acc'].append(tr_m['Accuracy'])
        history['tr_f1'].append(tr_m['F1'])
        history['tr_auc'].append(tr_m['ROC_AUC'])
        history['tr_mcc'].append(tr_m['MCC'])
        history['tr_prec'].append(tr_m['Precision'])
        history['tr_rec'].append(tr_m['Recall'])
        history['tr_bacc'].append(tr_m['Balanced_Acc'])
        history['tr_avgprec'].append(tr_m['Avg_Precision'])
        history['va_loss'].append(va_loss)
        history['va_acc'].append(va_m['Accuracy'])
        history['va_f1'].append(va_m['F1'])
        history['va_auc'].append(va_m['ROC_AUC'])
        history['va_mcc'].append(va_m['MCC'])
        history['va_prec'].append(va_m['Precision'])
        history['va_rec'].append(va_m['Recall'])
        history['va_bacc'].append(va_m['Balanced_Acc'])
        history['va_avgprec'].append(va_m['Avg_Precision'])
        note = ""
        if epoch > swa_start_epoch:
            update_swa(swa_model, raw_model)
            n_swa += 1
            note += "SWA "
        is_best = va_m['F1'] > best_val_f1
        if is_best:
            best_val_f1 = va_m['F1']
            best_epoch  = epoch
            best_state  = {k: v.cpu().clone() for k, v in raw_model.state_dict().items()}
            no_improve  = 0
            note       += "<--"
        else:
            no_improve += 1
        print(
            f"{epoch:>3} | {tr_loss:>8.4f} | {tr_m['Accuracy']:>7.4f} | {tr_m['F1']:>7.4f} | "
            f"{tr_m['ROC_AUC']:>7.4f} | {tr_m['MCC']:>7.4f} | "
            f"{va_loss:>8.4f} | {va_m['Accuracy']:>7.4f} | {va_m['F1']:>7.4f} | "
            f"{va_m['ROC_AUC']:>7.4f} | {va_m['MCC']:>7.4f} | {note:<12}"
        )
        if no_improve >= PATIENCE:
            print(f"\n  Early stopping at epoch {epoch} (no val F1 improvement for {PATIENCE} epochs)")
            break
    print(sep)
    print(f"\nBest Epoch: {best_epoch}  |  Best Val F1: {best_val_f1:.4f}")
    print("\n--- Evaluating: Best Checkpoint Model ---")
    raw_model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    _, best_test_m = eval_epoch(model, test_loader, criterion)
    print("\n--- Evaluating: SWA Model ---")
    swa_test_m = None
    if n_swa > 0:
        apply_swa_avg(swa_model, n_swa)
        swa_model.to(DEVICE)
        _, swa_test_m = eval_epoch(swa_model, test_loader, criterion)
    print(f"\n{'='*60}")
    print(f"  Val F1 trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_f1'][-5:]]}")
    print(f"  Val Loss trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_loss'][-5:]]}")
    print(f"\n{'='*60}")
    print(f"  TEST RESULTS — Best Checkpoint (Epoch {best_epoch})")
    print(f"{'='*60}")
    scalar_keys = [k for k in best_test_m if k not in ('TP', 'TN', 'FP', 'FN')]
    for k in scalar_keys:
        print(f"  {k:<22}: {best_test_m[k]:.4f}")
    print(f"  {'TP':<22}: {best_test_m['TP']}")
    print(f"  {'TN':<22}: {best_test_m['TN']}")
    print(f"  {'FP':<22}: {best_test_m['FP']}")
    print(f"  {'FN':<22}: {best_test_m['FN']}")
    if swa_test_m is not None:
        print(f"\n{'='*60}")
        print(f"  TEST RESULTS — SWA Model (averaged over last {n_swa} epochs)")
        print(f"{'='*60}")
        for k in scalar_keys:
            print(f"  {k:<22}: {swa_test_m[k]:.4f}")
        print(f"  {'TP':<22}: {swa_test_m['TP']}")
        print(f"  {'TN':<22}: {swa_test_m['TN']}")
        print(f"  {'FP':<22}: {swa_test_m['FP']}")
        print(f"  {'FN':<22}: {swa_test_m['FN']}")
        print(f"\n{'='*60}")
        print(f"  COMPARISON: Best Checkpoint vs SWA")
        print(f"{'='*60}")
        compare_keys = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall']
        print(f"  {'Metric':<22} {'BestCkpt':>12} {'SWA':>12} {'Delta':>12}")
        print(f"  {'-'*58}")
        for k in compare_keys:
            delta  = swa_test_m[k] - best_test_m[k]
            symbol = "+" if delta >= 0 else ""
            print(f"  {k:<22} {best_test_m[k]:>12.4f} {swa_test_m[k]:>12.4f} {symbol+f'{delta:.4f}':>12}")
    plot_training_curves(history,  save_path='training_curves.png')
    plot_loss_curve(history,       save_path='train_val_loss_curve.png')
    plot_final_comparison(best_test_m, swa_test_m, save_path='test_comparison.png')
    plot_confusion_matrix(best_test_m, f'Confusion Matrix — Best Ckpt (Ep {best_epoch})', 'cm_best.png')
    if swa_test_m:
        plot_confusion_matrix(swa_test_m, f'Confusion Matrix — SWA ({n_swa} epochs)', 'cm_swa.png')
    plot_metrics_radar(best_test_m, swa_test_m, save_path='radar_chart.png')
    print("\nAll figures saved.")


if __name__ == "__main__":
    main()

# HierAdapter codet5+ 220M, 1024 token code vulnerability

In [ ]:
import os
import sys
import copy
import platform
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_cosine_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix,
    balanced_accuracy_score, average_precision_score
)
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

MODEL_NAME      = "Salesforce/codet5p-220m"
TRAIN_CSV       = 'dev1024/trqw.csv'
TEST_CSV        = 'dev1024/teqw.csv'
MAX_LENGTH      = 1024
D_MODEL         = 768
BATCH_SIZE      = 16
GRAD_ACCUM      = 2
EPOCHS          = 30
LR              = 5e-5
WARMUP_RATIO    = 0.15
BOTTLENECK      = 6
NUM_LEVELS      = 4
LAYERS_LEVEL    = 3
VAL_SPLIT       = 0.20
SEED            = 42
RDROP_ALPHA     = 0.3
MIXUP_ALPHA     = 0.2
MIXUP_PROB      = 0.5
SWA_START_RATIO = 0.6
SWA_LR          = 1e-5
PATIENCE        = 7

if torch.cuda.is_available():
    DEVICE      = torch.device("cuda:0")
    NUM_WORKERS = 4
    PIN_MEMORY  = True
elif torch.backends.mps.is_available():
    DEVICE      = torch.device("mps")
    NUM_WORKERS = 0
    PIN_MEMORY  = False
else:
    DEVICE      = torch.device("cpu")
    NUM_WORKERS = 0
    PIN_MEMORY  = False

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


class CodeDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.codes     = df['func_cleaned'].astype(str).tolist()
        self.labels    = df['label'].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.codes[idx],
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


class AttentionPooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attn = nn.Linear(d_model, 1)

    def forward(self, hidden, attention_mask):
        scores  = self.attn(hidden).squeeze(-1)
        scores  = scores.masked_fill(attention_mask == 0, -1e9)
        weights = torch.softmax(scores, dim=-1)
        return (hidden * weights.unsqueeze(-1)).sum(1)


class DynamicAlphaFusion(nn.Module):
    def __init__(self, d_model, num_levels):
        super().__init__()
        self.gate = nn.Linear(d_model, num_levels)

    def forward(self, base_cls, residuals):
        alphas   = torch.softmax(self.gate(base_cls), dim=-1)
        weighted = sum(alphas[:, i:i+1] * residuals[i] for i in range(len(residuals)))
        return base_cls + weighted


class LaplacianResidualAdapter(nn.Module):
    def __init__(self, d_model, bottleneck_dim):
        super().__init__()
        self.down    = nn.Linear(d_model, bottleneck_dim)
        self.act     = nn.GELU()
        self.up      = nn.Linear(bottleneck_dim, d_model)
        self.dropout = nn.Dropout(0.1)
        nn.init.kaiming_normal_(self.down.weight, nonlinearity='linear')
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return self.up(self.dropout(self.act(self.down(x))))


class LAPPEFT(nn.Module):
    def __init__(self):
        super().__init__()
        full_model   = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
        self.encoder = full_model.encoder
        for p in self.encoder.parameters():
            p.requires_grad = False
        del full_model
        self.pool_layers = nn.ModuleList([
            AttentionPooling(D_MODEL) for _ in range(NUM_LEVELS + 1)
        ])
        self.lras = nn.ModuleList([
            LaplacianResidualAdapter(D_MODEL, BOTTLENECK)
            for _ in range(NUM_LEVELS)
        ])
        self.level_norms = nn.ModuleList([nn.LayerNorm(D_MODEL) for _ in range(NUM_LEVELS)])
        self.fusion      = DynamicAlphaFusion(D_MODEL, NUM_LEVELS)
        self.classifier  = nn.Sequential(
            nn.Linear(D_MODEL, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )

    def encode(self, input_ids, attention_mask):
        out        = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden     = out.hidden_states
        num_layers = len(hidden)
        base       = self.pool_layers[NUM_LEVELS](hidden[-1], attention_mask)
        level_indices = [
            min((lvl + 1) * LAYERS_LEVEL, num_layers - 1) for lvl in range(NUM_LEVELS)
        ]
        level_cls = [
            self.pool_layers[lvl](hidden[level_indices[lvl]], attention_mask)
            for lvl in range(NUM_LEVELS)
        ]
        residuals = []
        for lvl in range(NUM_LEVELS):
            diff = level_cls[lvl] - level_cls[lvl + 1] if lvl < NUM_LEVELS - 1 else level_cls[lvl]
            r    = self.lras[lvl](diff)
            residuals.append(self.level_norms[lvl](r))
        recon = self.fusion(base, residuals)
        return recon

    def forward(self, input_ids, attention_mask, cls_embed=None):
        if cls_embed is not None:
            return self.classifier(cls_embed)
        recon = self.encode(input_ids, attention_mask)
        return self.classifier(recon)


def mixup_embeddings(embed, labels, alpha=MIXUP_ALPHA):
    lam   = np.random.beta(alpha, alpha)
    idx   = torch.randperm(embed.size(0), device=embed.device)
    mixed = lam * embed + (1 - lam) * embed[idx]
    return mixed, labels, labels[idx], lam


def rdrop_kl_loss(logits1, logits2):
    p = F.log_softmax(logits1, dim=-1)
    q = F.log_softmax(logits2, dim=-1)
    return (F.kl_div(p, q.exp(), reduction='batchmean') +
            F.kl_div(q, p.exp(), reduction='batchmean')) / 2


def mixup_ce_loss(logits, labels_a, labels_b, lam, weight=None):
    loss_fn = nn.CrossEntropyLoss(weight=weight, label_smoothing=0.05, reduction='none')
    return (lam * loss_fn(logits, labels_a) + (1 - lam) * loss_fn(logits, labels_b)).mean()


def compute_metrics(labels, preds, probs):
    labels = np.array(labels)
    preds  = np.array(preds)
    probs  = np.array(probs)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        'Accuracy':      accuracy_score(labels, preds),
        'Balanced_Acc':  balanced_accuracy_score(labels, preds),
        'Precision':     precision_score(labels, preds, zero_division=0),
        'Recall':        recall_score(labels, preds, zero_division=0),
        'F1':            f1_score(labels, preds, zero_division=0),
        'ROC_AUC':       roc_auc_score(labels, probs),
        'Avg_Precision': average_precision_score(labels, probs),
        'MCC':           matthews_corrcoef(labels, preds),
        'Specificity':   tn / (tn + fp + 1e-9),
        'Sensitivity':   tp / (tp + fn + 1e-9),
        'FPR':           fp / (fp + tn + 1e-9),
        'FNR':           fn / (fn + tp + 1e-9),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn)
    }


def train_epoch(model, loader, criterion, optimizer, scheduler, class_weights, step_counter):
    model.train()
    raw_model                        = model.module if isinstance(model, nn.DataParallel) else model
    total_loss                       = 0.0
    all_labels, all_preds, all_probs = [], [], []
    optimizer.zero_grad()
    for batch_idx, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)
        with torch.no_grad():
            embed = raw_model.encode(input_ids, attention_mask)
        use_mixup = (np.random.rand() < MIXUP_PROB)
        if use_mixup:
            mixed_embed, la, lb, lam = mixup_embeddings(embed, labels)
            logits1 = model(None, None, cls_embed=mixed_embed)
            logits2 = model(None, None, cls_embed=mixed_embed)
            ce_loss = mixup_ce_loss(logits1, la, lb, lam, weight=class_weights)
        else:
            logits1 = model(None, None, cls_embed=embed)
            logits2 = model(None, None, cls_embed=embed)
            ce_loss = criterion(logits1, labels)
        kl_loss = rdrop_kl_loss(logits1, logits2)
        loss    = ce_loss + RDROP_ALPHA * kl_loss
        loss    = loss / GRAD_ACCUM
        loss.backward()
        if (batch_idx + 1) % GRAD_ACCUM == 0 or (batch_idx + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(
                [p for p in raw_model.parameters() if p.requires_grad], 1.0
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            step_counter[0] += 1
        total_loss += (ce_loss + RDROP_ALPHA * kl_loss).item()
        with torch.no_grad():
            probs = torch.softmax(logits1, dim=-1)[:, 1].cpu().numpy()
            preds = logits1.argmax(dim=-1).cpu().numpy()
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)
    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss                       = 0.0
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['label'].to(DEVICE)
            logits         = model(input_ids, attention_mask)
            loss           = criterion(logits, labels)
            total_loss    += loss.item()
            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            preds = logits.argmax(dim=-1).cpu().numpy()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds)
            all_probs.extend(probs)
    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)


def update_swa(swa_model, raw_model):
    with torch.no_grad():
        for swa_p, p in zip(swa_model.parameters(), raw_model.parameters()):
            swa_p.data.copy_(swa_p.data + p.data)


def apply_swa_avg(swa_model, n_swa):
    with torch.no_grad():
        for p in swa_model.parameters():
            p.data.div_(n_swa)


def plot_training_curves(history, save_path='training_curves.png'):
    epochs = list(range(1, len(history['tr_loss']) + 1))
    fig, axes = plt.subplots(3, 3, figsize=(18, 14))
    fig.suptitle('LAPPEFT (CodeT5+) Training Curves', fontsize=16, fontweight='bold')

    def plot_ax(ax, key, title, ylabel):
        ax.plot(epochs, history[f'tr_{key}'], 'b-o', markersize=3, label='Train')
        ax.plot(epochs, history[f'va_{key}'], 'r-o', markersize=3, label='Val')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(True, alpha=0.3)

    plot_ax(axes[0][0], 'loss',    'Loss',          'Loss')
    plot_ax(axes[0][1], 'acc',     'Accuracy',      'Accuracy')
    plot_ax(axes[0][2], 'f1',      'F1 Score',      'F1')
    plot_ax(axes[1][0], 'auc',     'ROC AUC',       'AUC')
    plot_ax(axes[1][1], 'mcc',     'MCC',           'MCC')
    plot_ax(axes[1][2], 'prec',    'Precision',     'Precision')
    plot_ax(axes[2][0], 'rec',     'Recall',        'Recall')
    plot_ax(axes[2][1], 'bacc',    'Balanced Acc',  'Balanced Accuracy')
    plot_ax(axes[2][2], 'avgprec', 'Avg Precision', 'Avg Precision')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Training curves saved to {save_path}")


def plot_loss_curve(history, save_path='train_val_loss_curve.png'):
    epochs = list(range(1, len(history['tr_loss']) + 1))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(epochs, history['tr_loss'], 'b-o', markersize=4, linewidth=1.8, label='Train Loss')
    ax.plot(epochs, history['va_loss'], 'r-o', markersize=4, linewidth=1.8, label='Val Loss')
    ax.set_title('LAPPEFT (CodeT5+) — Train vs Validation Loss per Epoch', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Train/Val loss curve saved to {save_path}")


def plot_final_comparison(best_test_m, swa_test_m, save_path='test_comparison.png'):
    keys      = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Avg_Precision']
    x         = np.arange(len(keys))
    w         = 0.35
    best_vals = [best_test_m[k] for k in keys]
    swa_vals  = [swa_test_m[k] for k in keys] if swa_test_m else None
    fig, ax   = plt.subplots(figsize=(14, 6))
    bars1     = ax.bar(x - w/2, best_vals, w, label='Best Checkpoint', color='steelblue', alpha=0.85)
    if swa_vals:
        bars2 = ax.bar(x + w/2, swa_vals, w, label='SWA Model', color='darkorange', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(keys, rotation=20, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title('Test Set: Best Checkpoint vs SWA Model', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
    if swa_vals:
        for bar in bars2:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Test comparison chart saved to {save_path}")


def plot_confusion_matrix(m, title, save_path):
    cm      = np.array([[m['TN'], m['FP']], [m['FN'], m['TP']]])
    fig, ax = plt.subplots(figsize=(5, 4))
    im      = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred 0', 'Pred 1'])
    ax.set_yticklabels(['True 0', 'True 1'])
    thresh = cm.max() / 2.0
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > thresh else 'black',
                    fontsize=14, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Confusion matrix saved to {save_path}")


def plot_metrics_radar(best_test_m, swa_test_m, save_path='radar_chart.png'):
    keys   = ['Accuracy', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Specificity', 'Sensitivity']
    n      = len(keys)
    angles = [x / float(n) * 2 * np.pi for x in range(n)]
    angles += angles[:1]
    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

    def draw(m, color, label):
        vals  = [m[k] for k in keys]
        vals += vals[:1]
        ax.plot(angles, vals, color=color, linewidth=2, label=label)
        ax.fill(angles, vals, color=color, alpha=0.15)

    draw(best_test_m, 'steelblue', 'Best Checkpoint')
    if swa_test_m:
        draw(swa_test_m, 'darkorange', 'SWA Model')
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(keys, size=9)
    ax.set_ylim(0, 1)
    ax.set_title('Test Metrics Radar', fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Radar chart saved to {save_path}")


def main():
    train_full = pd.read_csv(TRAIN_CSV)
    test_full  = pd.read_csv(TEST_CSV)
    train_df, val_df = train_test_split(
        train_full, test_size=VAL_SPLIT, random_state=SEED, stratify=train_full['label']
    )
    _, test_df = train_test_split(
        test_full, test_size=0.50, random_state=SEED, stratify=test_full['label']
    )
    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
    print(f"Device: {DEVICE}  |  GPUs available: {num_gpus}  |  num_workers: {NUM_WORKERS}  |  pin_memory: {PIN_MEMORY}")
    if num_gpus > 1:
        for i in range(num_gpus):
            print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
    print()
    tokenizer    = AutoTokenizer.from_pretrained(MODEL_NAME)
    train_ds     = CodeDataset(train_df.reset_index(drop=True), tokenizer)
    val_ds       = CodeDataset(val_df.reset_index(drop=True),   tokenizer)
    test_ds      = CodeDataset(test_df.reset_index(drop=True),  tokenizer)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    counts        = train_df['label'].value_counts().sort_index().values
    raw_w         = torch.tensor([1.0 / c for c in counts], dtype=torch.float).to(DEVICE)
    class_weights = raw_w / raw_w.sum() * len(counts)
    raw_model = LAPPEFT().to(DEVICE)
    swa_model = copy.deepcopy(raw_model)
    for p in swa_model.parameters():
        p.requires_grad = False
    if num_gpus > 1:
        model = nn.DataParallel(raw_model, device_ids=list(range(num_gpus)))
    else:
        model = raw_model
    total_p   = sum(p.numel() for p in raw_model.parameters())
    trainable = sum(p.numel() for p in raw_model.parameters() if p.requires_grad)
    print(f"Total Params     : {total_p:,}")
    print(f"Trainable Params : {trainable:,}")
    print(f"Trainable %      : {100.0 * trainable / total_p:.4f}%\n")
    criterion        = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
    trainable_params = [p for p in raw_model.parameters() if p.requires_grad]
    optimizer        = torch.optim.AdamW(
        trainable_params, lr=LR, weight_decay=0.05,
        betas=(0.9, 0.999), eps=1e-8
    )
    effective_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    warmup_steps    = int(WARMUP_RATIO * effective_steps)
    scheduler       = get_cosine_schedule_with_warmup(optimizer, warmup_steps, effective_steps)
    swa_start_epoch = int(SWA_START_RATIO * EPOCHS)
    n_swa           = 0
    step_counter    = [0]
    best_val_f1     = -1.0
    best_epoch      = 0
    best_state      = None
    no_improve      = 0
    history = {
        'tr_loss': [], 'tr_acc': [], 'tr_f1': [], 'tr_auc': [], 'tr_mcc': [],
        'tr_prec': [], 'tr_rec': [], 'tr_bacc': [], 'tr_avgprec': [],
        'va_loss': [], 'va_acc': [], 'va_f1': [], 'va_auc': [], 'va_mcc': [],
        'va_prec': [], 'va_rec': [], 'va_bacc': [], 'va_avgprec': []
    }
    hdr = (f"{'Ep':>3} | {'TrLoss':>8} | {'TrAcc':>7} | {'TrF1':>7} | {'TrAUC':>7} | {'TrMCC':>7} | "
           f"{'VaLoss':>8} | {'VaAcc':>7} | {'VaF1':>7} | {'VaAUC':>7} | {'VaMCC':>7} | {'Note':<12}")
    sep = "-" * len(hdr)
    print(sep)
    print(hdr)
    print(sep)
    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_m = train_epoch(model, train_loader, criterion, optimizer,
                                    scheduler, class_weights, step_counter)
        va_loss, va_m = eval_epoch(model, val_loader, criterion)
        history['tr_loss'].append(tr_loss)
        history['tr_acc'].append(tr_m['Accuracy'])
        history['tr_f1'].append(tr_m['F1'])
        history['tr_auc'].append(tr_m['ROC_AUC'])
        history['tr_mcc'].append(tr_m['MCC'])
        history['tr_prec'].append(tr_m['Precision'])
        history['tr_rec'].append(tr_m['Recall'])
        history['tr_bacc'].append(tr_m['Balanced_Acc'])
        history['tr_avgprec'].append(tr_m['Avg_Precision'])
        history['va_loss'].append(va_loss)
        history['va_acc'].append(va_m['Accuracy'])
        history['va_f1'].append(va_m['F1'])
        history['va_auc'].append(va_m['ROC_AUC'])
        history['va_mcc'].append(va_m['MCC'])
        history['va_prec'].append(va_m['Precision'])
        history['va_rec'].append(va_m['Recall'])
        history['va_bacc'].append(va_m['Balanced_Acc'])
        history['va_avgprec'].append(va_m['Avg_Precision'])
        note = ""
        if epoch > swa_start_epoch:
            update_swa(swa_model, raw_model)
            n_swa += 1
            note += "SWA "
        is_best = va_m['F1'] > best_val_f1
        if is_best:
            best_val_f1 = va_m['F1']
            best_epoch  = epoch
            best_state  = {k: v.cpu().clone() for k, v in raw_model.state_dict().items()}
            no_improve  = 0
            note       += "<--"
        else:
            no_improve += 1
        print(
            f"{epoch:>3} | {tr_loss:>8.4f} | {tr_m['Accuracy']:>7.4f} | {tr_m['F1']:>7.4f} | "
            f"{tr_m['ROC_AUC']:>7.4f} | {tr_m['MCC']:>7.4f} | "
            f"{va_loss:>8.4f} | {va_m['Accuracy']:>7.4f} | {va_m['F1']:>7.4f} | "
            f"{va_m['ROC_AUC']:>7.4f} | {va_m['MCC']:>7.4f} | {note:<12}"
        )
        if no_improve >= PATIENCE:
            print(f"\n  Early stopping at epoch {epoch} (no val F1 improvement for {PATIENCE} epochs)")
            break
    print(sep)
    print(f"\nBest Epoch: {best_epoch}  |  Best Val F1: {best_val_f1:.4f}")
    print("\n--- Evaluating: Best Checkpoint Model ---")
    raw_model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    _, best_test_m = eval_epoch(model, test_loader, criterion)
    print("\n--- Evaluating: SWA Model ---")
    swa_test_m = None
    if n_swa > 0:
        apply_swa_avg(swa_model, n_swa)
        swa_model.to(DEVICE)
        _, swa_test_m = eval_epoch(swa_model, test_loader, criterion)
    print(f"\n{'='*60}")
    print(f"  Val F1 trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_f1'][-5:]]}")
    print(f"  Val Loss trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_loss'][-5:]]}")
    print(f"\n{'='*60}")
    print(f"  TEST RESULTS — Best Checkpoint (Epoch {best_epoch})")
    print(f"{'='*60}")
    scalar_keys = [k for k in best_test_m if k not in ('TP', 'TN', 'FP', 'FN')]
    for k in scalar_keys:
        print(f"  {k:<22}: {best_test_m[k]:.4f}")
    print(f"  {'TP':<22}: {best_test_m['TP']}")
    print(f"  {'TN':<22}: {best_test_m['TN']}")
    print(f"  {'FP':<22}: {best_test_m['FP']}")
    print(f"  {'FN':<22}: {best_test_m['FN']}")
    if swa_test_m is not None:
        print(f"\n{'='*60}")
        print(f"  TEST RESULTS — SWA Model (averaged over last {n_swa} epochs)")
        print(f"{'='*60}")
        for k in scalar_keys:
            print(f"  {k:<22}: {swa_test_m[k]:.4f}")
        print(f"  {'TP':<22}: {swa_test_m['TP']}")
        print(f"  {'TN':<22}: {swa_test_m['TN']}")
        print(f"  {'FP':<22}: {swa_test_m['FP']}")
        print(f"  {'FN':<22}: {swa_test_m['FN']}")
        print(f"\n{'='*60}")
        print(f"  COMPARISON: Best Checkpoint vs SWA")
        print(f"{'='*60}")
        compare_keys = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall']
        print(f"  {'Metric':<22} {'BestCkpt':>12} {'SWA':>12} {'Delta':>12}")
        print(f"  {'-'*58}")
        for k in compare_keys:
            delta  = swa_test_m[k] - best_test_m[k]
            symbol = "+" if delta >= 0 else ""
            print(f"  {k:<22} {best_test_m[k]:>12.4f} {swa_test_m[k]:>12.4f} {symbol+f'{delta:.4f}':>12}")
    plot_training_curves(history,  save_path='training_curves.png')
    plot_loss_curve(history,       save_path='train_val_loss_curve.png')
    plot_final_comparison(best_test_m, swa_test_m, save_path='test_comparison.png')
    plot_confusion_matrix(best_test_m, f'Confusion Matrix — Best Ckpt (Ep {best_epoch})', 'cm_best.png')
    if swa_test_m:
        plot_confusion_matrix(swa_test_m, f'Confusion Matrix — SWA ({n_swa} epochs)', 'cm_swa.png')
    plot_metrics_radar(best_test_m, swa_test_m, save_path='radar_chart.png')
    print("\nAll figures saved.")


if __name__ == "__main__":
    main()

# HierAdapter codet5+ 770M, 1024 token code vulnerability

In [ ]:
import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_cosine_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix,
    balanced_accuracy_score, average_precision_score
)
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

MODEL_NAME      = "Salesforce/codet5p-770m"
TRAIN_CSV       = 'dev1024/trqw.csv'
TEST_CSV        = 'dev1024/teqw.csv'
MAX_LENGTH      = 1024
D_MODEL         = 1024
BATCH_SIZE      = 16
GRAD_ACCUM      = 2
EPOCHS          = 30
LR              = 5e-5
WARMUP_RATIO    = 0.15
BOTTLENECK      = 6
NUM_LEVELS      = 4
LAYERS_LEVEL    = 3
VAL_SPLIT       = 0.20
SEED            = 42
RDROP_ALPHA     = 0.3
MIXUP_ALPHA     = 0.2
MIXUP_PROB      = 0.5
SWA_START_RATIO = 0.6
SWA_LR          = 1e-5
PATIENCE        = 7
if torch.cuda.is_available():
    DEVICE      = torch.device("cuda:0")
    NUM_WORKERS = 4
    PIN_MEMORY  = True
elif torch.backends.mps.is_available():
    DEVICE      = torch.device("mps")
    NUM_WORKERS = 0
    PIN_MEMORY  = False
else:
    DEVICE      = torch.device("cpu")
    NUM_WORKERS = 0
    PIN_MEMORY  = False

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


class CodeDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.codes     = df['func_cleaned'].astype(str).tolist()
        self.labels    = df['label'].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.codes[idx],
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


class AttentionPooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attn = nn.Linear(d_model, 1)

    def forward(self, hidden, attention_mask):
        scores  = self.attn(hidden).squeeze(-1)
        scores  = scores.masked_fill(attention_mask == 0, -1e9)
        weights = torch.softmax(scores, dim=-1)
        return (hidden * weights.unsqueeze(-1)).sum(1)


class DynamicAlphaFusion(nn.Module):
    def __init__(self, d_model, num_levels):
        super().__init__()
        self.gate = nn.Linear(d_model, num_levels)

    def forward(self, base_cls, residuals):
        alphas   = torch.softmax(self.gate(base_cls), dim=-1)
        weighted = sum(alphas[:, i:i+1] * residuals[i] for i in range(len(residuals)))
        return base_cls + weighted


class LaplacianResidualAdapter(nn.Module):
    def __init__(self, d_model, bottleneck_dim):
        super().__init__()
        self.down    = nn.Linear(d_model, bottleneck_dim)
        self.act     = nn.GELU()
        self.up      = nn.Linear(bottleneck_dim, d_model)
        self.dropout = nn.Dropout(0.1)
        nn.init.kaiming_normal_(self.down.weight, nonlinearity='linear')
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return self.up(self.dropout(self.act(self.down(x))))


class LAPPEFT(nn.Module):
    def __init__(self):
        super().__init__()
        base_model   = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
        self.encoder = base_model.encoder
        del base_model
        for p in self.encoder.parameters():
            p.requires_grad = False

        self.pool_layers = nn.ModuleList([
            AttentionPooling(D_MODEL) for _ in range(NUM_LEVELS + 1)
        ])
        self.lras        = nn.ModuleList([
            LaplacianResidualAdapter(D_MODEL, BOTTLENECK) for _ in range(NUM_LEVELS)
        ])
        self.level_norms = nn.ModuleList([nn.LayerNorm(D_MODEL) for _ in range(NUM_LEVELS)])
        self.fusion      = DynamicAlphaFusion(D_MODEL, NUM_LEVELS)
        self.classifier  = nn.Sequential(
            nn.Linear(D_MODEL, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )

    def encode(self, input_ids, attention_mask):
        out        = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden     = out.hidden_states
        num_layers = len(hidden)
        base       = self.pool_layers[NUM_LEVELS](hidden[-1], attention_mask)
        level_indices = [
            min((lvl + 1) * LAYERS_LEVEL, num_layers - 1) for lvl in range(NUM_LEVELS)
        ]
        level_cls  = [
            self.pool_layers[lvl](hidden[level_indices[lvl]], attention_mask)
            for lvl in range(NUM_LEVELS)
        ]
        residuals  = []
        for lvl in range(NUM_LEVELS):
            diff = level_cls[lvl] - level_cls[lvl + 1] if lvl < NUM_LEVELS - 1 else level_cls[lvl]
            r    = self.lras[lvl](diff)
            residuals.append(self.level_norms[lvl](r))
        recon = self.fusion(base, residuals)
        return recon

    def forward(self, input_ids, attention_mask, cls_embed=None, return_embed=False):
        if cls_embed is not None:
            return self.classifier(cls_embed)
        recon = self.encode(input_ids, attention_mask)
        if return_embed:
            return recon
        return self.classifier(recon)


def mixup_embeddings(embed, labels, alpha=MIXUP_ALPHA):
    lam   = np.random.beta(alpha, alpha)
    idx   = torch.randperm(embed.size(0), device=embed.device)
    mixed = lam * embed + (1 - lam) * embed[idx]
    return mixed, labels, labels[idx], lam


def rdrop_kl_loss(logits1, logits2):
    p = F.log_softmax(logits1, dim=-1)
    q = F.log_softmax(logits2, dim=-1)
    return (F.kl_div(p, q.exp(), reduction='batchmean') +
            F.kl_div(q, p.exp(), reduction='batchmean')) / 2


def mixup_ce_loss(logits, labels_a, labels_b, lam, weight=None):
    loss_fn = nn.CrossEntropyLoss(weight=weight, label_smoothing=0.05, reduction='none')
    return (lam * loss_fn(logits, labels_a) + (1 - lam) * loss_fn(logits, labels_b)).mean()


def compute_metrics(labels, preds, probs):
    labels = np.array(labels)
    preds  = np.array(preds)
    probs  = np.array(probs)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        'Accuracy':      accuracy_score(labels, preds),
        'Balanced_Acc':  balanced_accuracy_score(labels, preds),
        'Precision':     precision_score(labels, preds, zero_division=0),
        'Recall':        recall_score(labels, preds, zero_division=0),
        'F1':            f1_score(labels, preds, zero_division=0),
        'ROC_AUC':       roc_auc_score(labels, probs),
        'Avg_Precision': average_precision_score(labels, probs),
        'MCC':           matthews_corrcoef(labels, preds),
        'Specificity':   tn / (tn + fp + 1e-9),
        'Sensitivity':   tp / (tp + fn + 1e-9),
        'FPR':           fp / (fp + tn + 1e-9),
        'FNR':           fn / (fn + tp + 1e-9),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn)
    }


def train_epoch(model, loader, criterion, optimizer, scheduler, class_weights, step_counter):
    model.train()
    total_loss                       = 0.0
    all_labels, all_preds, all_probs = [], [], []
    optimizer.zero_grad()

    for batch_idx, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)

        with torch.no_grad():
            embed = model(input_ids, attention_mask, return_embed=True)

        use_mixup = (np.random.rand() < MIXUP_PROB)

        if use_mixup:
            mixed_embed, la, lb, lam = mixup_embeddings(embed, labels)
            logits1 = model(None, None, cls_embed=mixed_embed)
            logits2 = model(None, None, cls_embed=mixed_embed)
            ce_loss = mixup_ce_loss(logits1, la, lb, lam, weight=class_weights)
        else:
            logits1 = model(None, None, cls_embed=embed)
            logits2 = model(None, None, cls_embed=embed)
            ce_loss = criterion(logits1, labels)

        kl_loss = rdrop_kl_loss(logits1, logits2)
        loss    = ce_loss + RDROP_ALPHA * kl_loss
        loss    = loss / GRAD_ACCUM
        loss.backward()

        if (batch_idx + 1) % GRAD_ACCUM == 0 or (batch_idx + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            step_counter[0] += 1

        total_loss += (ce_loss + RDROP_ALPHA * kl_loss).item()

        with torch.no_grad():
            probs = torch.softmax(logits1, dim=-1)[:, 1].cpu().numpy()
            preds = logits1.argmax(dim=-1).cpu().numpy()

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)

    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss                       = 0.0
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['label'].to(DEVICE)
            logits         = model(input_ids, attention_mask)
            loss           = criterion(logits, labels)
            total_loss    += loss.item()
            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            preds = logits.argmax(dim=-1).cpu().numpy()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds)
            all_probs.extend(probs)

    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)


def update_swa(swa_model, model):
    raw = model.module if isinstance(model, nn.DataParallel) else model
    with torch.no_grad():
        for swa_p, p in zip(swa_model.parameters(), raw.parameters()):
            swa_p.data.copy_(swa_p.data + p.data.cpu())


def apply_swa_avg(swa_model, n_swa):
    with torch.no_grad():
        for p in swa_model.parameters():
            p.data.div_(n_swa)


def plot_loss_curve(history, save_path='loss_curve.png'):
    epochs = list(range(1, len(history['tr_loss']) + 1))
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(epochs, history['tr_loss'], 'b-o', markersize=4, label='Train Loss')
    ax.plot(epochs, history['va_loss'], 'r-o', markersize=4, label='Validation Loss')
    ax.set_title('LAPPEFT (CodeT5+ 770M) — Train vs Validation Loss per Epoch',
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Loss curve saved to {save_path}")


def plot_training_curves(history, save_path='training_curves.png'):
    epochs = list(range(1, len(history['tr_loss']) + 1))
    fig, axes = plt.subplots(3, 3, figsize=(18, 14))
    fig.suptitle('LAPPEFT (CodeT5+ 770M) Training Curves', fontsize=16, fontweight='bold')

    def plot_ax(ax, key, title, ylabel):
        ax.plot(epochs, history[f'tr_{key}'], 'b-o', markersize=3, label='Train')
        ax.plot(epochs, history[f'va_{key}'], 'r-o', markersize=3, label='Val')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(True, alpha=0.3)

    plot_ax(axes[0][0], 'loss',    'Loss',          'Loss')
    plot_ax(axes[0][1], 'acc',     'Accuracy',      'Accuracy')
    plot_ax(axes[0][2], 'f1',      'F1 Score',      'F1')
    plot_ax(axes[1][0], 'auc',     'ROC AUC',       'AUC')
    plot_ax(axes[1][1], 'mcc',     'MCC',           'MCC')
    plot_ax(axes[1][2], 'prec',    'Precision',     'Precision')
    plot_ax(axes[2][0], 'rec',     'Recall',        'Recall')
    plot_ax(axes[2][1], 'bacc',    'Balanced Acc',  'Balanced Accuracy')
    plot_ax(axes[2][2], 'avgprec', 'Avg Precision', 'Avg Precision')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Training curves saved to {save_path}")


def plot_final_comparison(best_test_m, swa_test_m, save_path='test_comparison.png'):
    keys      = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Avg_Precision']
    x         = np.arange(len(keys))
    w         = 0.35
    best_vals = [best_test_m[k] for k in keys]
    swa_vals  = [swa_test_m[k] for k in keys] if swa_test_m else None

    fig, ax = plt.subplots(figsize=(14, 6))
    bars1   = ax.bar(x - w/2, best_vals, w, label='Best Checkpoint', color='steelblue', alpha=0.85)
    if swa_vals:
        bars2 = ax.bar(x + w/2, swa_vals, w, label='SWA Model', color='darkorange', alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(keys, rotation=20, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title('Test Set: Best Checkpoint vs SWA Model', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
    if swa_vals:
        for bar in bars2:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Test comparison chart saved to {save_path}")


def plot_confusion_matrix(m, title, save_path):
    cm      = np.array([[m['TN'], m['FP']], [m['FN'], m['TP']]])
    fig, ax = plt.subplots(figsize=(5, 4))
    im      = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred 0', 'Pred 1'])
    ax.set_yticklabels(['True 0', 'True 1'])
    thresh = cm.max() / 2.0
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > thresh else 'black',
                    fontsize=14, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Confusion matrix saved to {save_path}")


def plot_metrics_radar(best_test_m, swa_test_m, save_path='radar_chart.png'):
    keys   = ['Accuracy', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Specificity', 'Sensitivity']
    n      = len(keys)
    angles = [x / float(n) * 2 * np.pi for x in range(n)]
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

    def draw(m, color, label):
        vals  = [m[k] for k in keys]
        vals += vals[:1]
        ax.plot(angles, vals, color=color, linewidth=2, label=label)
        ax.fill(angles, vals, color=color, alpha=0.15)

    draw(best_test_m, 'steelblue', 'Best Checkpoint')
    if swa_test_m:
        draw(swa_test_m, 'darkorange', 'SWA Model')

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(keys, size=9)
    ax.set_ylim(0, 1)
    ax.set_title('Test Metrics Radar', fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Radar chart saved to {save_path}")


def main():
    train_full = pd.read_csv(TRAIN_CSV)
    test_full  = pd.read_csv(TEST_CSV)

    train_df, val_df = train_test_split(
        train_full, test_size=VAL_SPLIT, random_state=SEED, stratify=train_full['label']
    )
    _, test_df = train_test_split(
        test_full, test_size=0.50, random_state=SEED, stratify=test_full['label']
    )

    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    print(f"Device: {DEVICE}")
    num_gpus = torch.cuda.device_count()
    print(f"Number of GPUs available: {num_gpus}\n")

    tokenizer    = AutoTokenizer.from_pretrained(MODEL_NAME)
    train_ds     = CodeDataset(train_df.reset_index(drop=True), tokenizer)
    val_ds       = CodeDataset(val_df.reset_index(drop=True),   tokenizer)
    test_ds      = CodeDataset(test_df.reset_index(drop=True),  tokenizer)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    counts        = train_df['label'].value_counts().sort_index().values
    raw_w         = torch.tensor([1.0 / c for c in counts], dtype=torch.float).to(DEVICE)
    class_weights = raw_w / raw_w.sum() * len(counts)

    model = LAPPEFT().to(DEVICE)
    if num_gpus > 1:
        model = nn.DataParallel(model, device_ids=list(range(num_gpus)))
        print(f"Using {num_gpus} GPUs via DataParallel: {list(range(num_gpus))}")

    raw_model = model.module if isinstance(model, nn.DataParallel) else model
    swa_model = copy.deepcopy(raw_model)
    for p in swa_model.parameters():
        p.requires_grad = False

    total_p   = sum(p.numel() for p in raw_model.parameters())
    trainable = sum(p.numel() for p in raw_model.parameters() if p.requires_grad)
    print(f"Total Params     : {total_p:,}")
    print(f"Trainable Params : {trainable:,}")
    print(f"Trainable %      : {100.0 * trainable / total_p:.4f}%\n")

    criterion        = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
    trainable_params = [p for p in raw_model.parameters() if p.requires_grad]
    optimizer        = torch.optim.AdamW(
        trainable_params, lr=LR, weight_decay=0.05, betas=(0.9, 0.999), eps=1e-8
    )

    effective_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    warmup_steps    = int(WARMUP_RATIO * effective_steps)
    scheduler       = get_cosine_schedule_with_warmup(optimizer, warmup_steps, effective_steps)

    swa_start_epoch = int(SWA_START_RATIO * EPOCHS)
    n_swa           = 0
    step_counter    = [0]

    best_val_f1 = -1.0
    best_epoch  = 0
    best_state  = None
    no_improve  = 0

    history = {
        'tr_loss': [], 'tr_acc': [], 'tr_f1': [], 'tr_auc': [], 'tr_mcc': [],
        'tr_prec': [], 'tr_rec': [], 'tr_bacc': [], 'tr_avgprec': [],
        'va_loss': [], 'va_acc': [], 'va_f1': [], 'va_auc': [], 'va_mcc': [],
        'va_prec': [], 'va_rec': [], 'va_bacc': [], 'va_avgprec': []
    }

    hdr = (f"{'Ep':>3} | {'TrLoss':>8} | {'TrAcc':>7} | {'TrF1':>7} | {'TrAUC':>7} | {'TrMCC':>7} | "
           f"{'VaLoss':>8} | {'VaAcc':>7} | {'VaF1':>7} | {'VaAUC':>7} | {'VaMCC':>7} | {'Note':<12}")
    sep = "-" * len(hdr)
    print(sep)
    print(hdr)
    print(sep)

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_m = train_epoch(model, train_loader, criterion, optimizer,
                                    scheduler, class_weights, step_counter)
        va_loss, va_m = eval_epoch(model, val_loader, criterion)

        history['tr_loss'].append(tr_loss)
        history['tr_acc'].append(tr_m['Accuracy'])
        history['tr_f1'].append(tr_m['F1'])
        history['tr_auc'].append(tr_m['ROC_AUC'])
        history['tr_mcc'].append(tr_m['MCC'])
        history['tr_prec'].append(tr_m['Precision'])
        history['tr_rec'].append(tr_m['Recall'])
        history['tr_bacc'].append(tr_m['Balanced_Acc'])
        history['tr_avgprec'].append(tr_m['Avg_Precision'])

        history['va_loss'].append(va_loss)
        history['va_acc'].append(va_m['Accuracy'])
        history['va_f1'].append(va_m['F1'])
        history['va_auc'].append(va_m['ROC_AUC'])
        history['va_mcc'].append(va_m['MCC'])
        history['va_prec'].append(va_m['Precision'])
        history['va_rec'].append(va_m['Recall'])
        history['va_bacc'].append(va_m['Balanced_Acc'])
        history['va_avgprec'].append(va_m['Avg_Precision'])

        note = ""
        if epoch > swa_start_epoch:
            update_swa(swa_model, model)
            n_swa += 1
            note += "SWA "

        is_best = va_m['F1'] > best_val_f1
        if is_best:
            best_val_f1 = va_m['F1']
            best_epoch  = epoch
            raw_m       = model.module if isinstance(model, nn.DataParallel) else model
            best_state  = {k: v.cpu().clone() for k, v in raw_m.state_dict().items()}
            no_improve  = 0
            note       += "<--"
        else:
            no_improve += 1

        print(
            f"{epoch:>3} | {tr_loss:>8.4f} | {tr_m['Accuracy']:>7.4f} | {tr_m['F1']:>7.4f} | "
            f"{tr_m['ROC_AUC']:>7.4f} | {tr_m['MCC']:>7.4f} | "
            f"{va_loss:>8.4f} | {va_m['Accuracy']:>7.4f} | {va_m['F1']:>7.4f} | "
            f"{va_m['ROC_AUC']:>7.4f} | {va_m['MCC']:>7.4f} | {note:<12}"
        )

        if no_improve >= PATIENCE:
            print(f"\n  Early stopping at epoch {epoch} (no val F1 improvement for {PATIENCE} epochs)")
            break

    print(sep)
    print(f"\nBest Epoch: {best_epoch}  |  Best Val F1: {best_val_f1:.4f}")

    print("\n--- Evaluating: Best Checkpoint Model ---")
    raw_m = model.module if isinstance(model, nn.DataParallel) else model
    raw_m.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    _, best_test_m = eval_epoch(model, test_loader, criterion)

    print("\n--- Evaluating: SWA Model ---")
    swa_test_m = None
    if n_swa > 0:
        apply_swa_avg(swa_model, n_swa)
        swa_model.to(DEVICE)
        _, swa_test_m = eval_epoch(swa_model, test_loader, criterion)

    print(f"\n{'='*60}")
    print(f"  Val F1 trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_f1'][-5:]]}")
    print(f"  Val Loss trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_loss'][-5:]]}")

    print(f"\n{'='*60}")
    print(f"  TEST RESULTS — Best Checkpoint (Epoch {best_epoch})")
    print(f"{'='*60}")
    scalar_keys = [k for k in best_test_m if k not in ('TP', 'TN', 'FP', 'FN')]
    for k in scalar_keys:
        print(f"  {k:<22}: {best_test_m[k]:.4f}")
    print(f"  {'TP':<22}: {best_test_m['TP']}")
    print(f"  {'TN':<22}: {best_test_m['TN']}")
    print(f"  {'FP':<22}: {best_test_m['FP']}")
    print(f"  {'FN':<22}: {best_test_m['FN']}")

    if swa_test_m is not None:
        print(f"\n{'='*60}")
        print(f"  TEST RESULTS — SWA Model (averaged over last {n_swa} epochs)")
        print(f"{'='*60}")
        for k in scalar_keys:
            print(f"  {k:<22}: {swa_test_m[k]:.4f}")
        print(f"  {'TP':<22}: {swa_test_m['TP']}")
        print(f"  {'TN':<22}: {swa_test_m['TN']}")
        print(f"  {'FP':<22}: {swa_test_m['FP']}")
        print(f"  {'FN':<22}: {swa_test_m['FN']}")

        print(f"\n{'='*60}")
        print(f"  COMPARISON: Best Checkpoint vs SWA")
        print(f"{'='*60}")
        compare_keys = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall']
        print(f"  {'Metric':<22} {'BestCkpt':>12} {'SWA':>12} {'Delta':>12}")
        print(f"  {'-'*58}")
        for k in compare_keys:
            delta  = swa_test_m[k] - best_test_m[k]
            symbol = "+" if delta >= 0 else ""
            print(f"  {k:<22} {best_test_m[k]:>12.4f} {swa_test_m[k]:>12.4f} {symbol+f'{delta:.4f}':>12}")

    plot_loss_curve(history,              save_path='loss_curve.png')
    plot_training_curves(history,         save_path='training_curves.png')
    plot_final_comparison(best_test_m, swa_test_m, save_path='test_comparison.png')
    plot_confusion_matrix(best_test_m, f'Confusion Matrix — Best Ckpt (Ep {best_epoch})', 'cm_best.png')
    if swa_test_m:
        plot_confusion_matrix(swa_test_m, f'Confusion Matrix — SWA ({n_swa} epochs)', 'cm_swa.png')
    plot_metrics_radar(best_test_m, swa_test_m, save_path='radar_chart.png')

    print("\nAll figures saved.")


if __name__ == "__main__":
    main()

# HierAdapter codet5+ 770M, 512 token code vulnerability

In [ ]:
import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_cosine_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix,
    balanced_accuracy_score, average_precision_score
)
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

MODEL_NAME      = "Salesforce/codet5p-770m"
TRAIN_CSV       = '/traincodex.csv'
TEST_CSV        = '/testcodex.csv'
MAX_LENGTH      = 512
D_MODEL         = 1024
BATCH_SIZE      = 16
GRAD_ACCUM      = 2
EPOCHS          = 30
LR              = 5e-5
WARMUP_RATIO    = 0.15
BOTTLENECK      = 6
NUM_LEVELS      = 4
LAYERS_LEVEL    = 3
VAL_SPLIT       = 0.20
SEED            = 42
RDROP_ALPHA     = 0.3
MIXUP_ALPHA     = 0.2
MIXUP_PROB      = 0.5
SWA_START_RATIO = 0.6
SWA_LR          = 1e-5
PATIENCE        = 7
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


class CodeDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.codes     = df['code'].astype(str).tolist()
        self.labels    = df['label'].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.codes[idx],
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


class AttentionPooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attn = nn.Linear(d_model, 1)

    def forward(self, hidden, attention_mask):
        scores  = self.attn(hidden).squeeze(-1)
        scores  = scores.masked_fill(attention_mask == 0, -1e9)
        weights = torch.softmax(scores, dim=-1)
        return (hidden * weights.unsqueeze(-1)).sum(1)


class DynamicAlphaFusion(nn.Module):
    def __init__(self, d_model, num_levels):
        super().__init__()
        self.gate = nn.Linear(d_model, num_levels)

    def forward(self, base_cls, residuals):
        alphas   = torch.softmax(self.gate(base_cls), dim=-1)
        weighted = sum(alphas[:, i:i+1] * residuals[i] for i in range(len(residuals)))
        return base_cls + weighted


class LaplacianResidualAdapter(nn.Module):
    def __init__(self, d_model, bottleneck_dim):
        super().__init__()
        self.down    = nn.Linear(d_model, bottleneck_dim)
        self.act     = nn.GELU()
        self.up      = nn.Linear(bottleneck_dim, d_model)
        self.dropout = nn.Dropout(0.1)
        nn.init.kaiming_normal_(self.down.weight, nonlinearity='linear')
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return self.up(self.dropout(self.act(self.down(x))))


class LAPPEFT(nn.Module):
    def __init__(self):
        super().__init__()
        base_model   = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
        self.encoder = base_model.encoder
        del base_model
        for p in self.encoder.parameters():
            p.requires_grad = False

        self.pool_layers = nn.ModuleList([
            AttentionPooling(D_MODEL) for _ in range(NUM_LEVELS + 1)
        ])
        self.lras        = nn.ModuleList([
            LaplacianResidualAdapter(D_MODEL, BOTTLENECK) for _ in range(NUM_LEVELS)
        ])
        self.level_norms = nn.ModuleList([nn.LayerNorm(D_MODEL) for _ in range(NUM_LEVELS)])
        self.fusion      = DynamicAlphaFusion(D_MODEL, NUM_LEVELS)
        self.classifier  = nn.Sequential(
            nn.Linear(D_MODEL, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )

    def encode(self, input_ids, attention_mask):
        out        = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden     = out.hidden_states
        num_layers = len(hidden)
        base       = self.pool_layers[NUM_LEVELS](hidden[-1], attention_mask)
        level_indices = [
            min((lvl + 1) * LAYERS_LEVEL, num_layers - 1) for lvl in range(NUM_LEVELS)
        ]
        level_cls  = [
            self.pool_layers[lvl](hidden[level_indices[lvl]], attention_mask)
            for lvl in range(NUM_LEVELS)
        ]
        residuals  = []
        for lvl in range(NUM_LEVELS):
            diff = level_cls[lvl] - level_cls[lvl + 1] if lvl < NUM_LEVELS - 1 else level_cls[lvl]
            r    = self.lras[lvl](diff)
            residuals.append(self.level_norms[lvl](r))
        recon = self.fusion(base, residuals)
        return recon

    def forward(self, input_ids, attention_mask, cls_embed=None, return_embed=False):
        if cls_embed is not None:
            return self.classifier(cls_embed)
        recon = self.encode(input_ids, attention_mask)
        if return_embed:
            return recon
        return self.classifier(recon)


def mixup_embeddings(embed, labels, alpha=MIXUP_ALPHA):
    lam   = np.random.beta(alpha, alpha)
    idx   = torch.randperm(embed.size(0), device=embed.device)
    mixed = lam * embed + (1 - lam) * embed[idx]
    return mixed, labels, labels[idx], lam


def rdrop_kl_loss(logits1, logits2):
    p = F.log_softmax(logits1, dim=-1)
    q = F.log_softmax(logits2, dim=-1)
    return (F.kl_div(p, q.exp(), reduction='batchmean') +
            F.kl_div(q, p.exp(), reduction='batchmean')) / 2


def mixup_ce_loss(logits, labels_a, labels_b, lam, weight=None):
    loss_fn = nn.CrossEntropyLoss(weight=weight, label_smoothing=0.05, reduction='none')
    return (lam * loss_fn(logits, labels_a) + (1 - lam) * loss_fn(logits, labels_b)).mean()


def compute_metrics(labels, preds, probs):
    labels = np.array(labels)
    preds  = np.array(preds)
    probs  = np.array(probs)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        'Accuracy':      accuracy_score(labels, preds),
        'Balanced_Acc':  balanced_accuracy_score(labels, preds),
        'Precision':     precision_score(labels, preds, zero_division=0),
        'Recall':        recall_score(labels, preds, zero_division=0),
        'F1':            f1_score(labels, preds, zero_division=0),
        'ROC_AUC':       roc_auc_score(labels, probs),
        'Avg_Precision': average_precision_score(labels, probs),
        'MCC':           matthews_corrcoef(labels, preds),
        'Specificity':   tn / (tn + fp + 1e-9),
        'Sensitivity':   tp / (tp + fn + 1e-9),
        'FPR':           fp / (fp + tn + 1e-9),
        'FNR':           fn / (fn + tp + 1e-9),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn)
    }


def train_epoch(model, loader, criterion, optimizer, scheduler, class_weights, step_counter):
    model.train()
    total_loss                       = 0.0
    all_labels, all_preds, all_probs = [], [], []
    optimizer.zero_grad()

    for batch_idx, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)

        with torch.no_grad():
            embed = model(input_ids, attention_mask, return_embed=True)

        use_mixup = (np.random.rand() < MIXUP_PROB)

        if use_mixup:
            mixed_embed, la, lb, lam = mixup_embeddings(embed, labels)
            logits1 = model(None, None, cls_embed=mixed_embed)
            logits2 = model(None, None, cls_embed=mixed_embed)
            ce_loss = mixup_ce_loss(logits1, la, lb, lam, weight=class_weights)
        else:
            logits1 = model(None, None, cls_embed=embed)
            logits2 = model(None, None, cls_embed=embed)
            ce_loss = criterion(logits1, labels)

        kl_loss = rdrop_kl_loss(logits1, logits2)
        loss    = ce_loss + RDROP_ALPHA * kl_loss
        loss    = loss / GRAD_ACCUM
        loss.backward()

        if (batch_idx + 1) % GRAD_ACCUM == 0 or (batch_idx + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            step_counter[0] += 1

        total_loss += (ce_loss + RDROP_ALPHA * kl_loss).item()

        with torch.no_grad():
            probs = torch.softmax(logits1, dim=-1)[:, 1].cpu().numpy()
            preds = logits1.argmax(dim=-1).cpu().numpy()

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)

    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss                       = 0.0
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['label'].to(DEVICE)
            logits         = model(input_ids, attention_mask)
            loss           = criterion(logits, labels)
            total_loss    += loss.item()
            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            preds = logits.argmax(dim=-1).cpu().numpy()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds)
            all_probs.extend(probs)

    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)


def update_swa(swa_model, model):
    raw = model.module if isinstance(model, nn.DataParallel) else model
    with torch.no_grad():
        for swa_p, p in zip(swa_model.parameters(), raw.parameters()):
            swa_p.data.copy_(swa_p.data + p.data.cpu())


def apply_swa_avg(swa_model, n_swa):
    with torch.no_grad():
        for p in swa_model.parameters():
            p.data.div_(n_swa)


def plot_loss_curve(history, save_path='loss_curve.png'):
    epochs = list(range(1, len(history['tr_loss']) + 1))
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(epochs, history['tr_loss'], 'b-o', markersize=4, label='Train Loss')
    ax.plot(epochs, history['va_loss'], 'r-o', markersize=4, label='Validation Loss')
    ax.set_title('LAPPEFT (CodeT5+ 770M) — Train vs Validation Loss per Epoch',
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Loss curve saved to {save_path}")


def plot_training_curves(history, save_path='training_curves.png'):
    epochs = list(range(1, len(history['tr_loss']) + 1))
    fig, axes = plt.subplots(3, 3, figsize=(18, 14))
    fig.suptitle('LAPPEFT (CodeT5+ 770M) Training Curves', fontsize=16, fontweight='bold')

    def plot_ax(ax, key, title, ylabel):
        ax.plot(epochs, history[f'tr_{key}'], 'b-o', markersize=3, label='Train')
        ax.plot(epochs, history[f'va_{key}'], 'r-o', markersize=3, label='Val')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(True, alpha=0.3)

    plot_ax(axes[0][0], 'loss',    'Loss',          'Loss')
    plot_ax(axes[0][1], 'acc',     'Accuracy',      'Accuracy')
    plot_ax(axes[0][2], 'f1',      'F1 Score',      'F1')
    plot_ax(axes[1][0], 'auc',     'ROC AUC',       'AUC')
    plot_ax(axes[1][1], 'mcc',     'MCC',           'MCC')
    plot_ax(axes[1][2], 'prec',    'Precision',     'Precision')
    plot_ax(axes[2][0], 'rec',     'Recall',        'Recall')
    plot_ax(axes[2][1], 'bacc',    'Balanced Acc',  'Balanced Accuracy')
    plot_ax(axes[2][2], 'avgprec', 'Avg Precision', 'Avg Precision')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Training curves saved to {save_path}")


def plot_final_comparison(best_test_m, swa_test_m, save_path='test_comparison.png'):
    keys      = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Avg_Precision']
    x         = np.arange(len(keys))
    w         = 0.35
    best_vals = [best_test_m[k] for k in keys]
    swa_vals  = [swa_test_m[k] for k in keys] if swa_test_m else None

    fig, ax = plt.subplots(figsize=(14, 6))
    bars1   = ax.bar(x - w/2, best_vals, w, label='Best Checkpoint', color='steelblue', alpha=0.85)
    if swa_vals:
        bars2 = ax.bar(x + w/2, swa_vals, w, label='SWA Model', color='darkorange', alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(keys, rotation=20, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title('Test Set: Best Checkpoint vs SWA Model', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
    if swa_vals:
        for bar in bars2:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Test comparison chart saved to {save_path}")


def plot_confusion_matrix(m, title, save_path):
    cm      = np.array([[m['TN'], m['FP']], [m['FN'], m['TP']]])
    fig, ax = plt.subplots(figsize=(5, 4))
    im      = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred 0', 'Pred 1'])
    ax.set_yticklabels(['True 0', 'True 1'])
    thresh = cm.max() / 2.0
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > thresh else 'black',
                    fontsize=14, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Confusion matrix saved to {save_path}")


def plot_metrics_radar(best_test_m, swa_test_m, save_path='radar_chart.png'):
    keys   = ['Accuracy', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Specificity', 'Sensitivity']
    n      = len(keys)
    angles = [x / float(n) * 2 * np.pi for x in range(n)]
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

    def draw(m, color, label):
        vals  = [m[k] for k in keys]
        vals += vals[:1]
        ax.plot(angles, vals, color=color, linewidth=2, label=label)
        ax.fill(angles, vals, color=color, alpha=0.15)

    draw(best_test_m, 'steelblue', 'Best Checkpoint')
    if swa_test_m:
        draw(swa_test_m, 'darkorange', 'SWA Model')

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(keys, size=9)
    ax.set_ylim(0, 1)
    ax.set_title('Test Metrics Radar', fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Radar chart saved to {save_path}")


def main():
    train_full = pd.read_csv(TRAIN_CSV)
    test_full  = pd.read_csv(TEST_CSV)

    train_df, val_df = train_test_split(
        train_full, test_size=VAL_SPLIT, random_state=SEED, stratify=train_full['label']
    )
    _, test_df = train_test_split(
        test_full, test_size=0.50, random_state=SEED, stratify=test_full['label']
    )

    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    print(f"Device: {DEVICE}")
    num_gpus = torch.cuda.device_count()
    print(f"Number of GPUs available: {num_gpus}\n")

    tokenizer    = AutoTokenizer.from_pretrained(MODEL_NAME)
    train_ds     = CodeDataset(train_df.reset_index(drop=True), tokenizer)
    val_ds       = CodeDataset(val_df.reset_index(drop=True),   tokenizer)
    test_ds      = CodeDataset(test_df.reset_index(drop=True),  tokenizer)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=4, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)

    counts        = train_df['label'].value_counts().sort_index().values
    raw_w         = torch.tensor([1.0 / c for c in counts], dtype=torch.float).to(DEVICE)
    class_weights = raw_w / raw_w.sum() * len(counts)

    model = LAPPEFT().to(DEVICE)
    if num_gpus > 1:
        model = nn.DataParallel(model, device_ids=list(range(num_gpus)))
        print(f"Using {num_gpus} GPUs via DataParallel: {list(range(num_gpus))}")

    raw_model = model.module if isinstance(model, nn.DataParallel) else model
    swa_model = copy.deepcopy(raw_model)
    for p in swa_model.parameters():
        p.requires_grad = False

    total_p   = sum(p.numel() for p in raw_model.parameters())
    trainable = sum(p.numel() for p in raw_model.parameters() if p.requires_grad)
    print(f"Total Params     : {total_p:,}")
    print(f"Trainable Params : {trainable:,}")
    print(f"Trainable %      : {100.0 * trainable / total_p:.4f}%\n")

    criterion        = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
    trainable_params = [p for p in raw_model.parameters() if p.requires_grad]
    optimizer        = torch.optim.AdamW(
        trainable_params, lr=LR, weight_decay=0.05, betas=(0.9, 0.999), eps=1e-8
    )

    effective_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    warmup_steps    = int(WARMUP_RATIO * effective_steps)
    scheduler       = get_cosine_schedule_with_warmup(optimizer, warmup_steps, effective_steps)

    swa_start_epoch = int(SWA_START_RATIO * EPOCHS)
    n_swa           = 0
    step_counter    = [0]

    best_val_f1 = -1.0
    best_epoch  = 0
    best_state  = None
    no_improve  = 0

    history = {
        'tr_loss': [], 'tr_acc': [], 'tr_f1': [], 'tr_auc': [], 'tr_mcc': [],
        'tr_prec': [], 'tr_rec': [], 'tr_bacc': [], 'tr_avgprec': [],
        'va_loss': [], 'va_acc': [], 'va_f1': [], 'va_auc': [], 'va_mcc': [],
        'va_prec': [], 'va_rec': [], 'va_bacc': [], 'va_avgprec': []
    }

    hdr = (f"{'Ep':>3} | {'TrLoss':>8} | {'TrAcc':>7} | {'TrF1':>7} | {'TrAUC':>7} | {'TrMCC':>7} | "
           f"{'VaLoss':>8} | {'VaAcc':>7} | {'VaF1':>7} | {'VaAUC':>7} | {'VaMCC':>7} | {'Note':<12}")
    sep = "-" * len(hdr)
    print(sep)
    print(hdr)
    print(sep)

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_m = train_epoch(model, train_loader, criterion, optimizer,
                                    scheduler, class_weights, step_counter)
        va_loss, va_m = eval_epoch(model, val_loader, criterion)

        history['tr_loss'].append(tr_loss)
        history['tr_acc'].append(tr_m['Accuracy'])
        history['tr_f1'].append(tr_m['F1'])
        history['tr_auc'].append(tr_m['ROC_AUC'])
        history['tr_mcc'].append(tr_m['MCC'])
        history['tr_prec'].append(tr_m['Precision'])
        history['tr_rec'].append(tr_m['Recall'])
        history['tr_bacc'].append(tr_m['Balanced_Acc'])
        history['tr_avgprec'].append(tr_m['Avg_Precision'])

        history['va_loss'].append(va_loss)
        history['va_acc'].append(va_m['Accuracy'])
        history['va_f1'].append(va_m['F1'])
        history['va_auc'].append(va_m['ROC_AUC'])
        history['va_mcc'].append(va_m['MCC'])
        history['va_prec'].append(va_m['Precision'])
        history['va_rec'].append(va_m['Recall'])
        history['va_bacc'].append(va_m['Balanced_Acc'])
        history['va_avgprec'].append(va_m['Avg_Precision'])

        note = ""
        if epoch > swa_start_epoch:
            update_swa(swa_model, model)
            n_swa += 1
            note += "SWA "

        is_best = va_m['F1'] > best_val_f1
        if is_best:
            best_val_f1 = va_m['F1']
            best_epoch  = epoch
            raw_m       = model.module if isinstance(model, nn.DataParallel) else model
            best_state  = {k: v.cpu().clone() for k, v in raw_m.state_dict().items()}
            no_improve  = 0
            note       += "<--"
        else:
            no_improve += 1

        print(
            f"{epoch:>3} | {tr_loss:>8.4f} | {tr_m['Accuracy']:>7.4f} | {tr_m['F1']:>7.4f} | "
            f"{tr_m['ROC_AUC']:>7.4f} | {tr_m['MCC']:>7.4f} | "
            f"{va_loss:>8.4f} | {va_m['Accuracy']:>7.4f} | {va_m['F1']:>7.4f} | "
            f"{va_m['ROC_AUC']:>7.4f} | {va_m['MCC']:>7.4f} | {note:<12}"
        )

        if no_improve >= PATIENCE:
            print(f"\n  Early stopping at epoch {epoch} (no val F1 improvement for {PATIENCE} epochs)")
            break

    print(sep)
    print(f"\nBest Epoch: {best_epoch}  |  Best Val F1: {best_val_f1:.4f}")

    print("\n--- Evaluating: Best Checkpoint Model ---")
    raw_m = model.module if isinstance(model, nn.DataParallel) else model
    raw_m.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    _, best_test_m = eval_epoch(model, test_loader, criterion)

    print("\n--- Evaluating: SWA Model ---")
    swa_test_m = None
    if n_swa > 0:
        apply_swa_avg(swa_model, n_swa)
        swa_model.to(DEVICE)
        _, swa_test_m = eval_epoch(swa_model, test_loader, criterion)

    print(f"\n{'='*60}")
    print(f"  Val F1 trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_f1'][-5:]]}")
    print(f"  Val Loss trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_loss'][-5:]]}")

    print(f"\n{'='*60}")
    print(f"  TEST RESULTS — Best Checkpoint (Epoch {best_epoch})")
    print(f"{'='*60}")
    scalar_keys = [k for k in best_test_m if k not in ('TP', 'TN', 'FP', 'FN')]
    for k in scalar_keys:
        print(f"  {k:<22}: {best_test_m[k]:.4f}")
    print(f"  {'TP':<22}: {best_test_m['TP']}")
    print(f"  {'TN':<22}: {best_test_m['TN']}")
    print(f"  {'FP':<22}: {best_test_m['FP']}")
    print(f"  {'FN':<22}: {best_test_m['FN']}")

    if swa_test_m is not None:
        print(f"\n{'='*60}")
        print(f"  TEST RESULTS — SWA Model (averaged over last {n_swa} epochs)")
        print(f"{'='*60}")
        for k in scalar_keys:
            print(f"  {k:<22}: {swa_test_m[k]:.4f}")
        print(f"  {'TP':<22}: {swa_test_m['TP']}")
        print(f"  {'TN':<22}: {swa_test_m['TN']}")
        print(f"  {'FP':<22}: {swa_test_m['FP']}")
        print(f"  {'FN':<22}: {swa_test_m['FN']}")

        print(f"\n{'='*60}")
        print(f"  COMPARISON: Best Checkpoint vs SWA")
        print(f"{'='*60}")
        compare_keys = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall']
        print(f"  {'Metric':<22} {'BestCkpt':>12} {'SWA':>12} {'Delta':>12}")
        print(f"  {'-'*58}")
        for k in compare_keys:
            delta  = swa_test_m[k] - best_test_m[k]
            symbol = "+" if delta >= 0 else ""
            print(f"  {k:<22} {best_test_m[k]:>12.4f} {swa_test_m[k]:>12.4f} {symbol+f'{delta:.4f}':>12}")

    plot_loss_curve(history,              save_path='loss_curve.png')
    plot_training_curves(history,         save_path='training_curves.png')
    plot_final_comparison(best_test_m, swa_test_m, save_path='test_comparison.png')
    plot_confusion_matrix(best_test_m, f'Confusion Matrix — Best Ckpt (Ep {best_epoch})', 'cm_best.png')
    if swa_test_m:
        plot_confusion_matrix(swa_test_m, f'Confusion Matrix — SWA ({n_swa} epochs)', 'cm_swa.png')
    plot_metrics_radar(best_test_m, swa_test_m, save_path='radar_chart.png')

    print("\nAll figures saved.")


if __name__ == "__main__":
    main()

# HierAdapter with Qwen2.5-Coder-3B, 1024 token - code vulnerability

In [ ]:
import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix,@@@@
    balanced_accuracy_score, average_precision_score
)
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

MODEL_NAME      = "Qwen/Qwen2.5-Coder-3B-Instruct"
TRAIN_CSV       = '/trqw.csv'
TEST_CSV        =  '/teqw.csv'
MAX_LENGTH      = 1024
D_MODEL         = 2048
BATCH_SIZE      = 16
GRAD_ACCUM      = 2
EPOCHS          = 30
LR              = 5e-5
WARMUP_RATIO    = 0.15
BOTTLENECK      = 6
NUM_LEVELS      = 4
LAYERS_LEVEL    = 3
VAL_SPLIT       = 0.20
SEED            = 42
RDROP_ALPHA     = 0.3
MIXUP_ALPHA     = 0.2
MIXUP_PROB      = 0.5
SWA_START_RATIO = 0.6
SWA_LR          = 1e-5
PATIENCE        = 7
NUM_GPUS        = torch.cuda.device_count() if torch.cuda.is_available() else 1
DEVICE          = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


class CodeDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.codes     = df['func_cleaned'].astype(str).tolist()
        self.labels    = df['label'].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.codes[idx],
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


class AttentionPooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attn = nn.Linear(d_model, 1)

    def forward(self, hidden, attention_mask):
        scores  = self.attn(hidden).squeeze(-1)
        scores  = scores.masked_fill(attention_mask == 0, -1e9)
        weights = torch.softmax(scores, dim=-1)
        return (hidden * weights.unsqueeze(-1)).sum(1)


class DynamicAlphaFusion(nn.Module):
    def __init__(self, d_model, num_levels):
        super().__init__()
        self.gate = nn.Linear(d_model, num_levels)

    def forward(self, base_cls, residuals):
        alphas   = torch.softmax(self.gate(base_cls), dim=-1)
        weighted = sum(alphas[:, i:i+1] * residuals[i] for i in range(len(residuals)))
        return base_cls + weighted


class LaplacianResidualAdapter(nn.Module):
    def __init__(self, d_model, bottleneck_dim):
        super().__init__()
        self.down    = nn.Linear(d_model, bottleneck_dim)
        self.act     = nn.GELU()
        self.up      = nn.Linear(bottleneck_dim, d_model)
        self.dropout = nn.Dropout(0.1)
        nn.init.kaiming_normal_(self.down.weight, nonlinearity='linear')
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return self.up(self.dropout(self.act(self.down(x))))


class LAPPEFT(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16,
            trust_remote_code=True
        )
        for p in self.encoder.parameters():
            p.requires_grad = False
        self.pool_layers = nn.ModuleList([
            AttentionPooling(D_MODEL) for _ in range(NUM_LEVELS + 1)
        ])
        self.lras = nn.ModuleList([
            LaplacianResidualAdapter(D_MODEL, BOTTLENECK)
            for _ in range(NUM_LEVELS)
        ])
        self.level_norms = nn.ModuleList([nn.LayerNorm(D_MODEL) for _ in range(NUM_LEVELS)])
        self.fusion      = DynamicAlphaFusion(D_MODEL, NUM_LEVELS)
        self.classifier  = nn.Sequential(
            nn.Linear(D_MODEL, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )

    def encode(self, input_ids, attention_mask):
        out        = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden     = [h.float() for h in out.hidden_states]
        num_layers = len(hidden)
        base = self.pool_layers[NUM_LEVELS](hidden[-1], attention_mask)
        level_indices = [
            min((lvl + 1) * LAYERS_LEVEL, num_layers - 1) for lvl in range(NUM_LEVELS)
        ]
        level_cls = [
            self.pool_layers[lvl](hidden[level_indices[lvl]], attention_mask)
            for lvl in range(NUM_LEVELS)
        ]
        residuals = []
        for lvl in range(NUM_LEVELS):
            diff = level_cls[lvl] - level_cls[lvl + 1] if lvl < NUM_LEVELS - 1 else level_cls[lvl]
            r    = self.lras[lvl](diff)
            residuals.append(self.level_norms[lvl](r))
        recon = self.fusion(base, residuals)
        return recon

    def forward(self, input_ids, attention_mask, cls_embed=None, encode_only=False):
        if cls_embed is not None:
            return self.classifier(cls_embed)
        recon = self.encode(input_ids, attention_mask)
        if encode_only:
            return recon
        return self.classifier(recon)


def mixup_embeddings(embed, labels, alpha=MIXUP_ALPHA):
    lam   = np.random.beta(alpha, alpha)
    idx   = torch.randperm(embed.size(0), device=embed.device)
    mixed = lam * embed + (1 - lam) * embed[idx]
    return mixed, labels, labels[idx], lam


def rdrop_kl_loss(logits1, logits2):
    p = F.log_softmax(logits1, dim=-1)
    q = F.log_softmax(logits2, dim=-1)
    return (F.kl_div(p, q.exp(), reduction='batchmean') +
            F.kl_div(q, p.exp(), reduction='batchmean')) / 2


def mixup_ce_loss(logits, labels_a, labels_b, lam, weight=None):
    loss_fn = nn.CrossEntropyLoss(weight=weight, label_smoothing=0.05, reduction='none')
    return (lam * loss_fn(logits, labels_a) + (1 - lam) * loss_fn(logits, labels_b)).mean()


def compute_metrics(labels, preds, probs):
    labels = np.array(labels)
    preds  = np.array(preds)
    probs  = np.array(probs)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        'Accuracy':      accuracy_score(labels, preds),
        'Balanced_Acc':  balanced_accuracy_score(labels, preds),
        'Precision':     precision_score(labels, preds, zero_division=0),
        'Recall':        recall_score(labels, preds, zero_division=0),
        'F1':            f1_score(labels, preds, zero_division=0),
        'ROC_AUC':       roc_auc_score(labels, probs),
        'Avg_Precision': average_precision_score(labels, probs),
        'MCC':           matthews_corrcoef(labels, preds),
        'Specificity':   tn / (tn + fp + 1e-9),
        'Sensitivity':   tp / (tp + fn + 1e-9),
        'FPR':           fp / (fp + tn + 1e-9),
        'FNR':           fn / (fn + tp + 1e-9),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn)
    }


def train_epoch(model, raw_model, loader, criterion, optimizer, scheduler, class_weights, step_counter):
    model.train()
    total_loss                       = 0.0
    all_labels, all_preds, all_probs = [], [], []
    optimizer.zero_grad()

    for batch_idx, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)

        with torch.no_grad():
            embed = model(input_ids, attention_mask, encode_only=True)

        use_mixup = (np.random.rand() < MIXUP_PROB)

        if use_mixup:
            mixed_embed, la, lb, lam = mixup_embeddings(embed, labels)
            logits1 = raw_model.classifier(mixed_embed)
            logits2 = raw_model.classifier(mixed_embed)
            ce_loss = mixup_ce_loss(logits1, la, lb, lam, weight=class_weights)
        else:
            logits1 = raw_model.classifier(embed)
            logits2 = raw_model.classifier(embed)
            ce_loss = criterion(logits1, labels)

        kl_loss = rdrop_kl_loss(logits1, logits2)
        loss    = ce_loss + RDROP_ALPHA * kl_loss
        loss    = loss / GRAD_ACCUM
        loss.backward()

        if (batch_idx + 1) % GRAD_ACCUM == 0 or (batch_idx + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(
                [p for p in raw_model.parameters() if p.requires_grad], 1.0
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            step_counter[0] += 1

        total_loss += (ce_loss + RDROP_ALPHA * kl_loss).item()

        with torch.no_grad():
            probs = torch.softmax(logits1, dim=-1)[:, 1].cpu().numpy()
            preds = logits1.argmax(dim=-1).cpu().numpy()

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)

    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss                       = 0.0
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['label'].to(DEVICE)

            logits     = model(input_ids, attention_mask)
            loss       = criterion(logits, labels)
            total_loss += loss.item()

            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            preds = logits.argmax(dim=-1).cpu().numpy()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds)
            all_probs.extend(probs)

    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)


def update_swa(swa_model, model):
    with torch.no_grad():
        for swa_p, p in zip(swa_model.parameters(), model.parameters()):
            swa_p.data.add_(p.data.cpu())


def apply_swa_avg(swa_model, n_swa):
    with torch.no_grad():
        for p in swa_model.parameters():
            p.data.div_(n_swa)


def plot_loss_curve(history, save_path='train_val_loss_curve.png'):
    epochs = list(range(1, len(history['tr_loss']) + 1))
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(epochs, history['tr_loss'], 'b-o', markersize=4, linewidth=2, label='Train Loss')
    ax.plot(epochs, history['va_loss'], 'r-o', markersize=4, linewidth=2, label='Validation Loss')
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.set_title('Training vs Validation Loss — LAPPEFT (Qwen2.5-Coder-3B)', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Loss curve saved to {save_path}")


def plot_training_curves(history, save_path='training_curves.png'):
    epochs = list(range(1, len(history['tr_loss']) + 1))
    fig, axes = plt.subplots(3, 3, figsize=(18, 14))
    fig.suptitle('LAPPEFT (Qwen2.5-Coder-3B) Training Curves', fontsize=16, fontweight='bold')

    def plot_ax(ax, key, title, ylabel):
        ax.plot(epochs, history[f'tr_{key}'], 'b-o', markersize=3, label='Train')
        ax.plot(epochs, history[f'va_{key}'], 'r-o', markersize=3, label='Val')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(True, alpha=0.3)

    plot_ax(axes[0][0], 'loss',    'Loss',          'Loss')
    plot_ax(axes[0][1], 'acc',     'Accuracy',      'Accuracy')
    plot_ax(axes[0][2], 'f1',      'F1 Score',      'F1')
    plot_ax(axes[1][0], 'auc',     'ROC AUC',       'AUC')
    plot_ax(axes[1][1], 'mcc',     'MCC',           'MCC')
    plot_ax(axes[1][2], 'prec',    'Precision',     'Precision')
    plot_ax(axes[2][0], 'rec',     'Recall',        'Recall')
    plot_ax(axes[2][1], 'bacc',    'Balanced Acc',  'Balanced Accuracy')
    plot_ax(axes[2][2], 'avgprec', 'Avg Precision', 'Avg Precision')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Training curves saved to {save_path}")


def plot_final_comparison(best_test_m, swa_test_m, save_path='test_comparison.png'):
    keys      = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Avg_Precision']
    x         = np.arange(len(keys))
    w         = 0.35
    best_vals = [best_test_m[k] for k in keys]
    swa_vals  = [swa_test_m[k] for k in keys] if swa_test_m else None

    fig, ax = plt.subplots(figsize=(14, 6))
    bars1   = ax.bar(x - w/2, best_vals, w, label='Best Checkpoint', color='steelblue', alpha=0.85)
    if swa_vals:
        bars2 = ax.bar(x + w/2, swa_vals, w, label='SWA Model', color='darkorange', alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(keys, rotation=20, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title('Test Set: Best Checkpoint vs SWA Model', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
    if swa_vals:
        for bar in bars2:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Test comparison chart saved to {save_path}")


def plot_confusion_matrix(m, title, save_path):
    cm      = np.array([[m['TN'], m['FP']], [m['FN'], m['TP']]])
    fig, ax = plt.subplots(figsize=(5, 4))
    im      = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred 0', 'Pred 1'])
    ax.set_yticklabels(['True 0', 'True 1'])
    thresh = cm.max() / 2.0
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > thresh else 'black',
                    fontsize=14, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Confusion matrix saved to {save_path}")


def plot_metrics_radar(best_test_m, swa_test_m, save_path='radar_chart.png'):
    keys   = ['Accuracy', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Specificity', 'Sensitivity']
    n      = len(keys)
    angles = [x / float(n) * 2 * np.pi for x in range(n)]
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

    def draw(m, color, label):
        vals  = [m[k] for k in keys]
        vals += vals[:1]
        ax.plot(angles, vals, color=color, linewidth=2, label=label)
        ax.fill(angles, vals, color=color, alpha=0.15)

    draw(best_test_m, 'steelblue', 'Best Checkpoint')
    if swa_test_m:
        draw(swa_test_m, 'darkorange', 'SWA Model')

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(keys, size=9)
    ax.set_ylim(0, 1)
    ax.set_title('Test Metrics Radar', fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Radar chart saved to {save_path}")


def main():
    train_full = pd.read_csv(TRAIN_CSV)
    test_full  = pd.read_csv(TEST_CSV)

    train_df, val_df = train_test_split(
        train_full, test_size=VAL_SPLIT, random_state=SEED, stratify=train_full['label']
    )
    _, test_df = train_test_split(
        test_full, test_size=0.50, random_state=SEED, stratify=test_full['label']
    )

    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    print(f"Device: {DEVICE} | GPUs used: {NUM_GPUS}\n")

    tokenizer              = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.padding_side = 'right'

    train_ds = CodeDataset(train_df.reset_index(drop=True), tokenizer)
    val_ds   = CodeDataset(val_df.reset_index(drop=True),   tokenizer)
    test_ds  = CodeDataset(test_df.reset_index(drop=True),  tokenizer)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=4, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)

    counts        = train_df['label'].value_counts().sort_index().values
    raw_w         = torch.tensor([1.0 / c for c in counts], dtype=torch.float).to(DEVICE)
    class_weights = raw_w / raw_w.sum() * len(counts)

    raw_model = LAPPEFT()
    swa_raw   = copy.deepcopy(raw_model)
    for p in swa_raw.parameters():
        p.requires_grad = False

    raw_model = raw_model.to(DEVICE)

    if NUM_GPUS > 1:
        model = nn.DataParallel(raw_model, device_ids=list(range(NUM_GPUS)))
    else:
        model = raw_model

    total_p   = sum(p.numel() for p in raw_model.parameters())
    trainable = sum(p.numel() for p in raw_model.parameters() if p.requires_grad)
    print(f"Total Params     : {total_p:,}")
    print(f"Trainable Params : {trainable:,}")
    print(f"Trainable %      : {100.0 * trainable / total_p:.4f}%\n")

    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

    trainable_params = [p for p in raw_model.parameters() if p.requires_grad]
    optimizer        = torch.optim.AdamW(
        trainable_params, lr=LR, weight_decay=0.05,
        betas=(0.9, 0.999), eps=1e-8
    )

    effective_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    warmup_steps    = int(WARMUP_RATIO * effective_steps)
    scheduler       = get_cosine_schedule_with_warmup(optimizer, warmup_steps, effective_steps)

    swa_start_epoch = int(SWA_START_RATIO * EPOCHS)
    n_swa           = 0
    step_counter    = [0]

    best_val_f1 = -1.0
    best_epoch  = 0
    best_state  = None
    no_improve  = 0

    history = {
        'tr_loss': [], 'tr_acc': [], 'tr_f1': [], 'tr_auc': [], 'tr_mcc': [],
        'tr_prec': [], 'tr_rec': [], 'tr_bacc': [], 'tr_avgprec': [],
        'va_loss': [], 'va_acc': [], 'va_f1': [], 'va_auc': [], 'va_mcc': [],
        'va_prec': [], 'va_rec': [], 'va_bacc': [], 'va_avgprec': []
    }

    hdr = (f"{'Ep':>3} | {'TrLoss':>8} | {'TrAcc':>7} | {'TrF1':>7} | {'TrAUC':>7} | {'TrMCC':>7} | "
           f"{'VaLoss':>8} | {'VaAcc':>7} | {'VaF1':>7} | {'VaAUC':>7} | {'VaMCC':>7} | {'Note':<12}")
    sep = "-" * len(hdr)
    print(sep)
    print(hdr)
    print(sep)

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_m = train_epoch(model, raw_model, train_loader, criterion, optimizer,
                                    scheduler, class_weights, step_counter)
        va_loss, va_m = eval_epoch(model, val_loader, criterion)

        history['tr_loss'].append(tr_loss)
        history['tr_acc'].append(tr_m['Accuracy'])
        history['tr_f1'].append(tr_m['F1'])
        history['tr_auc'].append(tr_m['ROC_AUC'])
        history['tr_mcc'].append(tr_m['MCC'])
        history['tr_prec'].append(tr_m['Precision'])
        history['tr_rec'].append(tr_m['Recall'])
        history['tr_bacc'].append(tr_m['Balanced_Acc'])
        history['tr_avgprec'].append(tr_m['Avg_Precision'])
        history['va_loss'].append(va_loss)
        history['va_acc'].append(va_m['Accuracy'])
        history['va_f1'].append(va_m['F1'])
        history['va_auc'].append(va_m['ROC_AUC'])
        history['va_mcc'].append(va_m['MCC'])
        history['va_prec'].append(va_m['Precision'])
        history['va_rec'].append(va_m['Recall'])
        history['va_bacc'].append(va_m['Balanced_Acc'])
        history['va_avgprec'].append(va_m['Avg_Precision'])

        note = ""
        if epoch > swa_start_epoch:
            update_swa(swa_raw, raw_model)
            n_swa += 1
            note += "SWA "

        is_best = va_m['F1'] > best_val_f1
        if is_best:
            best_val_f1 = va_m['F1']
            best_epoch  = epoch
            best_state  = {k: v.cpu().clone() for k, v in raw_model.state_dict().items()}
            no_improve  = 0
            note       += "<--"
        else:
            no_improve += 1

        print(
            f"{epoch:>3} | {tr_loss:>8.4f} | {tr_m['Accuracy']:>7.4f} | {tr_m['F1']:>7.4f} | "
            f"{tr_m['ROC_AUC']:>7.4f} | {tr_m['MCC']:>7.4f} | "
            f"{va_loss:>8.4f} | {va_m['Accuracy']:>7.4f} | {va_m['F1']:>7.4f} | "
            f"{va_m['ROC_AUC']:>7.4f} | {va_m['MCC']:>7.4f} | {note:<12}"
        )

        if no_improve >= PATIENCE:
            print(f"\n  Early stopping at epoch {epoch} (no val F1 improvement for {PATIENCE} epochs)")
            break

    print(sep)
    print(f"\nBest Epoch: {best_epoch}  |  Best Val F1: {best_val_f1:.4f}")

    print("\n--- Evaluating: Best Checkpoint Model ---")
    raw_model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    _, best_test_m = eval_epoch(model, test_loader, criterion)

    print("\n--- Evaluating: SWA Model ---")
    swa_test_m = None
    if n_swa > 0:
        apply_swa_avg(swa_raw, n_swa)
        swa_raw_gpu = swa_raw.to(DEVICE)
        _, swa_test_m = eval_epoch(swa_raw_gpu, test_loader, criterion)

    print(f"\n{'='*60}")
    print(f"  Val F1 trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_f1'][-5:]]}")
    print(f"  Val Loss trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_loss'][-5:]]}")

    print(f"\n{'='*60}")
    print(f"  TEST RESULTS — Best Checkpoint (Epoch {best_epoch})")
    print(f"{'='*60}")
    scalar_keys = [k for k in best_test_m if k not in ('TP', 'TN', 'FP', 'FN')]
    for k in scalar_keys:
        print(f"  {k:<22}: {best_test_m[k]:.4f}")
    print(f"  {'TP':<22}: {best_test_m['TP']}")
    print(f"  {'TN':<22}: {best_test_m['TN']}")
    print(f"  {'FP':<22}: {best_test_m['FP']}")
    print(f"  {'FN':<22}: {best_test_m['FN']}")

    if swa_test_m is not None:
        print(f"\n{'='*60}")
        print(f"  TEST RESULTS — SWA Model (averaged over last {n_swa} epochs)")
        print(f"{'='*60}")
        for k in scalar_keys:
            print(f"  {k:<22}: {swa_test_m[k]:.4f}")
        print(f"  {'TP':<22}: {swa_test_m['TP']}")
        print(f"  {'TN':<22}: {swa_test_m['TN']}")
        print(f"  {'FP':<22}: {swa_test_m['FP']}")
        print(f"  {'FN':<22}: {swa_test_m['FN']}")

        print(f"\n{'='*60}")
        print(f"  COMPARISON: Best Checkpoint vs SWA")
        print(f"{'='*60}")
        compare_keys = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall']
        print(f"  {'Metric':<22} {'BestCkpt':>12} {'SWA':>12} {'Delta':>12}")
        print(f"  {'-'*58}")
        for k in compare_keys:
            delta  = swa_test_m[k] - best_test_m[k]
            symbol = "+" if delta >= 0 else ""
            print(f"  {k:<22} {best_test_m[k]:>12.4f} {swa_test_m[k]:>12.4f} {symbol+f'{delta:.4f}':>12}")

    plot_loss_curve(history,     save_path='train_val_loss_curve1024.png')
    plot_training_curves(history, save_path='training_curves1024.png')
    plot_final_comparison(best_test_m, swa_test_m, save_path='test_comparison1024.png')
    plot_confusion_matrix(best_test_m, f'Confusion Matrix — Best Ckpt (Ep {best_epoch})', 'cm_best.png')
    if swa_test_m:
        plot_confusion_matrix(swa_test_m, f'Confusion Matrix — SWA ({n_swa} epochs)', 'cm_swa.png')
    plot_metrics_radar(best_test_m, swa_test_m, save_path='radar_chart.png')

    print("\nAll figures saved.")


if __name__ == "__main__":
    main()

# HierAdapter with Qwen2.5-Coder-3B, 512token - code vulnerability

In [ ]:
import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix,
    balanced_accuracy_score, average_precision_score
)
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

MODEL_NAME      = "Qwen/Qwen2.5-Coder-3B-Instruct"
TRAIN_CSV       = '/traincodex.csv'
TEST_CSV        = '/testcodex.csv'
MAX_LENGTH      = 512
D_MODEL         = 2048
BATCH_SIZE      = 16
GRAD_ACCUM      = 2
EPOCHS          = 30
LR              = 5e-5
WARMUP_RATIO    = 0.15
BOTTLENECK      = 6
NUM_LEVELS      = 4
LAYERS_LEVEL    = 3
VAL_SPLIT       = 0.20
SEED            = 42
RDROP_ALPHA     = 0.3
MIXUP_ALPHA     = 0.2
MIXUP_PROB      = 0.5
SWA_START_RATIO = 0.6
SWA_LR          = 1e-5
PATIENCE        = 7
NUM_GPUS        = torch.cuda.device_count() if torch.cuda.is_available() else 1
DEVICE          = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


class CodeDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.codes     = df['code'].astype(str).tolist()
        self.labels    = df['label'].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.codes[idx],
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


class AttentionPooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attn = nn.Linear(d_model, 1)

    def forward(self, hidden, attention_mask):
        scores  = self.attn(hidden).squeeze(-1)
        scores  = scores.masked_fill(attention_mask == 0, -1e9)
        weights = torch.softmax(scores, dim=-1)
        return (hidden * weights.unsqueeze(-1)).sum(1)


class DynamicAlphaFusion(nn.Module):
    def __init__(self, d_model, num_levels):
        super().__init__()
        self.gate = nn.Linear(d_model, num_levels)

    def forward(self, base_cls, residuals):
        alphas   = torch.softmax(self.gate(base_cls), dim=-1)
        weighted = sum(alphas[:, i:i+1] * residuals[i] for i in range(len(residuals)))
        return base_cls + weighted


class LaplacianResidualAdapter(nn.Module):
    def __init__(self, d_model, bottleneck_dim):
        super().__init__()
        self.down    = nn.Linear(d_model, bottleneck_dim)
        self.act     = nn.GELU()
        self.up      = nn.Linear(bottleneck_dim, d_model)
        self.dropout = nn.Dropout(0.1)
        nn.init.kaiming_normal_(self.down.weight, nonlinearity='linear')
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return self.up(self.dropout(self.act(self.down(x))))


class LAPPEFT(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16,
            trust_remote_code=True
        )
        for p in self.encoder.parameters():
            p.requires_grad = False
        self.pool_layers = nn.ModuleList([
            AttentionPooling(D_MODEL) for _ in range(NUM_LEVELS + 1)
        ])
        self.lras = nn.ModuleList([
            LaplacianResidualAdapter(D_MODEL, BOTTLENECK)
            for _ in range(NUM_LEVELS)
        ])
        self.level_norms = nn.ModuleList([nn.LayerNorm(D_MODEL) for _ in range(NUM_LEVELS)])
        self.fusion      = DynamicAlphaFusion(D_MODEL, NUM_LEVELS)
        self.classifier  = nn.Sequential(
            nn.Linear(D_MODEL, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )

    def encode(self, input_ids, attention_mask):
        out        = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden     = [h.float() for h in out.hidden_states]
        num_layers = len(hidden)
        base = self.pool_layers[NUM_LEVELS](hidden[-1], attention_mask)
        level_indices = [
            min((lvl + 1) * LAYERS_LEVEL, num_layers - 1) for lvl in range(NUM_LEVELS)
        ]
        level_cls = [
            self.pool_layers[lvl](hidden[level_indices[lvl]], attention_mask)
            for lvl in range(NUM_LEVELS)
        ]
        residuals = []
        for lvl in range(NUM_LEVELS):
            diff = level_cls[lvl] - level_cls[lvl + 1] if lvl < NUM_LEVELS - 1 else level_cls[lvl]
            r    = self.lras[lvl](diff)
            residuals.append(self.level_norms[lvl](r))
        recon = self.fusion(base, residuals)
        return recon

    def forward(self, input_ids, attention_mask, cls_embed=None, encode_only=False):
        if cls_embed is not None:
            return self.classifier(cls_embed)
        recon = self.encode(input_ids, attention_mask)
        if encode_only:
            return recon
        return self.classifier(recon)


def mixup_embeddings(embed, labels, alpha=MIXUP_ALPHA):
    lam   = np.random.beta(alpha, alpha)
    idx   = torch.randperm(embed.size(0), device=embed.device)
    mixed = lam * embed + (1 - lam) * embed[idx]
    return mixed, labels, labels[idx], lam


def rdrop_kl_loss(logits1, logits2):
    p = F.log_softmax(logits1, dim=-1)
    q = F.log_softmax(logits2, dim=-1)
    return (F.kl_div(p, q.exp(), reduction='batchmean') +
            F.kl_div(q, p.exp(), reduction='batchmean')) / 2


def mixup_ce_loss(logits, labels_a, labels_b, lam, weight=None):
    loss_fn = nn.CrossEntropyLoss(weight=weight, label_smoothing=0.05, reduction='none')
    return (lam * loss_fn(logits, labels_a) + (1 - lam) * loss_fn(logits, labels_b)).mean()


def compute_metrics(labels, preds, probs):
    labels = np.array(labels)
    preds  = np.array(preds)
    probs  = np.array(probs)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        'Accuracy':      accuracy_score(labels, preds),
        'Balanced_Acc':  balanced_accuracy_score(labels, preds),
        'Precision':     precision_score(labels, preds, zero_division=0),
        'Recall':        recall_score(labels, preds, zero_division=0),
        'F1':            f1_score(labels, preds, zero_division=0),
        'ROC_AUC':       roc_auc_score(labels, probs),
        'Avg_Precision': average_precision_score(labels, probs),
        'MCC':           matthews_corrcoef(labels, preds),
        'Specificity':   tn / (tn + fp + 1e-9),
        'Sensitivity':   tp / (tp + fn + 1e-9),
        'FPR':           fp / (fp + tn + 1e-9),
        'FNR':           fn / (fn + tp + 1e-9),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn)
    }


def train_epoch(model, raw_model, loader, criterion, optimizer, scheduler, class_weights, step_counter):
    model.train()
    total_loss                       = 0.0
    all_labels, all_preds, all_probs = [], [], []
    optimizer.zero_grad()

    for batch_idx, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)

        with torch.no_grad():
            embed = model(input_ids, attention_mask, encode_only=True)

        use_mixup = (np.random.rand() < MIXUP_PROB)

        if use_mixup:
            mixed_embed, la, lb, lam = mixup_embeddings(embed, labels)
            logits1 = raw_model.classifier(mixed_embed)
            logits2 = raw_model.classifier(mixed_embed)
            ce_loss = mixup_ce_loss(logits1, la, lb, lam, weight=class_weights)
        else:
            logits1 = raw_model.classifier(embed)
            logits2 = raw_model.classifier(embed)
            ce_loss = criterion(logits1, labels)

        kl_loss = rdrop_kl_loss(logits1, logits2)
        loss    = ce_loss + RDROP_ALPHA * kl_loss
        loss    = loss / GRAD_ACCUM
        loss.backward()

        if (batch_idx + 1) % GRAD_ACCUM == 0 or (batch_idx + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(
                [p for p in raw_model.parameters() if p.requires_grad], 1.0
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            step_counter[0] += 1

        total_loss += (ce_loss + RDROP_ALPHA * kl_loss).item()

        with torch.no_grad():
            probs = torch.softmax(logits1, dim=-1)[:, 1].cpu().numpy()
            preds = logits1.argmax(dim=-1).cpu().numpy()

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)

    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss                       = 0.0
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['label'].to(DEVICE)

            logits     = model(input_ids, attention_mask)
            loss       = criterion(logits, labels)
            total_loss += loss.item()

            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            preds = logits.argmax(dim=-1).cpu().numpy()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds)
            all_probs.extend(probs)

    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)


def update_swa(swa_model, model):
    with torch.no_grad():
        for swa_p, p in zip(swa_model.parameters(), model.parameters()):
            swa_p.data.add_(p.data.cpu())


def apply_swa_avg(swa_model, n_swa):
    with torch.no_grad():
        for p in swa_model.parameters():
            p.data.div_(n_swa)


def plot_loss_curve(history, save_path='train_val_loss_curve.png'):
    epochs = list(range(1, len(history['tr_loss']) + 1))
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(epochs, history['tr_loss'], 'b-o', markersize=4, linewidth=2, label='Train Loss')
    ax.plot(epochs, history['va_loss'], 'r-o', markersize=4, linewidth=2, label='Validation Loss')
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.set_title('Training vs Validation Loss — LAPPEFT (Qwen2.5-Coder-3B)', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Loss curve saved to {save_path}")


def plot_training_curves(history, save_path='training_curves.png'):
    epochs = list(range(1, len(history['tr_loss']) + 1))
    fig, axes = plt.subplots(3, 3, figsize=(18, 14))
    fig.suptitle('LAPPEFT (Qwen2.5-Coder-3B) Training Curves', fontsize=16, fontweight='bold')

    def plot_ax(ax, key, title, ylabel):
        ax.plot(epochs, history[f'tr_{key}'], 'b-o', markersize=3, label='Train')
        ax.plot(epochs, history[f'va_{key}'], 'r-o', markersize=3, label='Val')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(True, alpha=0.3)

    plot_ax(axes[0][0], 'loss',    'Loss',          'Loss')
    plot_ax(axes[0][1], 'acc',     'Accuracy',      'Accuracy')
    plot_ax(axes[0][2], 'f1',      'F1 Score',      'F1')
    plot_ax(axes[1][0], 'auc',     'ROC AUC',       'AUC')
    plot_ax(axes[1][1], 'mcc',     'MCC',           'MCC')
    plot_ax(axes[1][2], 'prec',    'Precision',     'Precision')
    plot_ax(axes[2][0], 'rec',     'Recall',        'Recall')
    plot_ax(axes[2][1], 'bacc',    'Balanced Acc',  'Balanced Accuracy')
    plot_ax(axes[2][2], 'avgprec', 'Avg Precision', 'Avg Precision')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Training curves saved to {save_path}")


def plot_final_comparison(best_test_m, swa_test_m, save_path='test_comparison.png'):
    keys      = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Avg_Precision']
    x         = np.arange(len(keys))
    w         = 0.35
    best_vals = [best_test_m[k] for k in keys]
    swa_vals  = [swa_test_m[k] for k in keys] if swa_test_m else None

    fig, ax = plt.subplots(figsize=(14, 6))
    bars1   = ax.bar(x - w/2, best_vals, w, label='Best Checkpoint', color='steelblue', alpha=0.85)
    if swa_vals:
        bars2 = ax.bar(x + w/2, swa_vals, w, label='SWA Model', color='darkorange', alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(keys, rotation=20, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title('Test Set: Best Checkpoint vs SWA Model', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
    if swa_vals:
        for bar in bars2:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Test comparison chart saved to {save_path}")


def plot_confusion_matrix(m, title, save_path):
    cm      = np.array([[m['TN'], m['FP']], [m['FN'], m['TP']]])
    fig, ax = plt.subplots(figsize=(5, 4))
    im      = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred 0', 'Pred 1'])
    ax.set_yticklabels(['True 0', 'True 1'])
    thresh = cm.max() / 2.0
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > thresh else 'black',
                    fontsize=14, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Confusion matrix saved to {save_path}")


def plot_metrics_radar(best_test_m, swa_test_m, save_path='radar_chart.png'):
    keys   = ['Accuracy', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Specificity', 'Sensitivity']
    n      = len(keys)
    angles = [x / float(n) * 2 * np.pi for x in range(n)]
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

    def draw(m, color, label):
        vals  = [m[k] for k in keys]
        vals += vals[:1]
        ax.plot(angles, vals, color=color, linewidth=2, label=label)
        ax.fill(angles, vals, color=color, alpha=0.15)

    draw(best_test_m, 'steelblue', 'Best Checkpoint')
    if swa_test_m:
        draw(swa_test_m, 'darkorange', 'SWA Model')

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(keys, size=9)
    ax.set_ylim(0, 1)
    ax.set_title('Test Metrics Radar', fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Radar chart saved to {save_path}")


def main():
    train_full = pd.read_csv(TRAIN_CSV)
    test_full  = pd.read_csv(TEST_CSV)

    train_df, val_df = train_test_split(
        train_full, test_size=VAL_SPLIT, random_state=SEED, stratify=train_full['label']
    )
    _, test_df = train_test_split(
        test_full, test_size=0.50, random_state=SEED, stratify=test_full['label']
    )

    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    print(f"Device: {DEVICE} | GPUs used: {NUM_GPUS}\n")

    tokenizer              = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.padding_side = 'right'

    train_ds = CodeDataset(train_df.reset_index(drop=True), tokenizer)
    val_ds   = CodeDataset(val_df.reset_index(drop=True),   tokenizer)
    test_ds  = CodeDataset(test_df.reset_index(drop=True),  tokenizer)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=4, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)

    counts        = train_df['label'].value_counts().sort_index().values
    raw_w         = torch.tensor([1.0 / c for c in counts], dtype=torch.float).to(DEVICE)
    class_weights = raw_w / raw_w.sum() * len(counts)

    raw_model = LAPPEFT()
    swa_raw   = copy.deepcopy(raw_model)
    for p in swa_raw.parameters():
        p.requires_grad = False

    raw_model = raw_model.to(DEVICE)

    if NUM_GPUS > 1:
        model = nn.DataParallel(raw_model, device_ids=list(range(NUM_GPUS)))
    else:
        model = raw_model

    total_p   = sum(p.numel() for p in raw_model.parameters())
    trainable = sum(p.numel() for p in raw_model.parameters() if p.requires_grad)
    print(f"Total Params     : {total_p:,}")
    print(f"Trainable Params : {trainable:,}")
    print(f"Trainable %      : {100.0 * trainable / total_p:.4f}%\n")

    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

    trainable_params = [p for p in raw_model.parameters() if p.requires_grad]
    optimizer        = torch.optim.AdamW(
        trainable_params, lr=LR, weight_decay=0.05,
        betas=(0.9, 0.999), eps=1e-8
    )

    effective_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    warmup_steps    = int(WARMUP_RATIO * effective_steps)
    scheduler       = get_cosine_schedule_with_warmup(optimizer, warmup_steps, effective_steps)

    swa_start_epoch = int(SWA_START_RATIO * EPOCHS)
    n_swa           = 0
    step_counter    = [0]

    best_val_f1 = -1.0
    best_epoch  = 0
    best_state  = None
    no_improve  = 0

    history = {
        'tr_loss': [], 'tr_acc': [], 'tr_f1': [], 'tr_auc': [], 'tr_mcc': [],
        'tr_prec': [], 'tr_rec': [], 'tr_bacc': [], 'tr_avgprec': [],
        'va_loss': [], 'va_acc': [], 'va_f1': [], 'va_auc': [], 'va_mcc': [],
        'va_prec': [], 'va_rec': [], 'va_bacc': [], 'va_avgprec': []
    }

    hdr = (f"{'Ep':>3} | {'TrLoss':>8} | {'TrAcc':>7} | {'TrF1':>7} | {'TrAUC':>7} | {'TrMCC':>7} | "
           f"{'VaLoss':>8} | {'VaAcc':>7} | {'VaF1':>7} | {'VaAUC':>7} | {'VaMCC':>7} | {'Note':<12}")
    sep = "-" * len(hdr)
    print(sep)
    print(hdr)
    print(sep)

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_m = train_epoch(model, raw_model, train_loader, criterion, optimizer,
                                    scheduler, class_weights, step_counter)
        va_loss, va_m = eval_epoch(model, val_loader, criterion)

        history['tr_loss'].append(tr_loss)
        history['tr_acc'].append(tr_m['Accuracy'])
        history['tr_f1'].append(tr_m['F1'])
        history['tr_auc'].append(tr_m['ROC_AUC'])
        history['tr_mcc'].append(tr_m['MCC'])
        history['tr_prec'].append(tr_m['Precision'])
        history['tr_rec'].append(tr_m['Recall'])
        history['tr_bacc'].append(tr_m['Balanced_Acc'])
        history['tr_avgprec'].append(tr_m['Avg_Precision'])
        history['va_loss'].append(va_loss)
        history['va_acc'].append(va_m['Accuracy'])
        history['va_f1'].append(va_m['F1'])
        history['va_auc'].append(va_m['ROC_AUC'])
        history['va_mcc'].append(va_m['MCC'])
        history['va_prec'].append(va_m['Precision'])
        history['va_rec'].append(va_m['Recall'])
        history['va_bacc'].append(va_m['Balanced_Acc'])
        history['va_avgprec'].append(va_m['Avg_Precision'])

        note = ""
        if epoch > swa_start_epoch:
            update_swa(swa_raw, raw_model)
            n_swa += 1
            note += "SWA "

        is_best = va_m['F1'] > best_val_f1
        if is_best:
            best_val_f1 = va_m['F1']
            best_epoch  = epoch
            best_state  = {k: v.cpu().clone() for k, v in raw_model.state_dict().items()}
            no_improve  = 0
            note       += "<--"
        else:
            no_improve += 1

        print(
            f"{epoch:>3} | {tr_loss:>8.4f} | {tr_m['Accuracy']:>7.4f} | {tr_m['F1']:>7.4f} | "
            f"{tr_m['ROC_AUC']:>7.4f} | {tr_m['MCC']:>7.4f} | "
            f"{va_loss:>8.4f} | {va_m['Accuracy']:>7.4f} | {va_m['F1']:>7.4f} | "
            f"{va_m['ROC_AUC']:>7.4f} | {va_m['MCC']:>7.4f} | {note:<12}"
        )

        if no_improve >= PATIENCE:
            print(f"\n  Early stopping at epoch {epoch} (no val F1 improvement for {PATIENCE} epochs)")
            break

    print(sep)
    print(f"\nBest Epoch: {best_epoch}  |  Best Val F1: {best_val_f1:.4f}")

    print("\n--- Evaluating: Best Checkpoint Model ---")
    raw_model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    _, best_test_m = eval_epoch(model, test_loader, criterion)

    print("\n--- Evaluating: SWA Model ---")
    swa_test_m = None
    if n_swa > 0:
        apply_swa_avg(swa_raw, n_swa)
        swa_raw_gpu = swa_raw.to(DEVICE)
        _, swa_test_m = eval_epoch(swa_raw_gpu, test_loader, criterion)

    print(f"\n{'='*60}")
    print(f"  Val F1 trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_f1'][-5:]]}")
    print(f"  Val Loss trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_loss'][-5:]]}")

    print(f"\n{'='*60}")
    print(f"  TEST RESULTS — Best Checkpoint (Epoch {best_epoch})")
    print(f"{'='*60}")
    scalar_keys = [k for k in best_test_m if k not in ('TP', 'TN', 'FP', 'FN')]
    for k in scalar_keys:
        print(f"  {k:<22}: {best_test_m[k]:.4f}")
    print(f"  {'TP':<22}: {best_test_m['TP']}")
    print(f"  {'TN':<22}: {best_test_m['TN']}")
    print(f"  {'FP':<22}: {best_test_m['FP']}")
    print(f"  {'FN':<22}: {best_test_m['FN']}")

    if swa_test_m is not None:
        print(f"\n{'='*60}")
        print(f"  TEST RESULTS — SWA Model (averaged over last {n_swa} epochs)")
        print(f"{'='*60}")
        for k in scalar_keys:
            print(f"  {k:<22}: {swa_test_m[k]:.4f}")
        print(f"  {'TP':<22}: {swa_test_m['TP']}")
        print(f"  {'TN':<22}: {swa_test_m['TN']}")
        print(f"  {'FP':<22}: {swa_test_m['FP']}")
        print(f"  {'FN':<22}: {swa_test_m['FN']}")

        print(f"\n{'='*60}")
        print(f"  COMPARISON: Best Checkpoint vs SWA")
        print(f"{'='*60}")
        compare_keys = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall']
        print(f"  {'Metric':<22} {'BestCkpt':>12} {'SWA':>12} {'Delta':>12}")
        print(f"  {'-'*58}")
        for k in compare_keys:
            delta  = swa_test_m[k] - best_test_m[k]
            symbol = "+" if delta >= 0 else ""
            print(f"  {k:<22} {best_test_m[k]:>12.4f} {swa_test_m[k]:>12.4f} {symbol+f'{delta:.4f}':>12}")

    plot_loss_curve(history,     save_path='train_val_loss_curve.png')
    plot_training_curves(history, save_path='training_curves.png')
    plot_final_comparison(best_test_m, swa_test_m, save_path='test_comparison.png')
    plot_confusion_matrix(best_test_m, f'Confusion Matrix — Best Ckpt (Ep {best_epoch})', 'cm_best.png')
    if swa_test_m:
        plot_confusion_matrix(swa_test_m, f'Confusion Matrix — SWA ({n_swa} epochs)', 'cm_swa.png')
    plot_metrics_radar(best_test_m, swa_test_m, save_path='radar_chart.png')

    print("\nAll figures saved.")


if __name__ == "__main__":
    main()

# HierAdapter with Qwen2.5-Coder-1.5B, 512token - code vulnerability

In [ ]:
import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix,
    balanced_accuracy_score, average_precision_score
)
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
MODEL_NAME      = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
TRAIN_CSV       = '/traincodex.csv'
TEST_CSV        = '/testcodex.csv'
MAX_LENGTH      = 512
D_MODEL         = 1536
BATCH_SIZE      = 16
GRAD_ACCUM      = 2
EPOCHS          = 30
LR              = 5e-5
WARMUP_RATIO    = 0.15
BOTTLENECK      = 6
NUM_LEVELS      = 4
LAYERS_LEVEL    = 7
VAL_SPLIT       = 0.20
SEED            = 42
RDROP_ALPHA     = 0.3
MIXUP_ALPHA     = 0.2
MIXUP_PROB      = 0.5
SWA_START_RATIO = 0.6
SWA_LR          = 1e-5
PATIENCE        = 7
NUM_WORKERS     = 8
DEVICE          = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
class CodeDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.codes     = df['code'].astype(str).tolist()
        self.labels    = df['label'].tolist()
        self.tokenizer = tokenizer
    def __len__(self):
        return len(self.codes)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.codes[idx],
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }
class AttentionPooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attn = nn.Linear(d_model, 1)
    def forward(self, hidden, attention_mask):
        scores  = self.attn(hidden).squeeze(-1)
        scores  = scores.masked_fill(attention_mask == 0, -1e9)
        weights = torch.softmax(scores, dim=-1)
        return (hidden * weights.unsqueeze(-1)).sum(1)
class DynamicAlphaFusion(nn.Module):
    def __init__(self, d_model, num_levels):
        super().__init__()
        self.gate = nn.Linear(d_model, num_levels)
    def forward(self, base_cls, residuals):
        alphas   = torch.softmax(self.gate(base_cls), dim=-1)
        weighted = sum(alphas[:, i:i+1] * residuals[i] for i in range(len(residuals)))
        return base_cls + weighted
class LaplacianResidualAdapter(nn.Module):
    def __init__(self, d_model, bottleneck_dim):
        super().__init__()
        self.down    = nn.Linear(d_model, bottleneck_dim)
        self.act     = nn.GELU()
        self.up      = nn.Linear(bottleneck_dim, d_model)
        self.dropout = nn.Dropout(0.1)
        nn.init.kaiming_normal_(self.down.weight, nonlinearity='linear')
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)
    def forward(self, x):
        return self.up(self.dropout(self.act(self.down(x))))
class LAPPEFT(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(
            MODEL_NAME,
            output_hidden_states=True,
            torch_dtype=torch.float16,
            trust_remote_code=True
        )
        for p in self.encoder.parameters():
            p.requires_grad = False
        self.pool_layers = nn.ModuleList([
            AttentionPooling(D_MODEL) for _ in range(NUM_LEVELS + 1)
        ])
        self.lras = nn.ModuleList([
            LaplacianResidualAdapter(D_MODEL, BOTTLENECK)
            for _ in range(NUM_LEVELS)
        ])
        self.level_norms = nn.ModuleList([nn.LayerNorm(D_MODEL) for _ in range(NUM_LEVELS)])
        self.fusion      = DynamicAlphaFusion(D_MODEL, NUM_LEVELS)
        self.classifier  = nn.Sequential(
            nn.Linear(D_MODEL, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )
    def encode(self, input_ids, attention_mask):
        with torch.cuda.amp.autocast(dtype=torch.float16):
            out = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
        hidden     = [h.float() for h in out.hidden_states]
        num_layers = len(hidden)
        base = self.pool_layers[NUM_LEVELS](hidden[-1], attention_mask)
        level_indices = [
            min((lvl + 1) * LAYERS_LEVEL, num_layers - 1) for lvl in range(NUM_LEVELS)
        ]
        level_cls = [
            self.pool_layers[lvl](hidden[level_indices[lvl]], attention_mask)
            for lvl in range(NUM_LEVELS)
        ]
        residuals = []
        for lvl in range(NUM_LEVELS):
            diff = level_cls[lvl] - level_cls[lvl + 1] if lvl < NUM_LEVELS - 1 else level_cls[lvl]
            r    = self.lras[lvl](diff)
            residuals.append(self.level_norms[lvl](r))
        recon = self.fusion(base, residuals)
        return recon
    def forward(self, input_ids, attention_mask, cls_embed=None):
        if cls_embed is not None:
            return self.classifier(cls_embed)
        recon = self.encode(input_ids, attention_mask)
        return self.classifier(recon)
def mixup_embeddings(embed, labels, alpha=MIXUP_ALPHA):
    lam   = np.random.beta(alpha, alpha)
    idx   = torch.randperm(embed.size(0), device=embed.device)
    mixed = lam * embed + (1 - lam) * embed[idx]
    return mixed, labels, labels[idx], lam
def rdrop_kl_loss(logits1, logits2):
    p = F.log_softmax(logits1, dim=-1)
    q = F.log_softmax(logits2, dim=-1)
    return (F.kl_div(p, q.exp(), reduction='batchmean') +
            F.kl_div(q, p.exp(), reduction='batchmean')) / 2
def mixup_ce_loss(logits, labels_a, labels_b, lam, weight=None):
    loss_fn = nn.CrossEntropyLoss(weight=weight, label_smoothing=0.05, reduction='none')
    return (lam * loss_fn(logits, labels_a) + (1 - lam) * loss_fn(logits, labels_b)).mean()
def compute_metrics(labels, preds, probs):
    labels = np.array(labels)
    preds  = np.array(preds)
    probs  = np.array(probs)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        'Accuracy':      accuracy_score(labels, preds),
        'Balanced_Acc':  balanced_accuracy_score(labels, preds),
        'Precision':     precision_score(labels, preds, zero_division=0),
        'Recall':        recall_score(labels, preds, zero_division=0),
        'F1':            f1_score(labels, preds, zero_division=0),
        'ROC_AUC':       roc_auc_score(labels, probs),
        'Avg_Precision': average_precision_score(labels, probs),
        'MCC':           matthews_corrcoef(labels, preds),
        'Specificity':   tn / (tn + fp + 1e-9),
        'Sensitivity':   tp / (tp + fn + 1e-9),
        'FPR':           fp / (fp + tn + 1e-9),
        'FNR':           fn / (fn + tp + 1e-9),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn)
    }
def get_encode_fn(model):
    if isinstance(model, nn.DataParallel):
        return model.module.encode
    return model.encode
def train_epoch(model, loader, criterion, optimizer, scheduler, class_weights, step_counter):
    model.train()
    total_loss                       = 0.0
    all_labels, all_preds, all_probs = [], [], []
    optimizer.zero_grad()
    encode_fn = get_encode_fn(model)
    for batch_idx, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)
        with torch.no_grad():
            embed = encode_fn(input_ids, attention_mask)
        use_mixup = (np.random.rand() < MIXUP_PROB)
        if use_mixup:
            mixed_embed, la, lb, lam = mixup_embeddings(embed, labels)
            logits1 = model(None, None, cls_embed=mixed_embed)
            logits2 = model(None, None, cls_embed=mixed_embed)
            ce_loss = mixup_ce_loss(logits1, la, lb, lam, weight=class_weights)
        else:
            logits1 = model(None, None, cls_embed=embed)
            logits2 = model(None, None, cls_embed=embed)
            ce_loss = criterion(logits1, labels)
        kl_loss = rdrop_kl_loss(logits1, logits2)
        loss    = ce_loss + RDROP_ALPHA * kl_loss
        loss    = loss / GRAD_ACCUM
        loss.backward()
        if (batch_idx + 1) % GRAD_ACCUM == 0 or (batch_idx + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            step_counter[0] += 1
        total_loss += (ce_loss + RDROP_ALPHA * kl_loss).item()
        with torch.no_grad():
            probs = torch.softmax(logits1, dim=-1)[:, 1].cpu().numpy()
            preds = logits1.argmax(dim=-1).cpu().numpy()
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)
    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss                       = 0.0
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['label'].to(DEVICE)
            logits         = model(input_ids, attention_mask)
            loss           = criterion(logits, labels)
            total_loss     += loss.item()
            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            preds = logits.argmax(dim=-1).cpu().numpy()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds)
            all_probs.extend(probs)
    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)
def update_swa(swa_model, model):
    src = model.module if isinstance(model, nn.DataParallel) else model
    with torch.no_grad():
        for swa_p, p in zip(swa_model.parameters(), src.parameters()):
            swa_p.data.copy_(swa_p.data + p.data)
def apply_swa_avg(swa_model, n_swa):
    with torch.no_grad():
        for p in swa_model.parameters():
            p.data.div_(n_swa)
def plot_loss_curve(history, save_path='loss_curve.png'):
    epochs = list(range(1, len(history['tr_loss']) + 1))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(epochs, history['tr_loss'], 'b-o', markersize=4, linewidth=1.8, label='Train Loss')
    ax.plot(epochs, history['va_loss'], 'r-o', markersize=4, linewidth=1.8, label='Val Loss')
    ax.set_title('LAPPEFT (Qwen2.5-Coder-1.5B) — Train vs Validation Loss', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Loss curve saved to {save_path}")
def plot_training_curves(history, save_path='training_curves.png'):
    epochs = list(range(1, len(history['tr_loss']) + 1))
    fig, axes = plt.subplots(3, 3, figsize=(18, 14))
    fig.suptitle('LAPPEFT (Qwen2.5-Coder-1.5B) Training Curves', fontsize=16, fontweight='bold')
    def plot_ax(ax, key, title, ylabel):
        ax.plot(epochs, history[f'tr_{key}'], 'b-o', markersize=3, label='Train')
        ax.plot(epochs, history[f'va_{key}'], 'r-o', markersize=3, label='Val')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(True, alpha=0.3)
    plot_ax(axes[0][0], 'loss',    'Loss',          'Loss')
    plot_ax(axes[0][1], 'acc',     'Accuracy',      'Accuracy')
    plot_ax(axes[0][2], 'f1',      'F1 Score',      'F1')
    plot_ax(axes[1][0], 'auc',     'ROC AUC',       'AUC')
    plot_ax(axes[1][1], 'mcc',     'MCC',           'MCC')
    plot_ax(axes[1][2], 'prec',    'Precision',     'Precision')
    plot_ax(axes[2][0], 'rec',     'Recall',        'Recall')
    plot_ax(axes[2][1], 'bacc',    'Balanced Acc',  'Balanced Accuracy')
    plot_ax(axes[2][2], 'avgprec', 'Avg Precision', 'Avg Precision')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Training curves saved to {save_path}")
def plot_final_comparison(best_test_m, swa_test_m, save_path='test_comparison.png'):
    keys      = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Avg_Precision']
    x         = np.arange(len(keys))
    w         = 0.35
    best_vals = [best_test_m[k] for k in keys]
    swa_vals  = [swa_test_m[k] for k in keys] if swa_test_m else None
    fig, ax   = plt.subplots(figsize=(14, 6))
    bars1     = ax.bar(x - w/2, best_vals, w, label='Best Checkpoint', color='steelblue', alpha=0.85)
    if swa_vals:
        bars2 = ax.bar(x + w/2, swa_vals, w, label='SWA Model', color='darkorange', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(keys, rotation=20, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title('Test Set: Best Checkpoint vs SWA Model', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
    if swa_vals:
        for bar in bars2:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Test comparison chart saved to {save_path}")
def plot_confusion_matrix(m, title, save_path):
    cm      = np.array([[m['TN'], m['FP']], [m['FN'], m['TP']]])
    fig, ax = plt.subplots(figsize=(5, 4))
    im      = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred 0', 'Pred 1'])
    ax.set_yticklabels(['True 0', 'True 1'])
    thresh = cm.max() / 2.0
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > thresh else 'black',
                    fontsize=14, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Confusion matrix saved to {save_path}")
def plot_metrics_radar(best_test_m, swa_test_m, save_path='radar_chart.png'):
    keys   = ['Accuracy', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Specificity', 'Sensitivity']
    n      = len(keys)
    angles = [x / float(n) * 2 * np.pi for x in range(n)]
    angles += angles[:1]
    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    def draw(m, color, label):
        vals  = [m[k] for k in keys]
        vals += vals[:1]
        ax.plot(angles, vals, color=color, linewidth=2, label=label)
        ax.fill(angles, vals, color=color, alpha=0.15)
    draw(best_test_m, 'steelblue', 'Best Checkpoint')
    if swa_test_m:
        draw(swa_test_m, 'darkorange', 'SWA Model')
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(keys, size=9)
    ax.set_ylim(0, 1)
    ax.set_title('Test Metrics Radar', fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Radar chart saved to {save_path}")
def main():
    n_gpus = torch.cuda.device_count()
    print(f"Using {n_gpus} GPU(s): {[torch.cuda.get_device_name(i) for i in range(n_gpus)]}")
    train_full = pd.read_csv(TRAIN_CSV)
    test_full  = pd.read_csv(TEST_CSV)
    train_df, val_df = train_test_split(
        train_full, test_size=VAL_SPLIT, random_state=SEED, stratify=train_full['label']
    )
    _, test_df = train_test_split(
        test_full, test_size=0.50, random_state=SEED, stratify=test_full['label']
    )
    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    print(f"Device: {DEVICE}\n")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = 'right'
    train_ds     = CodeDataset(train_df.reset_index(drop=True), tokenizer)
    val_ds       = CodeDataset(val_df.reset_index(drop=True),   tokenizer)
    test_ds      = CodeDataset(test_df.reset_index(drop=True),  tokenizer)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    counts        = train_df['label'].value_counts().sort_index().values
    raw_w         = torch.tensor([1.0 / c for c in counts], dtype=torch.float).to(DEVICE)
    class_weights = raw_w / raw_w.sum() * len(counts)
    base_model = LAPPEFT().to(DEVICE)
    if n_gpus > 1:
        model = nn.DataParallel(base_model, device_ids=list(range(n_gpus)))
    else:
        model = base_model
    swa_model = copy.deepcopy(base_model)
    for p in swa_model.parameters():
        p.requires_grad = False
    total_p   = sum(p.numel() for p in base_model.parameters())
    trainable = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
    print(f"Total Params     : {total_p:,}")
    print(f"Trainable Params : {trainable:,}")
    print(f"Trainable %      : {100.0 * trainable / total_p:.4f}%\n")
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer        = torch.optim.AdamW(
        trainable_params, lr=LR, weight_decay=0.05,
        betas=(0.9, 0.999), eps=1e-8
    )
    effective_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    warmup_steps    = int(WARMUP_RATIO * effective_steps)
    scheduler       = get_cosine_schedule_with_warmup(optimizer, warmup_steps, effective_steps)
    swa_start_epoch = int(SWA_START_RATIO * EPOCHS)
    n_swa           = 0
    step_counter    = [0]
    best_val_f1 = -1.0
    best_epoch  = 0
    best_state  = None
    no_improve  = 0
    history = {
        'tr_loss': [], 'tr_acc': [], 'tr_f1': [], 'tr_auc': [], 'tr_mcc': [],
        'tr_prec': [], 'tr_rec': [], 'tr_bacc': [], 'tr_avgprec': [],
        'va_loss': [], 'va_acc': [], 'va_f1': [], 'va_auc': [], 'va_mcc': [],
        'va_prec': [], 'va_rec': [], 'va_bacc': [], 'va_avgprec': []
    }
    hdr = (f"{'Ep':>3} | {'TrLoss':>8} | {'TrAcc':>7} | {'TrF1':>7} | {'TrAUC':>7} | {'TrMCC':>7} | "
           f"{'VaLoss':>8} | {'VaAcc':>7} | {'VaF1':>7} | {'VaAUC':>7} | {'VaMCC':>7} | {'Note':<12}")
    sep = "-" * len(hdr)
    print(sep)
    print(hdr)
    print(sep)
    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_m = train_epoch(model, train_loader, criterion, optimizer,
                                    scheduler, class_weights, step_counter)
        va_loss, va_m = eval_epoch(model, val_loader, criterion)
        history['tr_loss'].append(tr_loss)
        history['tr_acc'].append(tr_m['Accuracy'])
        history['tr_f1'].append(tr_m['F1'])
        history['tr_auc'].append(tr_m['ROC_AUC'])
        history['tr_mcc'].append(tr_m['MCC'])
        history['tr_prec'].append(tr_m['Precision'])
        history['tr_rec'].append(tr_m['Recall'])
        history['tr_bacc'].append(tr_m['Balanced_Acc'])
        history['tr_avgprec'].append(tr_m['Avg_Precision'])
        history['va_loss'].append(va_loss)
        history['va_acc'].append(va_m['Accuracy'])
        history['va_f1'].append(va_m['F1'])
        history['va_auc'].append(va_m['ROC_AUC'])
        history['va_mcc'].append(va_m['MCC'])
        history['va_prec'].append(va_m['Precision'])
        history['va_rec'].append(va_m['Recall'])
        history['va_bacc'].append(va_m['Balanced_Acc'])
        history['va_avgprec'].append(va_m['Avg_Precision'])
        note = ""
        if epoch > swa_start_epoch:
            update_swa(swa_model, model)
            n_swa += 1
            note += "SWA "
        is_best = va_m['F1'] > best_val_f1
        if is_best:
            best_val_f1 = va_m['F1']
            best_epoch  = epoch
            src_state   = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            best_state  = {k: v.cpu().clone() for k, v in src_state.items()}
            no_improve  = 0
            note       += "<--"
        else:
            no_improve += 1
        print(
            f"{epoch:>3} | {tr_loss:>8.4f} | {tr_m['Accuracy']:>7.4f} | {tr_m['F1']:>7.4f} | "
            f"{tr_m['ROC_AUC']:>7.4f} | {tr_m['MCC']:>7.4f} | "
            f"{va_loss:>8.4f} | {va_m['Accuracy']:>7.4f} | {va_m['F1']:>7.4f} | "
            f"{va_m['ROC_AUC']:>7.4f} | {va_m['MCC']:>7.4f} | {note:<12}"
        )
        if no_improve >= PATIENCE:
            print(f"\n  Early stopping at epoch {epoch} (no val F1 improvement for {PATIENCE} epochs)")
            break
    print(sep)
    print(f"\nBest Epoch: {best_epoch}  |  Best Val F1: {best_val_f1:.4f}")
    print("\n--- Evaluating: Best Checkpoint Model ---")
    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    else:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    _, best_test_m = eval_epoch(model, test_loader, criterion)
    print("\n--- Evaluating: SWA Model ---")
    swa_test_m = None
    if n_swa > 0:
        apply_swa_avg(swa_model, n_swa)
        swa_model.to(DEVICE)
        if n_gpus > 1:
            swa_eval = nn.DataParallel(swa_model, device_ids=list(range(n_gpus)))
        else:
            swa_eval = swa_model
        _, swa_test_m = eval_epoch(swa_eval, test_loader, criterion)
    print(f"\n{'='*60}")
    print(f"  Val F1 trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_f1'][-5:]]}")
    print(f"  Val Loss trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_loss'][-5:]]}")
    print(f"\n{'='*60}")
    print(f"  TEST RESULTS — Best Checkpoint (Epoch {best_epoch})")
    print(f"{'='*60}")
    scalar_keys = [k for k in best_test_m if k not in ('TP', 'TN', 'FP', 'FN')]
    for k in scalar_keys:
        print(f"  {k:<22}: {best_test_m[k]:.4f}")
    print(f"  {'TP':<22}: {best_test_m['TP']}")
    print(f"  {'TN':<22}: {best_test_m['TN']}")
    print(f"  {'FP':<22}: {best_test_m['FP']}")
    print(f"  {'FN':<22}: {best_test_m['FN']}")
    if swa_test_m is not None:
        print(f"\n{'='*60}")
        print(f"  TEST RESULTS — SWA Model (averaged over last {n_swa} epochs)")
        print(f"{'='*60}")
        for k in scalar_keys:
            print(f"  {k:<22}: {swa_test_m[k]:.4f}")
        print(f"  {'TP':<22}: {swa_test_m['TP']}")
        print(f"  {'TN':<22}: {swa_test_m['TN']}")
        print(f"  {'FP':<22}: {swa_test_m['FP']}")
        print(f"  {'FN':<22}: {swa_test_m['FN']}")
        print(f"\n{'='*60}")
        print(f"  COMPARISON: Best Checkpoint vs SWA")
        print(f"{'='*60}")
        compare_keys = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall']
        print(f"  {'Metric':<22} {'BestCkpt':>12} {'SWA':>12} {'Delta':>12}")
        print(f"  {'-'*58}")
        for k in compare_keys:
            delta  = swa_test_m[k] - best_test_m[k]
            symbol = "+" if delta >= 0 else ""
            print(f"  {k:<22} {best_test_m[k]:>12.4f} {swa_test_m[k]:>12.4f} {symbol+f'{delta:.4f}':>12}")
    plot_loss_curve(history,       save_path='loss_curve.png')
    plot_training_curves(history,  save_path='training_curves.png')
    plot_final_comparison(best_test_m, swa_test_m, save_path='test_comparison.png')
    plot_confusion_matrix(best_test_m, f'Confusion Matrix — Best Ckpt (Ep {best_epoch})', 'cm_best.png')
    if swa_test_m:
        plot_confusion_matrix(swa_test_m, f'Confusion Matrix — SWA ({n_swa} epochs)', 'cm_swa.png')
    plot_metrics_radar(best_test_m, swa_test_m, save_path='radar_chart.png')
    print("\nAll figures saved.")
if __name__ == "__main__":
    main()

# HierAdapter with Qwen2.5-Coder-1.5B, 1024token - code vulnerability

In [ ]:
import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix,
    balanced_accuracy_score, average_precision_score
)
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings


warnings.filterwarnings('ignore')
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
MODEL_NAME      = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
TRAIN_CSV       = '/code vul/trqw.csv'
TEST_CSV        = '/code vul/teqw.csv'
MAX_LENGTH      = 1024
D_MODEL         = 1536
BATCH_SIZE      = 16
GRAD_ACCUM      = 2
EPOCHS          = 30
LR              = 5e-5
WARMUP_RATIO    = 0.15
BOTTLENECK      = 6
NUM_LEVELS      = 4
LAYERS_LEVEL    = 7
VAL_SPLIT       = 0.20
SEED            = 42
RDROP_ALPHA     = 0.3
MIXUP_ALPHA     = 0.2
MIXUP_PROB      = 0.5
SWA_START_RATIO = 0.6
SWA_LR          = 1e-5
PATIENCE        = 7
NUM_WORKERS     = 8
DEVICE          = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
class CodeDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.codes     = df['func_cleaned'].astype(str).tolist()
        self.labels    = df['label'].tolist()
        self.tokenizer = tokenizer
    def __len__(self):
        return len(self.codes)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.codes[idx],
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }
class AttentionPooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attn = nn.Linear(d_model, 1)
    def forward(self, hidden, attention_mask):
        scores  = self.attn(hidden).squeeze(-1)
        scores  = scores.masked_fill(attention_mask == 0, -1e9)
        weights = torch.softmax(scores, dim=-1)
        return (hidden * weights.unsqueeze(-1)).sum(1)
class DynamicAlphaFusion(nn.Module):
    def __init__(self, d_model, num_levels):
        super().__init__()
        self.gate = nn.Linear(d_model, num_levels)
    def forward(self, base_cls, residuals):
        alphas   = torch.softmax(self.gate(base_cls), dim=-1)
        weighted = sum(alphas[:, i:i+1] * residuals[i] for i in range(len(residuals)))
        return base_cls + weighted
class LaplacianResidualAdapter(nn.Module):
    def __init__(self, d_model, bottleneck_dim):
        super().__init__()
        self.down    = nn.Linear(d_model, bottleneck_dim)
        self.act     = nn.GELU()
        self.up      = nn.Linear(bottleneck_dim, d_model)
        self.dropout = nn.Dropout(0.1)
        nn.init.kaiming_normal_(self.down.weight, nonlinearity='linear')
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)
    def forward(self, x):
        return self.up(self.dropout(self.act(self.down(x))))
class LAPPEFT(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(
            MODEL_NAME,
            output_hidden_states=True,
            torch_dtype=torch.float16,
            trust_remote_code=True
        )
        for p in self.encoder.parameters():
            p.requires_grad = False
        self.pool_layers = nn.ModuleList([
            AttentionPooling(D_MODEL) for _ in range(NUM_LEVELS + 1)
        ])
        self.lras = nn.ModuleList([
            LaplacianResidualAdapter(D_MODEL, BOTTLENECK)
            for _ in range(NUM_LEVELS)
        ])
        self.level_norms = nn.ModuleList([nn.LayerNorm(D_MODEL) for _ in range(NUM_LEVELS)])
        self.fusion      = DynamicAlphaFusion(D_MODEL, NUM_LEVELS)
        self.classifier  = nn.Sequential(
            nn.Linear(D_MODEL, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )
    def encode(self, input_ids, attention_mask):
        with torch.cuda.amp.autocast(dtype=torch.float16):
            out = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
        hidden     = [h.float() for h in out.hidden_states]
        num_layers = len(hidden)
        base = self.pool_layers[NUM_LEVELS](hidden[-1], attention_mask)
        level_indices = [
            min((lvl + 1) * LAYERS_LEVEL, num_layers - 1) for lvl in range(NUM_LEVELS)
        ]
        level_cls = [
            self.pool_layers[lvl](hidden[level_indices[lvl]], attention_mask)
            for lvl in range(NUM_LEVELS)
        ]
        residuals = []
        for lvl in range(NUM_LEVELS):
            diff = level_cls[lvl] - level_cls[lvl + 1] if lvl < NUM_LEVELS - 1 else level_cls[lvl]
            r    = self.lras[lvl](diff)
            residuals.append(self.level_norms[lvl](r))
        recon = self.fusion(base, residuals)
        return recon
    def forward(self, input_ids, attention_mask, cls_embed=None):
        if cls_embed is not None:
            return self.classifier(cls_embed)
        recon = self.encode(input_ids, attention_mask)
        return self.classifier(recon)
def mixup_embeddings(embed, labels, alpha=MIXUP_ALPHA):
    lam   = np.random.beta(alpha, alpha)
    idx   = torch.randperm(embed.size(0), device=embed.device)
    mixed = lam * embed + (1 - lam) * embed[idx]
    return mixed, labels, labels[idx], lam
def rdrop_kl_loss(logits1, logits2):
    p = F.log_softmax(logits1, dim=-1)
    q = F.log_softmax(logits2, dim=-1)
    return (F.kl_div(p, q.exp(), reduction='batchmean') +
            F.kl_div(q, p.exp(), reduction='batchmean')) / 2
def mixup_ce_loss(logits, labels_a, labels_b, lam, weight=None):
    loss_fn = nn.CrossEntropyLoss(weight=weight, label_smoothing=0.05, reduction='none')
    return (lam * loss_fn(logits, labels_a) + (1 - lam) * loss_fn(logits, labels_b)).mean()
def compute_metrics(labels, preds, probs):
    labels = np.array(labels)
    preds  = np.array(preds)
    probs  = np.array(probs)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        'Accuracy':      accuracy_score(labels, preds),
        'Balanced_Acc':  balanced_accuracy_score(labels, preds),
        'Precision':     precision_score(labels, preds, zero_division=0),
        'Recall':        recall_score(labels, preds, zero_division=0),
        'F1':            f1_score(labels, preds, zero_division=0),
        'ROC_AUC':       roc_auc_score(labels, probs),
        'Avg_Precision': average_precision_score(labels, probs),
        'MCC':           matthews_corrcoef(labels, preds),
        'Specificity':   tn / (tn + fp + 1e-9),
        'Sensitivity':   tp / (tp + fn + 1e-9),
        'FPR':           fp / (fp + tn + 1e-9),
        'FNR':           fn / (fn + tp + 1e-9),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn)
    }
def get_encode_fn(model):
    if isinstance(model, nn.DataParallel):
        return model.module.encode
    return model.encode
def train_epoch(model, loader, criterion, optimizer, scheduler, class_weights, step_counter):
    model.train()
    total_loss                       = 0.0
    all_labels, all_preds, all_probs = [], [], []
    optimizer.zero_grad()
    encode_fn = get_encode_fn(model)
    for batch_idx, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)
        with torch.no_grad():
            embed = encode_fn(input_ids, attention_mask)
        use_mixup = (np.random.rand() < MIXUP_PROB)
        if use_mixup:
            mixed_embed, la, lb, lam = mixup_embeddings(embed, labels)
            logits1 = model(None, None, cls_embed=mixed_embed)
            logits2 = model(None, None, cls_embed=mixed_embed)
            ce_loss = mixup_ce_loss(logits1, la, lb, lam, weight=class_weights)
        else:
            logits1 = model(None, None, cls_embed=embed)
            logits2 = model(None, None, cls_embed=embed)
            ce_loss = criterion(logits1, labels)
        kl_loss = rdrop_kl_loss(logits1, logits2)
        loss    = ce_loss + RDROP_ALPHA * kl_loss
        loss    = loss / GRAD_ACCUM
        loss.backward()
        if (batch_idx + 1) % GRAD_ACCUM == 0 or (batch_idx + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            step_counter[0] += 1
        total_loss += (ce_loss + RDROP_ALPHA * kl_loss).item()
        with torch.no_grad():
            probs = torch.softmax(logits1, dim=-1)[:, 1].cpu().numpy()
            preds = logits1.argmax(dim=-1).cpu().numpy()
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)
    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss                       = 0.0
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['label'].to(DEVICE)
            logits         = model(input_ids, attention_mask)
            loss           = criterion(logits, labels)
            total_loss     += loss.item()
            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            preds = logits.argmax(dim=-1).cpu().numpy()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds)
            all_probs.extend(probs)
    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)
def update_swa(swa_model, model):
    src = model.module if isinstance(model, nn.DataParallel) else model
    with torch.no_grad():
        for swa_p, p in zip(swa_model.parameters(), src.parameters()):
            swa_p.data.copy_(swa_p.data + p.data)
def apply_swa_avg(swa_model, n_swa):
    with torch.no_grad():
        for p in swa_model.parameters():
            p.data.div_(n_swa)
def plot_loss_curve(history, save_path='loss_curve.png'):
    epochs = list(range(1, len(history['tr_loss']) + 1))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(epochs, history['tr_loss'], 'b-o', markersize=4, linewidth=1.8, label='Train Loss')
    ax.plot(epochs, history['va_loss'], 'r-o', markersize=4, linewidth=1.8, label='Val Loss')
    ax.set_title('LAPPEFT (Qwen2.5-Coder-1.5B) — Train vs Validation Loss', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Loss curve saved to {save_path}")
def plot_training_curves(history, save_path='training_curves.png'):
    epochs = list(range(1, len(history['tr_loss']) + 1))
    fig, axes = plt.subplots(3, 3, figsize=(18, 14))
    fig.suptitle('LAPPEFT (Qwen2.5-Coder-1.5B) Training Curves', fontsize=16, fontweight='bold')
    def plot_ax(ax, key, title, ylabel):
        ax.plot(epochs, history[f'tr_{key}'], 'b-o', markersize=3, label='Train')
        ax.plot(epochs, history[f'va_{key}'], 'r-o', markersize=3, label='Val')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(True, alpha=0.3)
    plot_ax(axes[0][0], 'loss',    'Loss',          'Loss')
    plot_ax(axes[0][1], 'acc',     'Accuracy',      'Accuracy')
    plot_ax(axes[0][2], 'f1',      'F1 Score',      'F1')
    plot_ax(axes[1][0], 'auc',     'ROC AUC',       'AUC')
    plot_ax(axes[1][1], 'mcc',     'MCC',           'MCC')
    plot_ax(axes[1][2], 'prec',    'Precision',     'Precision')
    plot_ax(axes[2][0], 'rec',     'Recall',        'Recall')
    plot_ax(axes[2][1], 'bacc',    'Balanced Acc',  'Balanced Accuracy')
    plot_ax(axes[2][2], 'avgprec', 'Avg Precision', 'Avg Precision')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Training curves saved to {save_path}")
def plot_final_comparison(best_test_m, swa_test_m, save_path='test_comparison.png'):
    keys      = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Avg_Precision']
    x         = np.arange(len(keys))
    w         = 0.35
    best_vals = [best_test_m[k] for k in keys]
    swa_vals  = [swa_test_m[k] for k in keys] if swa_test_m else None
    fig, ax   = plt.subplots(figsize=(14, 6))
    bars1     = ax.bar(x - w/2, best_vals, w, label='Best Checkpoint', color='steelblue', alpha=0.85)
    if swa_vals:
        bars2 = ax.bar(x + w/2, swa_vals, w, label='SWA Model', color='darkorange', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(keys, rotation=20, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title('Test Set: Best Checkpoint vs SWA Model', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
    if swa_vals:
        for bar in bars2:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Test comparison chart saved to {save_path}")
def plot_confusion_matrix(m, title, save_path):
    cm      = np.array([[m['TN'], m['FP']], [m['FN'], m['TP']]])
    fig, ax = plt.subplots(figsize=(5, 4))
    im      = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred 0', 'Pred 1'])
    ax.set_yticklabels(['True 0', 'True 1'])
    thresh = cm.max() / 2.0
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > thresh else 'black',
                    fontsize=14, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Confusion matrix saved to {save_path}")
def plot_metrics_radar(best_test_m, swa_test_m, save_path='radar_chart.png'):
    keys   = ['Accuracy', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Specificity', 'Sensitivity']
    n      = len(keys)
    angles = [x / float(n) * 2 * np.pi for x in range(n)]
    angles += angles[:1]
    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    def draw(m, color, label):
        vals  = [m[k] for k in keys]
        vals += vals[:1]
        ax.plot(angles, vals, color=color, linewidth=2, label=label)
        ax.fill(angles, vals, color=color, alpha=0.15)
    draw(best_test_m, 'steelblue', 'Best Checkpoint')
    if swa_test_m:
        draw(swa_test_m, 'darkorange', 'SWA Model')
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(keys, size=9)
    ax.set_ylim(0, 1)
    ax.set_title('Test Metrics Radar', fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Radar chart saved to {save_path}")
def main():
    n_gpus = torch.cuda.device_count()
    print(f"Using {n_gpus} GPU(s): {[torch.cuda.get_device_name(i) for i in range(n_gpus)]}")
    train_full = pd.read_csv(TRAIN_CSV)
    test_full  = pd.read_csv(TEST_CSV)
    train_df, val_df = train_test_split(
        train_full, test_size=VAL_SPLIT, random_state=SEED, stratify=train_full['label']
    )
    _, test_df = train_test_split(
        test_full, test_size=0.50, random_state=SEED, stratify=test_full['label']
    )
    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    print(f"Device: {DEVICE}\n")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = 'right'
    train_ds     = CodeDataset(train_df.reset_index(drop=True), tokenizer)
    val_ds       = CodeDataset(val_df.reset_index(drop=True),   tokenizer)
    test_ds      = CodeDataset(test_df.reset_index(drop=True),  tokenizer)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    counts        = train_df['label'].value_counts().sort_index().values
    raw_w         = torch.tensor([1.0 / c for c in counts], dtype=torch.float).to(DEVICE)
    class_weights = raw_w / raw_w.sum() * len(counts)
    base_model = LAPPEFT().to(DEVICE)
    if n_gpus > 1:
        model = nn.DataParallel(base_model, device_ids=list(range(n_gpus)))
    else:
        model = base_model
    swa_model = copy.deepcopy(base_model)
    for p in swa_model.parameters():
        p.requires_grad = False
    total_p   = sum(p.numel() for p in base_model.parameters())
    trainable = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
    print(f"Total Params     : {total_p:,}")
    print(f"Trainable Params : {trainable:,}")
    print(f"Trainable %      : {100.0 * trainable / total_p:.4f}%\n")
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer        = torch.optim.AdamW(
        trainable_params, lr=LR, weight_decay=0.05,
        betas=(0.9, 0.999), eps=1e-8
    )
    effective_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    warmup_steps    = int(WARMUP_RATIO * effective_steps)
    scheduler       = get_cosine_schedule_with_warmup(optimizer, warmup_steps, effective_steps)
    swa_start_epoch = int(SWA_START_RATIO * EPOCHS)
    n_swa           = 0
    step_counter    = [0]
    best_val_f1 = -1.0
    best_epoch  = 0
    best_state  = None
    no_improve  = 0
    history = {
        'tr_loss': [], 'tr_acc': [], 'tr_f1': [], 'tr_auc': [], 'tr_mcc': [],
        'tr_prec': [], 'tr_rec': [], 'tr_bacc': [], 'tr_avgprec': [],
        'va_loss': [], 'va_acc': [], 'va_f1': [], 'va_auc': [], 'va_mcc': [],
        'va_prec': [], 'va_rec': [], 'va_bacc': [], 'va_avgprec': []
    }
    hdr = (f"{'Ep':>3} | {'TrLoss':>8} | {'TrAcc':>7} | {'TrF1':>7} | {'TrAUC':>7} | {'TrMCC':>7} | "
           f"{'VaLoss':>8} | {'VaAcc':>7} | {'VaF1':>7} | {'VaAUC':>7} | {'VaMCC':>7} | {'Note':<12}")
    sep = "-" * len(hdr)
    print(sep)
    print(hdr)
    print(sep)
    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_m = train_epoch(model, train_loader, criterion, optimizer,
                                    scheduler, class_weights, step_counter)
        va_loss, va_m = eval_epoch(model, val_loader, criterion)
        history['tr_loss'].append(tr_loss)
        history['tr_acc'].append(tr_m['Accuracy'])
        history['tr_f1'].append(tr_m['F1'])
        history['tr_auc'].append(tr_m['ROC_AUC'])
        history['tr_mcc'].append(tr_m['MCC'])
        history['tr_prec'].append(tr_m['Precision'])
        history['tr_rec'].append(tr_m['Recall'])
        history['tr_bacc'].append(tr_m['Balanced_Acc'])
        history['tr_avgprec'].append(tr_m['Avg_Precision'])
        history['va_loss'].append(va_loss)
        history['va_acc'].append(va_m['Accuracy'])
        history['va_f1'].append(va_m['F1'])
        history['va_auc'].append(va_m['ROC_AUC'])
        history['va_mcc'].append(va_m['MCC'])
        history['va_prec'].append(va_m['Precision'])
        history['va_rec'].append(va_m['Recall'])
        history['va_bacc'].append(va_m['Balanced_Acc'])
        history['va_avgprec'].append(va_m['Avg_Precision'])
        note = ""
        if epoch > swa_start_epoch:
            update_swa(swa_model, model)
            n_swa += 1
            note += "SWA "
        is_best = va_m['F1'] > best_val_f1
        if is_best:
            best_val_f1 = va_m['F1']
            best_epoch  = epoch
            src_state   = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            best_state  = {k: v.cpu().clone() for k, v in src_state.items()}
            no_improve  = 0
            note       += "<--"
        else:
            no_improve += 1
        print(
            f"{epoch:>3} | {tr_loss:>8.4f} | {tr_m['Accuracy']:>7.4f} | {tr_m['F1']:>7.4f} | "
            f"{tr_m['ROC_AUC']:>7.4f} | {tr_m['MCC']:>7.4f} | "
            f"{va_loss:>8.4f} | {va_m['Accuracy']:>7.4f} | {va_m['F1']:>7.4f} | "
            f"{va_m['ROC_AUC']:>7.4f} | {va_m['MCC']:>7.4f} | {note:<12}"
        )
        if no_improve >= PATIENCE:
            print(f"\n  Early stopping at epoch {epoch} (no val F1 improvement for {PATIENCE} epochs)")
            break
    print(sep)
    print(f"\nBest Epoch: {best_epoch}  |  Best Val F1: {best_val_f1:.4f}")
    print("\n--- Evaluating: Best Checkpoint Model ---")
    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    else:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    _, best_test_m = eval_epoch(model, test_loader, criterion)
    print("\n--- Evaluating: SWA Model ---")
    swa_test_m = None
    if n_swa > 0:
        apply_swa_avg(swa_model, n_swa)
        swa_model.to(DEVICE)
        if n_gpus > 1:
            swa_eval = nn.DataParallel(swa_model, device_ids=list(range(n_gpus)))
        else:
            swa_eval = swa_model
        _, swa_test_m = eval_epoch(swa_eval, test_loader, criterion)
    print(f"\n{'='*60}")
    print(f"  Val F1 trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_f1'][-5:]]}")
    print(f"  Val Loss trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_loss'][-5:]]}")
    print(f"\n{'='*60}")
    print(f"  TEST RESULTS — Best Checkpoint (Epoch {best_epoch})")
    print(f"{'='*60}")
    scalar_keys = [k for k in best_test_m if k not in ('TP', 'TN', 'FP', 'FN')]
    for k in scalar_keys:
        print(f"  {k:<22}: {best_test_m[k]:.4f}")
    print(f"  {'TP':<22}: {best_test_m['TP']}")
    print(f"  {'TN':<22}: {best_test_m['TN']}")
    print(f"  {'FP':<22}: {best_test_m['FP']}")
    print(f"  {'FN':<22}: {best_test_m['FN']}")
    if swa_test_m is not None:
        print(f"\n{'='*60}")
        print(f"  TEST RESULTS — SWA Model (averaged over last {n_swa} epochs)")
        print(f"{'='*60}")
        for k in scalar_keys:
            print(f"  {k:<22}: {swa_test_m[k]:.4f}")
        print(f"  {'TP':<22}: {swa_test_m['TP']}")
        print(f"  {'TN':<22}: {swa_test_m['TN']}")
        print(f"  {'FP':<22}: {swa_test_m['FP']}")
        print(f"  {'FN':<22}: {swa_test_m['FN']}")
        print(f"\n{'='*60}")
        print(f"  COMPARISON: Best Checkpoint vs SWA")
        print(f"{'='*60}")
        compare_keys = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall']
        print(f"  {'Metric':<22} {'BestCkpt':>12} {'SWA':>12} {'Delta':>12}")
        print(f"  {'-'*58}")
        for k in compare_keys:
            delta  = swa_test_m[k] - best_test_m[k]
            symbol = "+" if delta >= 0 else ""
            print(f"  {k:<22} {best_test_m[k]:>12.4f} {swa_test_m[k]:>12.4f} {symbol+f'{delta:.4f}':>12}")
    plot_loss_curve(history,       save_path='loss_curve.png')
    plot_training_curves(history,  save_path='training_curves.png')
    plot_final_comparison(best_test_m, swa_test_m, save_path='test_comparison.png')
    plot_confusion_matrix(best_test_m, f'Confusion Matrix — Best Ckpt (Ep {best_epoch})', 'cm_best.png')
    if swa_test_m:
        plot_confusion_matrix(swa_test_m, f'Confusion Matrix — SWA ({n_swa} epochs)', 'cm_swa.png')
    plot_metrics_radar(best_test_m, swa_test_m, save_path='radar_chart.png')
    print("\nAll figures saved.")
if __name__ == "__main__":
    main()

# HierAdapater with UniXcoder (Frozen) in Code_vulnerability

In [ ]:
import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix,
    balanced_accuracy_score, average_precision_score
)
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

MODEL_NAME      = "microsoft/unixcoder-base"
TRAIN_CSV       = '/traincodex.csv'
TEST_CSV        = 'testcodex.csv'
MAX_LENGTH      = 512
D_MODEL         = 768
BATCH_SIZE      = 16
GRAD_ACCUM      = 2
EPOCHS          = 30
LR              = 5e-5
WARMUP_RATIO    = 0.15
BOTTLENECK      = 6
NUM_LEVELS      = 4
LAYERS_LEVEL    = 3
VAL_SPLIT       = 0.20
SEED            = 42
RDROP_ALPHA     = 0.3
MIXUP_ALPHA     = 0.2
MIXUP_PROB      = 0.5
SWA_START_RATIO = 0.6
SWA_LR          = 1e-5
PATIENCE        = 7
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


class CodeDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.codes     = df['code'].astype(str).tolist()
        self.labels    = df['label'].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.codes[idx],
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


class AttentionPooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attn = nn.Linear(d_model, 1)

    def forward(self, hidden, attention_mask):
        scores  = self.attn(hidden).squeeze(-1)
        scores  = scores.masked_fill(attention_mask == 0, -1e9)
        weights = torch.softmax(scores, dim=-1)
        return (hidden * weights.unsqueeze(-1)).sum(1)


class DynamicAlphaFusion(nn.Module):
    def __init__(self, d_model, num_levels):
        super().__init__()
        self.gate = nn.Linear(d_model, num_levels)

    def forward(self, base_cls, residuals):
        alphas   = torch.softmax(self.gate(base_cls), dim=-1)
        weighted = sum(alphas[:, i:i+1] * residuals[i] for i in range(len(residuals)))
        return base_cls + weighted


class LaplacianResidualAdapter(nn.Module):
    def __init__(self, d_model, bottleneck_dim):
        super().__init__()
        self.down    = nn.Linear(d_model, bottleneck_dim)
        self.act     = nn.GELU()
        self.up      = nn.Linear(bottleneck_dim, d_model)
        self.dropout = nn.Dropout(0.1)
        nn.init.kaiming_normal_(self.down.weight, nonlinearity='linear')
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return self.up(self.dropout(self.act(self.down(x))))


class LAPPEFT(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        for p in self.encoder.parameters():
            p.requires_grad = False

        self.pool_layers = nn.ModuleList([
            AttentionPooling(D_MODEL) for _ in range(NUM_LEVELS + 1)
        ])

        self.lras = nn.ModuleList([
            LaplacianResidualAdapter(D_MODEL, BOTTLENECK)
            for _ in range(NUM_LEVELS)
        ])

        self.level_norms = nn.ModuleList([nn.LayerNorm(D_MODEL) for _ in range(NUM_LEVELS)])

        self.fusion = DynamicAlphaFusion(D_MODEL, NUM_LEVELS)

        self.classifier = nn.Sequential(
            nn.Linear(D_MODEL, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )

    def encode(self, input_ids, attention_mask):
        out        = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden     = out.hidden_states
        num_layers = len(hidden)

        base = self.pool_layers[NUM_LEVELS](hidden[-1], attention_mask)

        level_indices = [
            min((lvl + 1) * LAYERS_LEVEL, num_layers - 1) for lvl in range(NUM_LEVELS)
        ]

        level_cls = [
            self.pool_layers[lvl](hidden[level_indices[lvl]], attention_mask)
            for lvl in range(NUM_LEVELS)
        ]

        residuals = []
        for lvl in range(NUM_LEVELS):
            diff = level_cls[lvl] - level_cls[lvl + 1] if lvl < NUM_LEVELS - 1 else level_cls[lvl]
            r    = self.lras[lvl](diff)
            residuals.append(self.level_norms[lvl](r))

        recon = self.fusion(base, residuals)
        return recon

    def forward(self, input_ids, attention_mask, cls_embed=None):
        if cls_embed is not None:
            return self.classifier(cls_embed)
        recon = self.encode(input_ids, attention_mask)
        return self.classifier(recon)


def mixup_embeddings(embed, labels, alpha=MIXUP_ALPHA):
    lam   = np.random.beta(alpha, alpha)
    idx   = torch.randperm(embed.size(0), device=embed.device)
    mixed = lam * embed + (1 - lam) * embed[idx]
    return mixed, labels, labels[idx], lam


def rdrop_kl_loss(logits1, logits2):
    p = F.log_softmax(logits1, dim=-1)
    q = F.log_softmax(logits2, dim=-1)
    return (F.kl_div(p, q.exp(), reduction='batchmean') +
            F.kl_div(q, p.exp(), reduction='batchmean')) / 2


def mixup_ce_loss(logits, labels_a, labels_b, lam, weight=None):
    loss_fn = nn.CrossEntropyLoss(weight=weight, label_smoothing=0.05, reduction='none')
    return (lam * loss_fn(logits, labels_a) + (1 - lam) * loss_fn(logits, labels_b)).mean()


def compute_metrics(labels, preds, probs):
    labels = np.array(labels)
    preds  = np.array(preds)
    probs  = np.array(probs)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        'Accuracy':      accuracy_score(labels, preds),
        'Balanced_Acc':  balanced_accuracy_score(labels, preds),
        'Precision':     precision_score(labels, preds, zero_division=0),
        'Recall':        recall_score(labels, preds, zero_division=0),
        'F1':            f1_score(labels, preds, zero_division=0),
        'ROC_AUC':       roc_auc_score(labels, probs),
        'Avg_Precision': average_precision_score(labels, probs),
        'MCC':           matthews_corrcoef(labels, preds),
        'Specificity':   tn / (tn + fp + 1e-9),
        'Sensitivity':   tp / (tp + fn + 1e-9),
        'FPR':           fp / (fp + tn + 1e-9),
        'FNR':           fn / (fn + tp + 1e-9),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn)
    }


def train_epoch(model, loader, criterion, optimizer, scheduler, class_weights, step_counter):
    model.train()
    total_loss                       = 0.0
    all_labels, all_preds, all_probs = [], [], []
    optimizer.zero_grad()

    for batch_idx, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)

        with torch.no_grad():
            embed = model.encode(input_ids, attention_mask)

        use_mixup = (np.random.rand() < MIXUP_PROB)

        if use_mixup:
            mixed_embed, la, lb, lam = mixup_embeddings(embed, labels)
            logits1 = model(None, None, cls_embed=mixed_embed)
            logits2 = model(None, None, cls_embed=mixed_embed)
            ce_loss = mixup_ce_loss(logits1, la, lb, lam, weight=class_weights)
        else:
            logits1 = model(None, None, cls_embed=embed)
            logits2 = model(None, None, cls_embed=embed)
            ce_loss = criterion(logits1, labels)

        kl_loss = rdrop_kl_loss(logits1, logits2)
        loss    = ce_loss + RDROP_ALPHA * kl_loss
        loss    = loss / GRAD_ACCUM
        loss.backward()

        if (batch_idx + 1) % GRAD_ACCUM == 0 or (batch_idx + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            step_counter[0] += 1

        total_loss += (ce_loss + RDROP_ALPHA * kl_loss).item()

        with torch.no_grad():
            probs = torch.softmax(logits1, dim=-1)[:, 1].cpu().numpy()
            preds = logits1.argmax(dim=-1).cpu().numpy()

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)

    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss                       = 0.0
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['label'].to(DEVICE)

            logits     = model(input_ids, attention_mask)
            loss       = criterion(logits, labels)
            total_loss += loss.item()

            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            preds = logits.argmax(dim=-1).cpu().numpy()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds)
            all_probs.extend(probs)

    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)


def update_swa(swa_model, model):
    with torch.no_grad():
        for swa_p, p in zip(swa_model.parameters(), model.parameters()):
            swa_p.data.copy_(swa_p.data + p.data)


def apply_swa_avg(swa_model, n_swa):
    with torch.no_grad():
        for p in swa_model.parameters():
            p.data.div_(n_swa)


def plot_training_curves(history, save_path='training_curves.png'):
    epochs = list(range(1, len(history['tr_loss']) + 1))
    fig, axes = plt.subplots(3, 3, figsize=(18, 14))
    fig.suptitle('LAPPEFT (UniXcoder) Training Curves', fontsize=16, fontweight='bold')

    def plot_ax(ax, key, title, ylabel):
        ax.plot(epochs, history[f'tr_{key}'], 'b-o', markersize=3, label='Train')
        ax.plot(epochs, history[f'va_{key}'], 'r-o', markersize=3, label='Val')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(True, alpha=0.3)

    plot_ax(axes[0][0], 'loss',    'Loss',          'Loss')
    plot_ax(axes[0][1], 'acc',     'Accuracy',      'Accuracy')
    plot_ax(axes[0][2], 'f1',      'F1 Score',      'F1')
    plot_ax(axes[1][0], 'auc',     'ROC AUC',       'AUC')
    plot_ax(axes[1][1], 'mcc',     'MCC',           'MCC')
    plot_ax(axes[1][2], 'prec',    'Precision',     'Precision')
    plot_ax(axes[2][0], 'rec',     'Recall',        'Recall')
    plot_ax(axes[2][1], 'bacc',    'Balanced Acc',  'Balanced Accuracy')
    plot_ax(axes[2][2], 'avgprec', 'Avg Precision', 'Avg Precision')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Training curves saved to {save_path}")




def plot_confusion_matrix(m, title, save_path):
    cm      = np.array([[m['TN'], m['FP']], [m['FN'], m['TP']]])
    fig, ax = plt.subplots(figsize=(5, 4))
    im      = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred 0', 'Pred 1'])
    ax.set_yticklabels(['True 0', 'True 1'])
    thresh = cm.max() / 2.0
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > thresh else 'black',
                    fontsize=14, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Confusion matrix saved to {save_path}")





def main():
    train_full = pd.read_csv(TRAIN_CSV)
    test_full  = pd.read_csv(TEST_CSV)

    train_df, val_df = train_test_split(
        train_full, test_size=VAL_SPLIT, random_state=SEED, stratify=train_full['label']
    )
    _, test_df = train_test_split(
        test_full, test_size=0.50, random_state=SEED, stratify=test_full['label']
    )

    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    print(f"Device: {DEVICE}\n")

    tokenizer    = AutoTokenizer.from_pretrained(MODEL_NAME)
    train_ds     = CodeDataset(train_df.reset_index(drop=True), tokenizer)
    val_ds       = CodeDataset(val_df.reset_index(drop=True),   tokenizer)
    test_ds      = CodeDataset(test_df.reset_index(drop=True),  tokenizer)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=4, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)

    counts        = train_df['label'].value_counts().sort_index().values
    raw_w         = torch.tensor([1.0 / c for c in counts], dtype=torch.float).to(DEVICE)
    class_weights = raw_w / raw_w.sum() * len(counts)

    model     = LAPPEFT().to(DEVICE)
    swa_model = copy.deepcopy(model)
    for p in swa_model.parameters():
        p.requires_grad = False

    total_p   = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Params     : {total_p:,}")
    print(f"Trainable Params : {trainable:,}")
    print(f"Trainable %      : {100.0 * trainable / total_p:.4f}%\n")

    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer        = torch.optim.AdamW(
        trainable_params, lr=LR, weight_decay=0.05,
        betas=(0.9, 0.999), eps=1e-8
    )

    effective_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    warmup_steps    = int(WARMUP_RATIO * effective_steps)
    scheduler       = get_cosine_schedule_with_warmup(optimizer, warmup_steps, effective_steps)

    swa_start_epoch = int(SWA_START_RATIO * EPOCHS)
    n_swa           = 0
    step_counter    = [0]

    best_val_f1 = -1.0
    best_epoch  = 0
    best_state  = None
    no_improve  = 0

    history = {
        'tr_loss': [], 'tr_acc': [], 'tr_f1': [], 'tr_auc': [], 'tr_mcc': [],
        'tr_prec': [], 'tr_rec': [], 'tr_bacc': [], 'tr_avgprec': [],
        'va_loss': [], 'va_acc': [], 'va_f1': [], 'va_auc': [], 'va_mcc': [],
        'va_prec': [], 'va_rec': [], 'va_bacc': [], 'va_avgprec': []
    }

    hdr = (f"{'Ep':>3} | {'TrLoss':>8} | {'TrAcc':>7} | {'TrF1':>7} | {'TrAUC':>7} | {'TrMCC':>7} | "
           f"{'VaLoss':>8} | {'VaAcc':>7} | {'VaF1':>7} | {'VaAUC':>7} | {'VaMCC':>7} | {'Note':<12}")
    sep = "-" * len(hdr)
    print(sep)
    print(hdr)
    print(sep)

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_m = train_epoch(model, train_loader, criterion, optimizer,
                                    scheduler, class_weights, step_counter)
        va_loss, va_m = eval_epoch(model, val_loader, criterion)

        history['tr_loss'].append(tr_loss)
        history['tr_acc'].append(tr_m['Accuracy'])
        history['tr_f1'].append(tr_m['F1'])
        history['tr_auc'].append(tr_m['ROC_AUC'])
        history['tr_mcc'].append(tr_m['MCC'])
        history['tr_prec'].append(tr_m['Precision'])
        history['tr_rec'].append(tr_m['Recall'])
        history['tr_bacc'].append(tr_m['Balanced_Acc'])
        history['tr_avgprec'].append(tr_m['Avg_Precision'])

        history['va_loss'].append(va_loss)
        history['va_acc'].append(va_m['Accuracy'])
        history['va_f1'].append(va_m['F1'])
        history['va_auc'].append(va_m['ROC_AUC'])
        history['va_mcc'].append(va_m['MCC'])
        history['va_prec'].append(va_m['Precision'])
        history['va_rec'].append(va_m['Recall'])
        history['va_bacc'].append(va_m['Balanced_Acc'])
        history['va_avgprec'].append(va_m['Avg_Precision'])

        note = ""
        if epoch > swa_start_epoch:
            update_swa(swa_model, model)
            n_swa += 1
            note += "SWA "

        is_best = va_m['F1'] > best_val_f1
        if is_best:
            best_val_f1 = va_m['F1']
            best_epoch  = epoch
            best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve  = 0
            note       += "<--"
        else:
            no_improve += 1

        print(
            f"{epoch:>3} | {tr_loss:>8.4f} | {tr_m['Accuracy']:>7.4f} | {tr_m['F1']:>7.4f} | "
            f"{tr_m['ROC_AUC']:>7.4f} | {tr_m['MCC']:>7.4f} | "
            f"{va_loss:>8.4f} | {va_m['Accuracy']:>7.4f} | {va_m['F1']:>7.4f} | "
            f"{va_m['ROC_AUC']:>7.4f} | {va_m['MCC']:>7.4f} | {note:<12}"
        )

        if no_improve >= PATIENCE:
            print(f"\n  Early stopping at epoch {epoch} (no val F1 improvement for {PATIENCE} epochs)")
            break

    print(sep)
    print(f"\nBest Epoch: {best_epoch}  |  Best Val F1: {best_val_f1:.4f}")

    print("\n--- Evaluating: Best Checkpoint Model ---")
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    _, best_test_m = eval_epoch(model, test_loader, criterion)

    print("\n--- Evaluating: SWA Model ---")
    swa_test_m = None
    if n_swa > 0:
        apply_swa_avg(swa_model, n_swa)
        swa_model.to(DEVICE)
        _, swa_test_m = eval_epoch(swa_model, test_loader, criterion)

    print(f"\n{'='*60}")
    print(f"  Val F1 trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_f1'][-5:]]}")
    print(f"  Val Loss trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_loss'][-5:]]}")

    print(f"\n{'='*60}")
    print(f"  TEST RESULTS — Best Checkpoint (Epoch {best_epoch})")
    print(f"{'='*60}")
    scalar_keys = [k for k in best_test_m if k not in ('TP', 'TN', 'FP', 'FN')]
    for k in scalar_keys:
        print(f"  {k:<22}: {best_test_m[k]:.4f}")
    print(f"  {'TP':<22}: {best_test_m['TP']}")
    print(f"  {'TN':<22}: {best_test_m['TN']}")
    print(f"  {'FP':<22}: {best_test_m['FP']}")
    print(f"  {'FN':<22}: {best_test_m['FN']}")

    if swa_test_m is not None:
        print(f"\n{'='*60}")
        print(f"  TEST RESULTS — SWA Model (averaged over last {n_swa} epochs)")
        print(f"{'='*60}")
        for k in scalar_keys:
            print(f"  {k:<22}: {swa_test_m[k]:.4f}")
        print(f"  {'TP':<22}: {swa_test_m['TP']}")
        print(f"  {'TN':<22}: {swa_test_m['TN']}")
        print(f"  {'FP':<22}: {swa_test_m['FP']}")
        print(f"  {'FN':<22}: {swa_test_m['FN']}")

        print(f"\n{'='*60}")
        print(f"  COMPARISON: Best Checkpoint vs SWA")
        print(f"{'='*60}")
        compare_keys = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall']
        print(f"  {'Metric':<22} {'BestCkpt':>12} {'SWA':>12} {'Delta':>12}")
        print(f"  {'-'*58}")
        for k in compare_keys:
            delta  = swa_test_m[k] - best_test_m[k]
            symbol = "+" if delta >= 0 else ""
            print(f"  {k:<22} {best_test_m[k]:>12.4f} {swa_test_m[k]:>12.4f} {symbol+f'{delta:.4f}':>12}")

    plot_training_curves(history, save_path='training_curves.png')
    plot_final_comparison(best_test_m, swa_test_m, save_path='test_comparison.png')
    plot_confusion_matrix(best_test_m, f'Confusion Matrix — Best Ckpt (Ep {best_epoch})', 'cm_best.png')
    if swa_test_m:
        plot_confusion_matrix(swa_test_m, f'Confusion Matrix — SWA ({n_swa} epochs)', 'cm_swa.png')
    plot_metrics_radar(best_test_m, swa_test_m, save_path='radar_chart.png')

    print("\nAll figures saved.")


if __name__ == "__main__":
    main()

# ablation Study

In [ ]:
import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import OrderedDict
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix, balanced_accuracy_score,
    average_precision_score, roc_curve)
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
try:
    import umap as umap_module
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
MODEL_NAME = "microsoft/unixcoder-base"
TRAIN_CSV = '/traincodex.csv'
TEST_CSV = 'testcodex.csv'
MAX_LENGTH = 512
D_MODEL = 768
BATCH_SIZE = 16
GRAD_ACCUM = 2
EPOCHS = 30
LR = 5e-5
WARMUP_RATIO = 0.15
VAL_SPLIT = 0.20
SEED = 42
SWA_START_RATIO = 0.6
PATIENCE = 7
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = "ablation_part2_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
ABLATION_CONFIGS = OrderedDict([
    ('Proposed',         {'num_levels': 4, 'bottleneck': 6, 'activation': 'gelu',    'layers_level': 3, 'layer_strategy': 'proposed', 'use_residual_diff': True,  'use_dyn_fusion': True,  'use_backbone_lra': False, 'use_bottleneck': True,  'rdrop_alpha': 0.3, 'mixup_alpha': 0.2, 'mixup_prob': 0.5}),
    ('Act_Sigmoid',      {'num_levels': 4, 'bottleneck': 6, 'activation': 'sigmoid', 'layers_level': 3, 'layer_strategy': 'proposed', 'use_residual_diff': True,  'use_dyn_fusion': True,  'use_backbone_lra': False, 'use_bottleneck': True,  'rdrop_alpha': 0.3, 'mixup_alpha': 0.2, 'mixup_prob': 0.5}),
    ('Backbone_LRA',     {'num_levels': 4, 'bottleneck': 6, 'activation': 'gelu',    'layers_level': 3, 'layer_strategy': 'proposed', 'use_residual_diff': True,  'use_dyn_fusion': True,  'use_backbone_lra': True,  'use_bottleneck': True,  'rdrop_alpha': 0.3, 'mixup_alpha': 0.2, 'mixup_prob': 0.5}),
    ('No_Residual_Diff', {'num_levels': 4, 'bottleneck': 6, 'activation': 'gelu',    'layers_level': 3, 'layer_strategy': 'proposed', 'use_residual_diff': False, 'use_dyn_fusion': True,  'use_backbone_lra': False, 'use_bottleneck': True,  'rdrop_alpha': 0.3, 'mixup_alpha': 0.2, 'mixup_prob': 0.5}),
    ('Uniform_Alpha',    {'num_levels': 4, 'bottleneck': 6, 'activation': 'gelu',    'layers_level': 3, 'layer_strategy': 'proposed', 'use_residual_diff': True,  'use_dyn_fusion': False, 'use_backbone_lra': False, 'use_bottleneck': True,  'rdrop_alpha': 0.3, 'mixup_alpha': 0.2, 'mixup_prob': 0.5}),
    ('Level6_Cluster',   {'num_levels': 6, 'bottleneck': 6, 'activation': 'gelu',    'layers_level': 2, 'layer_strategy': 'proposed', 'use_residual_diff': True,  'use_dyn_fusion': True,  'use_backbone_lra': False, 'use_bottleneck': True,  'rdrop_alpha': 0.3, 'mixup_alpha': 0.2, 'mixup_prob': 0.5}),
    ('Level2_Cluster',   {'num_levels': 2, 'bottleneck': 6, 'activation': 'gelu',    'layers_level': 6, 'layer_strategy': 'proposed', 'use_residual_diff': True,  'use_dyn_fusion': True,  'use_backbone_lra': False, 'use_bottleneck': True,  'rdrop_alpha': 0.3, 'mixup_alpha': 0.2, 'mixup_prob': 0.5}),
    ('No_Bottleneck',    {'num_levels': 4, 'bottleneck': 6, 'activation': 'gelu',    'layers_level': 3, 'layer_strategy': 'proposed', 'use_residual_diff': True,  'use_dyn_fusion': True,  'use_backbone_lra': False, 'use_bottleneck': False, 'rdrop_alpha': 0.3, 'mixup_alpha': 0.2, 'mixup_prob': 0.5}),
    ('Layer_Bottom',     {'num_levels': 4, 'bottleneck': 6, 'activation': 'gelu',    'layers_level': 3, 'layer_strategy': 'bottom',   'use_residual_diff': True,  'use_dyn_fusion': True,  'use_backbone_lra': False, 'use_bottleneck': True,  'rdrop_alpha': 0.3, 'mixup_alpha': 0.2, 'mixup_prob': 0.5}),
    ('Layer_Middle',     {'num_levels': 4, 'bottleneck': 6, 'activation': 'gelu',    'layers_level': 3, 'layer_strategy': 'middle',   'use_residual_diff': True,  'use_dyn_fusion': True,  'use_backbone_lra': False, 'use_bottleneck': True,  'rdrop_alpha': 0.3, 'mixup_alpha': 0.2, 'mixup_prob': 0.5}),
])
class CodeDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.codes = df['code'].astype(str).tolist()
        self.labels = df['label'].tolist()
        self.tokenizer = tokenizer
    def __len__(self):
        return len(self.codes)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.codes[idx], max_length=MAX_LENGTH, padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0), 'attention_mask': enc['attention_mask'].squeeze(0), 'label': torch.tensor(self.labels[idx], dtype=torch.long)}
class AttentionPooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attn = nn.Linear(d_model, 1)
    def forward(self, hidden, attention_mask):
        scores = self.attn(hidden).squeeze(-1)
        scores = scores.masked_fill(attention_mask == 0, -1e9)
        weights = torch.softmax(scores, dim=-1)
        return (hidden * weights.unsqueeze(-1)).sum(1)
class DynamicAlphaFusion(nn.Module):
    def __init__(self, d_model, num_levels):
        super().__init__()
        self.gate = nn.Linear(d_model, num_levels)
    def forward(self, base_cls, residuals):
        alphas = torch.softmax(self.gate(base_cls), dim=-1)
        weighted = sum(alphas[:, i:i+1] * residuals[i] for i in range(len(residuals)))
        return base_cls + weighted, alphas
class UniformAlphaFusion(nn.Module):
    def __init__(self, num_levels):
        super().__init__()
        self.num_levels = num_levels
    def forward(self, base_cls, residuals):
        weighted = sum(residuals) / float(self.num_levels)
        alphas = torch.ones(base_cls.shape[0], self.num_levels, device=base_cls.device) / float(self.num_levels)
        return base_cls + weighted, alphas
class LaplacianResidualAdapter(nn.Module):
    def __init__(self, d_model, bottleneck_dim, activation='gelu', use_bottleneck=True):
        super().__init__()
        act_map = {'gelu': nn.GELU(), 'selu': nn.SELU(), 'relu': nn.ReLU(), 'sigmoid': nn.Sigmoid()}
        self.act = act_map.get(activation, nn.GELU())
        self.use_bottleneck = use_bottleneck
        self.dropout = nn.Dropout(0.1)
        if use_bottleneck:
            self.down = nn.Linear(d_model, bottleneck_dim)
            self.up = nn.Linear(bottleneck_dim, d_model)
            nn.init.kaiming_normal_(self.down.weight, nonlinearity='linear')
            nn.init.zeros_(self.down.bias)
            nn.init.zeros_(self.up.weight)
            nn.init.zeros_(self.up.bias)
        else:
            self.linear = nn.Linear(d_model, d_model)
            nn.init.zeros_(self.linear.weight)
            nn.init.zeros_(self.linear.bias)
    def forward(self, x):
        if self.use_bottleneck:
            return self.up(self.dropout(self.act(self.down(x))))
        return self.dropout(self.act(self.linear(x)))
class FlexLAPPEFT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        for p in self.encoder.parameters():
            p.requires_grad = False
        num_levels = cfg['num_levels']
        bottleneck = cfg['bottleneck']
        activation = cfg['activation']
        use_bottleneck = cfg['use_bottleneck']
        use_backbone_lra = cfg['use_backbone_lra']
        use_dyn_fusion = cfg['use_dyn_fusion']
        self.pool_layers = nn.ModuleList([AttentionPooling(D_MODEL) for _ in range(num_levels + 1)])
        self.lras = nn.ModuleList([LaplacianResidualAdapter(D_MODEL, bottleneck, activation, use_bottleneck) for _ in range(num_levels)])
        if use_backbone_lra:
            n_bl = len(self.encoder.encoder.layer)
            pyramid_bns = [max(2, int(bottleneck * 2 * (n_bl - i) / n_bl)) for i in range(n_bl)]
            self.backbone_lras = nn.ModuleList([LaplacianResidualAdapter(D_MODEL, pyramid_bns[i], activation, True) for i in range(n_bl)])
        self.level_norms = nn.ModuleList([nn.LayerNorm(D_MODEL) for _ in range(num_levels)])
        if use_dyn_fusion:
            self.fusion = DynamicAlphaFusion(D_MODEL, num_levels)
        else:
            self.fusion = UniformAlphaFusion(num_levels)
        self.classifier = nn.Sequential(
            nn.Linear(D_MODEL, 256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(64, 2))
    def _get_hidden_states(self, input_ids, attention_mask):
        if self.cfg['use_backbone_lra']:
            emb = self.encoder.embeddings(input_ids=input_ids)
            ext_mask = attention_mask[:, None, None, :]
            ext_mask = (1.0 - ext_mask.float()) * -10000.0
            all_h = [emb]
            h = emb
            for i, layer in enumerate(self.encoder.encoder.layer):
                h_new = layer(h, attention_mask=ext_mask)[0]
                h_new = h_new + self.backbone_lras[i](h_new)
                all_h.append(h_new)
                h = h_new
            return tuple(all_h)
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        return out.hidden_states
    def encode(self, input_ids, attention_mask):
        hidden = self._get_hidden_states(input_ids, attention_mask)
        num_layers = len(hidden)
        num_levels = self.cfg['num_levels']
        layers_level = self.cfg['layers_level']
        layer_strategy = self.cfg['layer_strategy']
        if layer_strategy == 'bottom':
            half = num_layers // 2
            step = max(1, half // max(1, num_levels))
            level_indices = [min((lvl + 1) * step, num_layers - 1) for lvl in range(num_levels)]
        elif layer_strategy == 'middle':
            mid_start = num_layers // 4
            step = max(1, (num_layers // 2) // max(1, num_levels))
            level_indices = [min(mid_start + (lvl + 1) * step, num_layers - 1) for lvl in range(num_levels)]
        else:
            level_indices = [min((lvl + 1) * layers_level, num_layers - 1) for lvl in range(num_levels)]
        base = self.pool_layers[num_levels](hidden[-1], attention_mask)
        level_cls = [self.pool_layers[lvl](hidden[level_indices[lvl]], attention_mask) for lvl in range(num_levels)]
        residuals = []
        for lvl in range(num_levels):
            if self.cfg['use_residual_diff']:
                diff = level_cls[lvl] - level_cls[lvl + 1] if lvl < num_levels - 1 else level_cls[lvl]
            else:
                diff = level_cls[lvl]
            r = self.lras[lvl](diff)
            residuals.append(self.level_norms[lvl](r))
        fused, level_alphas = self.fusion(base, residuals)
        return fused, level_cls, level_alphas
    def forward(self, input_ids, attention_mask, cls_embed=None):
        if cls_embed is not None:
            return self.classifier(cls_embed)
        fused, _, _ = self.encode(input_ids, attention_mask)
        return self.classifier(fused)
def mixup_embeddings(embed, labels, alpha):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(embed.size(0), device=embed.device)
    return lam * embed + (1 - lam) * embed[idx], labels, labels[idx], lam
def rdrop_kl_loss(logits1, logits2):
    p = F.log_softmax(logits1, dim=-1)
    q = F.log_softmax(logits2, dim=-1)
    return (F.kl_div(p, q.exp(), reduction='batchmean') + F.kl_div(q, p.exp(), reduction='batchmean')) / 2
def mixup_ce_loss(logits, la, lb, lam, weight=None):
    loss_fn = nn.CrossEntropyLoss(weight=weight, label_smoothing=0.05, reduction='none')
    return (lam * loss_fn(logits, la) + (1 - lam) * loss_fn(logits, lb)).mean()
def compute_metrics(labels, preds, probs):
    labels = np.array(labels)
    preds = np.array(preds)
    probs = np.array(probs)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        'Accuracy': accuracy_score(labels, preds),
        'Balanced_Acc': balanced_accuracy_score(labels, preds),
        'Precision': precision_score(labels, preds, zero_division=0),
        'Recall': recall_score(labels, preds, zero_division=0),
        'F1': f1_score(labels, preds, zero_division=0),
        'ROC_AUC': roc_auc_score(labels, probs),
        'Avg_Precision': average_precision_score(labels, probs),
        'MCC': matthews_corrcoef(labels, preds),
        'Specificity': tn / (tn + fp + 1e-9),
        'Sensitivity': tp / (tp + fn + 1e-9),
        'FPR': fp / (fp + tn + 1e-9),
        'FNR': fn / (fn + tp + 1e-9),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn)}
def train_epoch(model, loader, criterion, optimizer, scheduler, class_weights, step_counter, cfg):
    model.train()
    total_loss = 0.0
    all_labels, all_preds, all_probs = [], [], []
    optimizer.zero_grad()
    rdrop_a = cfg['rdrop_alpha']
    mixup_p = cfg['mixup_prob']
    mixup_a = cfg['mixup_alpha']
    for batch_idx, batch in enumerate(loader):
        ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        labels = batch['label'].to(DEVICE)
        with torch.no_grad():
            embed, _, _ = model.encode(ids, mask)
        if np.random.rand() < mixup_p:
            mixed_embed, la, lb, lam = mixup_embeddings(embed, labels, mixup_a)
            logits1 = model(None, None, cls_embed=mixed_embed)
            logits2 = model(None, None, cls_embed=mixed_embed)
            ce_loss = mixup_ce_loss(logits1, la, lb, lam, weight=class_weights)
        else:
            logits1 = model(None, None, cls_embed=embed)
            logits2 = model(None, None, cls_embed=embed)
            ce_loss = criterion(logits1, labels)
        kl_loss = rdrop_kl_loss(logits1, logits2)
        loss = (ce_loss + rdrop_a * kl_loss) / GRAD_ACCUM
        loss.backward()
        if (batch_idx + 1) % GRAD_ACCUM == 0 or (batch_idx + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            step_counter[0] += 1
        total_loss += (ce_loss + rdrop_a * kl_loss).item()
        with torch.no_grad():
            probs = torch.softmax(logits1, dim=-1)[:, 1].cpu().numpy()
            preds = logits1.argmax(dim=-1).cpu().numpy()
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)
    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            logits = model(ids, mask)
            total_loss += criterion(logits, labels).item()
            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            preds = logits.argmax(dim=-1).cpu().numpy()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds)
            all_probs.extend(probs)
    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs), np.array(all_labels), np.array(all_probs)
def update_swa(swa_model, model):
    with torch.no_grad():
        for sp, p in zip(swa_model.parameters(), model.parameters()):
            sp.data.copy_(sp.data + p.data)
def apply_swa_avg(swa_model, n):
    with torch.no_grad():
        for p in swa_model.parameters():
            p.data.div_(float(n))
def extract_embeddings(model, loader):
    model.eval()
    all_emb, all_lbl, all_alp = [], [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            fused, _, alphas = model.encode(ids, mask)
            all_emb.append(fused.cpu().numpy())
            all_lbl.append(batch['label'].numpy())
            all_alp.append(alphas.cpu().numpy())
    return np.concatenate(all_emb), np.concatenate(all_lbl), np.concatenate(all_alp)
def linear_cka(X, Y):
    X = X - X.mean(0)
    Y = Y - Y.mean(0)
    nxx = np.linalg.norm(X.T @ X, 'fro')
    nyy = np.linalg.norm(Y.T @ Y, 'fro')
    nxy = np.linalg.norm(X.T @ Y, 'fro') ** 2
    return float(nxy / (nxx * nyy + 1e-10))
def plot_tsne_umap(embeddings, labels, config_name, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    scaler = StandardScaler()
    emb_s = scaler.fit_transform(embeddings)
    perp = min(30, max(5, len(labels) - 1))
    tsne_c = TSNE(n_components=2, random_state=SEED, perplexity=perp).fit_transform(emb_s)
    ncols = 2 if UMAP_AVAILABLE else 1
    fig, axes = plt.subplots(1, ncols, figsize=(7 * ncols, 6))
    if ncols == 1:
        axes = [axes]
    colors = ['#2196F3', '#F44336']
    lnames = ['Non-Vulnerable', 'Vulnerable']
    for c in [0, 1]:
        m = labels == c
        axes[0].scatter(tsne_c[m, 0], tsne_c[m, 1], c=colors[c], label=lnames[c], alpha=0.6, s=8)
    axes[0].set_title(f't-SNE: {config_name}', fontweight='bold')
    axes[0].legend(fontsize=8)
    axes[0].set_xlabel('Dim 1')
    axes[0].set_ylabel('Dim 2')
    axes[0].grid(True, alpha=0.3)
    if UMAP_AVAILABLE:
        umap_c = umap_module.UMAP(n_components=2, random_state=SEED).fit_transform(emb_s)
        for c in [0, 1]:
            m = labels == c
            axes[1].scatter(umap_c[m, 0], umap_c[m, 1], c=colors[c], label=lnames[c], alpha=0.6, s=8)
        axes[1].set_title(f'UMAP: {config_name}', fontweight='bold')
        axes[1].legend(fontsize=8)
        axes[1].set_xlabel('Dim 1')
        axes[1].set_ylabel('Dim 2')
        axes[1].grid(True, alpha=0.3)
    plt.suptitle(f'Representation Visualization: {config_name}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    sp = os.path.join(out_dir, f'tsne_umap_{config_name}.png')
    plt.savefig(sp, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'  Saved: {sp}')
def plot_layer_sensitivity(alpha_weights, config_name, num_levels, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    mean_a = alpha_weights.mean(0)
    fig, ax = plt.subplots(figsize=(max(6, num_levels * 1.5), 5))
    bars = ax.bar(range(num_levels), mean_a, color='steelblue', alpha=0.85, edgecolor='navy', linewidth=0.8)
    ax.set_xticks(range(num_levels))
    ax.set_xticklabels([f'Level {i+1}' for i in range(num_levels)], fontsize=9)
    ax.set_ylabel('Mean Alpha Weight', fontsize=10)
    ax.set_title(f'Layer Sensitivity: {config_name}', fontweight='bold', fontsize=11)
    y_max = float(mean_a.max())
    ax.set_ylim(0, y_max * 1.45 + 1e-6)
    ax.grid(axis='y', alpha=0.3)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + y_max * 0.03, f'{h:.4f}', ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    sp = os.path.join(out_dir, f'layer_sensitivity_{config_name}.png')
    plt.savefig(sp, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'  Saved: {sp}')
def plot_confusion_matrix_fig(m, title, save_path):
    dirn = os.path.dirname(save_path)
    if dirn:
        os.makedirs(dirn, exist_ok=True)
    cm_arr = np.array([[m['TN'], m['FP']], [m['FN'], m['TP']]])
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm_arr, interpolation='nearest', cmap=plt.cm.Blues)
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred Non-Vul', 'Pred Vul'])
    ax.set_yticklabels(['True Non-Vul', 'True Vul'])
    thresh = cm_arr.max() / 2.0
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm_arr[i, j]), ha='center', va='center', fontsize=14, fontweight='bold',
                color='white' if cm_arr[i, j] > thresh else 'black')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
def plot_roc_all(roc_data, title, save_path):
    dirn = os.path.dirname(save_path)
    if dirn:
        os.makedirs(dirn, exist_ok=True)
    fig, ax = plt.subplots(figsize=(12, 8))
    colors = plt.cm.tab20(np.linspace(0, 1, max(len(roc_data), 1)))
    for idx, (name, d) in enumerate(roc_data.items()):
        fpr_a, tpr_a, _ = roc_curve(d['labels'], d['probs'])
        ax.plot(fpr_a, tpr_a, color=colors[idx], linewidth=1.8, label=f"{name} (AUC={d['metrics']['ROC_AUC']:.4f})")
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1.0, label='Random Baseline')
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate', fontsize=11)
    ax.set_title(title, fontweight='bold', fontsize=13)
    ax.legend(loc='lower right', fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'  Saved ROC: {save_path}')
def plot_cka_heatmap(emb_dict, save_path):
    dirn = os.path.dirname(save_path)
    if dirn:
        os.makedirs(dirn, exist_ok=True)
    names = list(emb_dict.keys())
    n = len(names)
    mat = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            mat[i, j] = linear_cka(emb_dict[names[i]], emb_dict[names[j]])
    fig, ax = plt.subplots(figsize=(max(8, n), max(7, n - 1)))
    im = ax.imshow(mat, cmap='Blues', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, label='CKA Similarity')
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(names, fontsize=8)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f'{mat[i, j]:.2f}', ha='center', va='center', fontsize=7,
                color='white' if mat[i, j] > 0.7 else 'black')
    ax.set_title('CKA Similarity Heatmap — Part 2 Configs', fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'  Saved CKA: {save_path}')
def plot_performance_gain(all_metrics, proposed_key, save_path):
    dirn = os.path.dirname(save_path)
    if dirn:
        os.makedirs(dirn, exist_ok=True)
    metric_keys = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Avg_Precision']
    prop_m = all_metrics[proposed_key]
    others = [k for k in all_metrics if k != proposed_key]
    if not others:
        return
    x = np.arange(len(metric_keys))
    width = 0.7 / max(1, len(others))
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(others), 1)))
    fig, ax = plt.subplots(figsize=(max(14, len(others) * 2), 7))
    for idx, name in enumerate(others):
        deltas = [all_metrics[name][k] - prop_m[k] for k in metric_keys]
        offset = (idx - len(others) / 2.0 + 0.5) * width
        ax.bar(x + offset, deltas, width, label=name, color=colors[idx], alpha=0.82, edgecolor='black', linewidth=0.4)
    ax.axhline(0, color='black', linewidth=1.2)
    ax.set_xticks(x)
    ax.set_xticklabels(metric_keys, rotation=25, ha='right', fontsize=9)
    ax.set_ylabel('Delta vs Proposed', fontsize=10)
    ax.set_title('Performance Gain per Ablation Component vs Proposed — Part 2', fontweight='bold', fontsize=12)
    ax.legend(fontsize=7, ncol=3)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'  Saved performance gain: {save_path}')
def print_all_metrics(config_name, metrics, pct, best_ep, swa_metrics=None, n_swa=0):
    sep = '=' * 65
    scalar_k = [k for k in metrics if k not in ('TP', 'TN', 'FP', 'FN')]
    print(f'\n{sep}')
    print(f'  TEST RESULTS [Best Checkpoint]: {config_name}')
    print(f'  Best Epoch: {best_ep}  |  Trainable Parameters: {pct:.4f}%')
    print(sep)
    for k in scalar_k:
        print(f'  {k:<22}: {metrics[k]:.4f}')
    print(f'  {"TP":<22}: {metrics["TP"]}')
    print(f'  {"TN":<22}: {metrics["TN"]}')
    print(f'  {"FP":<22}: {metrics["FP"]}')
    print(f'  {"FN":<22}: {metrics["FN"]}')
    if swa_metrics is not None:
        print(f'\n{sep}')
        print(f'  TEST RESULTS [SWA — {n_swa} epochs averaged]: {config_name}')
        print(sep)
        for k in scalar_k:
            print(f'  {k:<22}: {swa_metrics[k]:.4f}')
        print(f'  {"TP":<22}: {swa_metrics["TP"]}')
        print(f'  {"TN":<22}: {swa_metrics["TN"]}')
        print(f'  {"FP":<22}: {swa_metrics["FP"]}')
        print(f'  {"FN":<22}: {swa_metrics["FN"]}')
        cmp_k = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Avg_Precision']
        print(f'\n  {"Metric":<22} {"BestCkpt":>12} {"SWA":>12} {"Delta":>12}')
        print(f'  {"-" * 58}')
        for k in cmp_k:
            delta = swa_metrics[k] - metrics[k]
            sym = '+' if delta >= 0 else ''
            print(f'  {k:<22} {metrics[k]:>12.4f} {swa_metrics[k]:>12.4f} {sym + f"{delta:.4f}":>12}')
    print(sep)
def run_ablation(config_name, cfg, train_loader, val_loader, test_loader, class_weights, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    print(f'\n{"#" * 70}')
    print(f'  ABLATION CONFIG: {config_name}')
    print(f'{"#" * 70}')
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    model = FlexLAPPEFT(cfg).to(DEVICE)
    swa_model = copy.deepcopy(model)
    for p in swa_model.parameters():
        p.requires_grad = False
    total_p = sum(p.numel() for p in model.parameters())
    train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    pct = 100.0 * train_p / total_p
    print(f'  Total Params: {total_p:,}  |  Trainable: {train_p:,}  ({pct:.4f}%)')
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR, weight_decay=0.05, betas=(0.9, 0.999), eps=1e-8)
    eff_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    warmup_s = int(WARMUP_RATIO * eff_steps)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_s, eff_steps)
    swa_start = int(SWA_START_RATIO * EPOCHS)
    n_swa = 0
    sc = [0]
    best_vf1 = -1.0
    best_ep = 0
    best_state = None
    no_imp = 0
    hdr = f'{"Ep":>3} | {"TrLoss":>8} | {"TrAcc":>7} | {"TrF1":>7} | {"TrAUC":>7} | {"TrMCC":>7} | {"VaLoss":>8} | {"VaAcc":>7} | {"VaF1":>7} | {"VaAUC":>7} | {"VaMCC":>7} | Note'
    print('-' * len(hdr))
    print(hdr)
    print('-' * len(hdr))
    for ep in range(1, EPOCHS + 1):
        tr_loss, tr_m = train_epoch(model, train_loader, criterion, optimizer, scheduler, class_weights, sc, cfg)
        va_loss, va_m, _, _ = eval_epoch(model, val_loader, criterion)
        note = ''
        if ep > swa_start:
            update_swa(swa_model, model)
            n_swa += 1
            note += 'SWA '
        if va_m['F1'] > best_vf1:
            best_vf1 = va_m['F1']
            best_ep = ep
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_imp = 0
            note += '<--'
        else:
            no_imp += 1
        print(f'{ep:>3} | {tr_loss:>8.4f} | {tr_m["Accuracy"]:>7.4f} | {tr_m["F1"]:>7.4f} | {tr_m["ROC_AUC"]:>7.4f} | {tr_m["MCC"]:>7.4f} | {va_loss:>8.4f} | {va_m["Accuracy"]:>7.4f} | {va_m["F1"]:>7.4f} | {va_m["ROC_AUC"]:>7.4f} | {va_m["MCC"]:>7.4f} | {note}')
        if no_imp >= PATIENCE:
            print(f'  Early stopping at epoch {ep} (no Val F1 improvement for {PATIENCE} epochs)')
            break
    print('-' * len(hdr))
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    pt_path = os.path.join(out_dir, f'{config_name}_best.pt')
    torch.save({'state_dict': model.state_dict(), 'cfg': cfg, 'best_epoch': best_ep, 'best_val_f1': best_vf1}, pt_path)
    print(f'  Best model saved: {pt_path}')
    _, best_m, t_labels, t_probs = eval_epoch(model, test_loader, criterion)
    swa_m = None
    if n_swa > 0:
        apply_swa_avg(swa_model, n_swa)
        swa_model.to(DEVICE)
        _, swa_m, _, _ = eval_epoch(swa_model, test_loader, criterion)
        swa_pt = os.path.join(out_dir, f'{config_name}_swa.pt')
        torch.save({'state_dict': swa_model.state_dict(), 'cfg': cfg, 'n_swa': n_swa}, swa_pt)
        print(f'  SWA model saved: {swa_pt}')
    print_all_metrics(config_name, best_m, pct, best_ep, swa_m, n_swa)
    plot_confusion_matrix_fig(best_m, f'CM Best: {config_name} (Ep {best_ep})', os.path.join(out_dir, f'cm_best_{config_name}.png'))
    if swa_m is not None:
        plot_confusion_matrix_fig(swa_m, f'CM SWA: {config_name}', os.path.join(out_dir, f'cm_swa_{config_name}.png'))
    emb, lbl, alp = extract_embeddings(model, test_loader)
    plot_tsne_umap(emb, lbl, config_name, out_dir)
    plot_layer_sensitivity(alp, config_name, cfg['num_levels'], out_dir)
    return best_m, t_labels, t_probs, emb
def main():
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    train_full = pd.read_csv(TRAIN_CSV)
    test_full = pd.read_csv(TEST_CSV)
    train_df, val_df = train_test_split(train_full, test_size=VAL_SPLIT, random_state=SEED, stratify=train_full['label'])
    _, test_df = train_test_split(test_full, test_size=0.50, random_state=SEED, stratify=test_full['label'])
    print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
    print(f'Device: {DEVICE}')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    train_ds = CodeDataset(train_df.reset_index(drop=True), tokenizer)
    val_ds = CodeDataset(val_df.reset_index(drop=True), tokenizer)
    test_ds = CodeDataset(test_df.reset_index(drop=True), tokenizer)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    counts = train_df['label'].value_counts().sort_index().values
    raw_w = torch.tensor([1.0 / c for c in counts], dtype=torch.float).to(DEVICE)
    class_weights = raw_w / raw_w.sum() * len(counts)
    all_metrics = {}
    roc_data = {}
    emb_dict = {}
    for config_name, cfg in ABLATION_CONFIGS.items():
        cfg_dir = os.path.join(OUTPUT_DIR, config_name)
        best_m, t_labels, t_probs, emb = run_ablation(config_name, cfg, train_loader, val_loader, test_loader, class_weights, cfg_dir)
        all_metrics[config_name] = best_m
        roc_data[config_name] = {'labels': t_labels, 'probs': t_probs, 'metrics': best_m}
        emb_dict[config_name] = emb
    plot_roc_all(roc_data, 'ROC Curves — Part 2 Ablation (Structural & Strategy Configs)', os.path.join(OUTPUT_DIR, 'roc_all_part2.png'))
    plot_cka_heatmap(emb_dict, os.path.join(OUTPUT_DIR, 'cka_heatmap_part2.png'))
    plot_performance_gain(all_metrics, 'Proposed', os.path.join(OUTPUT_DIR, 'performance_gain_part2.png'))
    metric_keys = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Avg_Precision']
    print(f'\n{"=" * 65}')
    print(f'  FINAL SUMMARY TABLE — Part 2 Ablation Configs')
    print(f'{"=" * 65}')
    col_w = 14
    header = f'  {"Config":<22}' + ''.join(f'{k:>{col_w}}' for k in metric_keys)
    print(header)
    print(f'  {"-" * (22 + col_w * len(metric_keys))}')
    for name, m in all_metrics.items():
        row = f'  {name:<22}' + ''.join(f'{m[k]:>{col_w}.4f}' for k in metric_keys)
        print(row)
    print(f'\n{"=" * 65}')
    print(f'  PERFORMANCE GAIN TABLE (Delta vs Proposed)')
    print(f'{"=" * 65}')
    prop_m = all_metrics['Proposed']
    header2 = f'  {"Config":<22}' + ''.join(f'{k:>{col_w}}' for k in metric_keys)
    print(header2)
    print(f'  {"-" * (22 + col_w * len(metric_keys))}')
    for name, m in all_metrics.items():
        if name == 'Proposed':
            continue
        row = f'  {name:<22}' + ''.join(f'{m[k] - prop_m[k]:>+{col_w}.4f}' for k in metric_keys)
        print(row)
    print(f'{"=" * 65}')
    print(f'\nPart 2 complete. All outputs saved in: {OUTPUT_DIR}')
if __name__ == '__main__':
    main()